# <center>OpenClaw 专题课第二节课：上下文、多智能体和记忆管理架构</center>

&emsp;&emsp;上节课我们跟着一条消息钻进了 agent 的心脏，看清了那个让它"能被打断"的双层 while 循环、它对外吐出的事件流、它调用工具时要穿过的策略管线。但当时我们其实绕过了一个最该问、却最难一眼看清的问题：**模型这一轮到底"看到"了什么**？ 你发了一句"你好"，可真正发给大模型的请求体里，绝不只有这三个字——它前面塞了系统指令、塞了你的长期记忆、塞了之前的对话历史。这一层"看不见的组装"，恰恰是 agent 跟普通 API 调用拉开差距的地方。今天我们就要把它彻底掀开。

&emsp;&emsp;而且不止于此。上节课讲的那些——双层 while、事件流、策略管线——都是发生在**单个 agent 内部**的事。可一旦放到真实任务里，单个 agent 常常就不够用了：context 会被越聊越长的历史撑爆、复杂任务一根 loop 扛不住、跨会话之后它又把你记过的事忘得一干二净。这三个"不够用"，正好引出本节要拆的三个机制。这一节课我们回答的核心问题就是——**一个 agent 不够用时怎么办**。我们沿着一根主轴往下走，这根轴就是 `context`（上下文）：先看 agent **当下**这一轮拿到的 context 是怎么一层层组装出来的（第 1 章，我们会用 `openclaw proxy` 亲眼看看实际发给模型的内容）；再看任务复杂到单个 loop 装不下时，它如何 spawn 出**子 agent** 来分治、子 agent 又隔离了什么共享了什么（第 2 章）；最后看跨会话的 context 最初**从哪来**——那套把上次对话的精华存下来、下次再检索回当下 context 的记忆系统（第 3 章）。一句话串起来：**context 的当下组装（1）→ 分治隔离（2）→ 持久来源（3）→ 收口成框架（4）**。

&emsp;&emsp;和上节课一样，我们会反复用两件武器：一边读真实的 TypeScript 源码锚点，建立"它真的是这么写的"的信任；一边用 Python 写最小可运行的重现版（我们后面统一叫它 MVP），在你眼前跑起来，建立"这个机制我能自己复现"的掌控感。本节课最特别的一件武器是 `openclaw proxy`——它能把 agent 实际发给模型的请求体原原本本捕获下来，让"context 组装"这件原本看不见的事，第一次变得肉眼可见。下面我们从第 0 章的承上开始，先把"单 agent 为什么不够用"这个问题立起来。

> 📌 **目标受众与前置要求**：本课承接第一节课的产物——你已经在自己机器上跑通了 `pnpm openclaw tui --local` 并发出过第一条消息，记得 agent loop 那个能被打断的双层 while 结构，对 EventStream 的事件流、工具策略管线、exec 安全档位有印象。技术上你需要能读懂基础的 TypeScript 函数签名、能跑 Python 脚本，**不需要**自己实现过 agent 运行时，也**不需要**改动 OpenClaw 源码。如果你是混合基础里偏零基础的那一拨，本节的源码锚点你可以先跳读，重点跟着 Python MVP 和真实命令观测走，照样能拿到核心收获。

> 📌 **学完本节你将带走 7 件产物**：① 能用 OpenClaw 的进程内捕获开关（`OPENCLAW_DEBUG_PROXY_ENABLED`）配合 `openclaw proxy sessions / blob` 命令，亲眼看见实际发给模型的 context，把它拆成系统指令 / 记忆注入 / 历史 / 当前消息四层，并算清一笔成分账单——context 的体积大头是工具结果和工具 schema，不是对话文字；② 能复述 ContextEngine 五个生命周期阶段（bootstrap → ingest → assemble → compact → prepareSubagentSpawn），以及其中哪两个是 fail-closed，还知道 compact 只是 context 治理"四道防线"（源头限流 → 落盘封顶 → 组装瘦身 → 压力触发）的最后一招；③ 一组能自己跑通的 Python MVP——重现 context 四层组装 + compact 压缩的核心逻辑，进阶再加四道防线流水线和"追加 vs 重写"缓存前缀对比两个 demo，外加一个在你自己机器上自动核算 context 成分账单的真机 cell；④ 能说清 subagent 的"两隔离 + 两共享"——session 和 context 隔离、lane（全局并发队列）和 auth（合并继承）共享，并知道 auth 不隔离是官方文档明说的当前现状，也知道子 agent 的工具约束**只严不松**（`sessions_send` 等永久禁用是策略层强制）、spawn 是非阻塞的、结果靠 `announce` 回报；⑤ 一段重现 subagent 两隔离两共享语义（isolated vs fork、lane 共享并发安全阀）的 Python MVP；⑥ 能说清 workspace 的 3 个官方记忆文件（MEMORY.md / 每日记忆 / DREAMS.md——第三个要开启 dreaming 后才会出现），并说清 dreaming 三处（两目录 + 一文件）各管什么，且知道 `MEMORY.md` 有默认 10K 字符的注入预算，还能按文件名形态归因记忆的三条自动来路——压缩前 flush 抢救、`/new` 收尾 hook 快照（这条解释了 `memory/` 下带时分后缀的文件哪来的）、dreaming 凌晨沉淀；⑦ 能讲清 `memory_search` 的 hybrid RAG 检索是向量 0.7 + 全文 0.3 加权合并，以及这两个权重在源码里的确切出处，并知道完整用法是 `memory_search` 定位 + `memory_get` 精读的双步范式。

> 📌 **学完不能做（诚实划界）**：本节有五条边界我们提前讲在明处，免得你产生错误预期——subagent 的"两隔离两共享"四个维度里，本课只对 **session 隔离做了运行时实证**（真机派生子 agent、session key 独立落盘眼见为实），context 隔离和共享侧（lane / auth）仍是"源码 + 官方文档证、运行时待补"；跨会话的对话转录记忆（session transcript memory）在当前版本**默认是关闭的**，需要你显式打开，不要以为它天然常驻；dreaming（做梦机制）在当前版本**默认同样是关闭的**，开启后才由定时任务后台触发，课堂现场很难即时逼出来，我们降级为翻看它已经产出的文件；ContextEngine 里 compact 和 prepareSubagentSpawn 两个阶段是 fail-closed（出错就直接抛、不静默兜底），这是这两个阶段的特例，**不代表**整套引擎都这么严格；最后，dreaming 的 **recovery 自愈目前只有配置预留、没有执行链**——源码里 `RECOVERY_` 常量组和配置解析链都在，但当前主分支没有找到消费它的执行代码，别把它当成已生效的自愈机制。

> 📅 **时效性说明**：本课全部源码引用基于 2026 年 5 月底 OpenClaw GitHub 主分支（地址 https://github.com/openclaw/openclaw）当时的代码状态。所有 `file:line` 引用都是真实可核对的——你可以在自己电脑 `git clone` 仓库后，用 `vim 路径 +行号` 打到对应位置亲自验证。

---

## <center>第 0 章：承上——从单 agent 到「不够用」</center>

&emsp;&emsp;在掀开 context 组装这层盖子之前，我们先花几分钟把一件事接续好——上节课我们到底走到了哪一步，又为什么说"单个 agent 不够用"。这一章没有任何代码，它是一座桥：左边连着上节课你已经建立的"单 agent 心脏"的认知，右边连着本节课三章要剖开的三个新机制。先复习左边那一半（第 0.1 节），再把"单 agent 撞到的三堵墙"立起来（第 0.2 节），这三堵墙正好一一对应本节后面三章的答案。

&emsp;&emsp;为什么要专门铺这一章？因为本节课的三个机制——上下文工程、多智能体、记忆系统——单独看每一个都容易陷进细节，但只要你心里有"它们各自在补单 agent 的哪块短板"这张地图，就能始终知道自己在整片森林的哪个位置。我们从回顾上节课的产物开始。

### 0.1 上节回顾：单 agent 的心脏，我们已经看清了什么

&emsp;&emsp;上节课我们跟着一条真实消息的生命周期，从"龙虾是什么"一路走到"它为什么这么设计"。如果把上节课的收获浓缩成一句话，那就是：**我们看清了单个 agent 的心脏是怎么跳的**。具体落到能力上，下面这张表是你上节课应该已经带走的五章产物，我们快速过一遍——它不是为了重复，而是为了标出哪些会在今天被用到。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>第一节课五章产物回顾（标注本节复用点）</font></p>
<div class="center">

| 上节章节 | 你已经能做到的 | 本节怎么用上 |
|----------|----------------|--------------|
| 第 0 章 定位 | 用跨渠道 × 跨设备 × 跨 model provider 复述 OpenClaw 定位；说出四代演进 | 帮你理解为什么"个人 agent"也要分治和记忆 |
| 第 1 章 装机 | 在本机跑通仓库内的 `pnpm openclaw tui --local`，发出第一条消息 | 本节真机观测换用同家族的**无头**命令 `openclaw agent --local`（notebook 里没有真终端，TUI 跑不起来——1.1 节会讲） |
| 第 2 章 心脏 | 读懂 agent loop 双层 while、解释 steering 分水岭、叫出 EventStream 的 10 种事件 | 本节的 compact、subagent 起停都是这套事件流的延伸 |
| 第 3 章 工具 | 分清 plugin / capability / tool 边界、说出 8 步策略管线 | `subagents`、`memory_search` 都是要走这套策略管线的工具 |
| 第 4 章 安全 | 区分 exec 三档安全级别 + Docker sandbox 定位、理解"信任 vs 约束"张力 | 第 2 章 subagent 隔离正是这套张力在多 agent 场景的延伸 |

</div>

&emsp;&emsp;这张表里最值得划重点的是第 2 章和第 3 章那两行。上节课我们看 agent loop 时，关注的是"循环怎么转、能不能被打断"；看工具系统时，关注的是"一次工具调用要穿过几道关"。但我们始终没回答一个问题：**每一轮循环开始时，发给模型的那个请求里到底装了什么**。上节课把这个问题搁置了，本节第 1 章开场就来正面回答它。

### 0.2 痛点：单 agent 撞到的三堵墙

&emsp;&emsp;上节课结尾我们说，要从"单个 agent 的心脏"往外走一步，回答"一个 agent 不够用时怎么办"。那么"不够用"具体不够在哪？我们把它拆成三堵实实在在的墙——每一堵墙，恰好就是本节后面一章要拆的。

&emsp;&emsp;**第一堵墙：context 组装是个黑盒**。 上节课我们能看到模型吐出的回答、能看到工具被一个个调用，但模型这一轮"输入端"看到的完整内容，我们其实从没见过。系统指令是从哪注入的？我的长期记忆是怎么混进去的？历史对话又是怎么排布的？这些上节课全没讲。这堵墙，第 1 章用 `openclaw proxy` 直接砸开——我们会把实际发给模型的请求体原样捞出来看。

&emsp;&emsp;**第二堵墙：复杂任务没法分治**。 单个 agent 只有一个 loop，所有事都串在这一条线上做。可现实里很多任务要么大到单轮 context 装不下，要么需要"你去查这个、我同时算那个"的并行。一根独木桥撑不住，这时就需要 spawn 出子 agent 来分摊。这堵墙，第 2 章用 OpenClaw 的 subagent 机制来拆——我们会看清子 agent 究竟独立到什么程度、又跟主 agent 共享什么。

&emsp;&emsp;**第三堵墙：跨会话的记忆是个谜**。 上节课你发了消息、agent 帮你做了事，但你关掉再开一个新会话，那些"它本该记得"的东西去哪了？怎么才能在新会话里把上次的精华找回来？这堵墙，第 3 章用 OpenClaw 的记忆系统来拆——我们会翻开真实的记忆文件，看它怎么存、再看 `memory_search` 怎么把它检索回来。

&emsp;&emsp;这三堵墙不是各自孤立的，它们围着同一根轴——`context`。第一堵讲的是当下这一轮的 context 怎么**组装**；第二堵讲的是分治时子 agent 的 context 怎么**隔离**；第三堵讲的是跨会话时 context 最初怎么**回灌**进来。本节课最后的第 4 章，我们会把这三块拼成一个完整的框架收口。下面我们先给这个框架一张全景图，让你心里先有个结论性的画面，细节留到第 4 章详解。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124803075.png" width=50%></div>

&emsp;&emsp;先说清楚一件事免得后面误会：上面这张图里的"当下脑 / 分治脑 / 长期脑"是**本课为了帮你记住三个模块职责而起的形象说法，不是 OpenClaw 官方术语**，业界也没有这套叫法。它的唯一作用是给你一个抓手——记住"有这么三块东西，都围着 context 这根轴转"。真正的源码里它们各有正式名字（ContextEngine、subagent 系统、记忆系统），我们后面都会一一对应上。读完这一章，你已经建立起本节最重要的那张认知地图——单 agent 的三个短板（context 黑盒、无法分治、跨会话失忆），以及本节三个机制各补哪一块。带着这张地图，我们正式进入第 1 章，先砸开第一堵墙。

---

## <center>第 1 章：上下文工程——agent 当下拿到什么</center>

&emsp;&emsp;这是本节课的第一个重头戏。我们要回答的核心问题只有一个：**你发一句话，模型这一轮到底收到了什么**？ 这个问题听起来简单，但它是整个 agent 工程里最关键、也最容易被忽略的一环——业界专门给它起了个名字叫"上下文工程"（context engineering），意思是：模型本身的能力是固定的，但你喂给它的 context 组装得好不好，直接决定了它这一轮表现的上限。

&emsp;&emsp;本章我们分三步走。第 1.1 节先动手——用 `openclaw proxy` 把实际发给模型的请求体原样捞出来，让你先**亲眼看见**结果，再回头讲机制（先看成品、再拆原理，比一上来讲抽象概念更直观）。第 1.2 节钻进源码，看 OpenClaw 的 `ContextEngine` 是怎么用五个生命周期阶段把这个 context 一步步组装出来的，并用 Python MVP 把核心逻辑在你眼前重现。第 1.3 节聚焦其中最有意思的 `compact`（上下文压缩）阶段——当对话长到快撑爆模型的窗口时，它怎么把历史压短。第 1.4 节收口，顺带把通往第 2 章 subagent 的那道门（`prepareSubagentSpawn`）预埋好。我们从砸开黑盒开始。

### 1.1 用 proxy 捕获实际发给模型的请求

&emsp;&emsp;先解释这个工具是什么：OpenClaw 自带一套 debug 捕获机制（命令族叫 `openclaw proxy`），能把 agent 跟大模型之间来回交换的 HTTP 请求/响应原原本本存档下来。它有两种工作方式：一种是**中间人代理**（`proxy run` / `proxy start`，把流量导过一个本地代理再复制存档），另一种是**进程内捕获**——agent 进程自己在每次发出模型请求时，顺手把请求体/响应体抄一份进本地档案库，像一台"旁路录音机"，完全不改变通信本身。本课用第二种（为什么不用第一种，下面的踩坑预警会讲清楚）。我们要捕获的，正是那个**发给模型的请求体**——它里面装的就是组装完成的完整 context。先用一张图把这台"录音机"的位置和整条观测链立起来，你再看下面的命令就不会迷路。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124758628.png" width=70%></div>

&emsp;&emsp;看清了"录音机"的位置，整条观测链就好记了：开开关跑一轮 → 确认录到了 → 查出 blob id → 读出请求体。

&emsp;&emsp;这套观测靠"一个开关 + 两条命令 + 一次档案查询"配合完成，在源码里都能精确定位。捕获开关是环境变量 `OPENCLAW_DEBUG_PROXY_ENABLED`（`src/proxy-capture/env.ts:11`，常量定义行）；agent 进程里"开关为真才记录"的判断在 `src/infra/net/fetch-guard.ts:267`（它所在的捕获函数 `captureGuardedFetchExchange` 定义在同文件 `:256`）。捕获档案落在本机 `~/.openclaw/debug-proxy/` 下：事件元数据进 `capture.sqlite`、请求体原文进 `blobs/` 目录（两个路径的解析函数在 `src/proxy-capture/paths.ts:8-14`）。查看侧的两条命令：`sessions` 列出最近捕获到的会话（`src/cli/proxy-cli.ts:130`，`.command("sessions")` 定义处），`blob` 按 id 读出某次捕获的具体请求体（`src/cli/proxy-cli.ts:157`，`.command("blob")` 定义处，注意它带一个 `--id` 必填参数）。下面我们先跑前两步——带着开关完成一轮真实对话，并确认它被录下来了。

> **【踩坑预警】**：千万**不要**用 `openclaw proxy query --preset ...` 去看 context。`--preset` 这一组枚举（`double-sends` / `retry-storms` / `cache-busting` / `ws-duplicate-frames` / `missing-ack` / `error-bursts`，定义在 `src/proxy-capture/types.ts:60-66`——`:60` 是 `CaptureQueryPreset` 类型名、`:61-66` 是这组枚举值）全都是**网络异常诊断**用的——它们是用来排查"是不是重复发送了""是不是重试风暴了"这类网络层问题，跟你这一轮发给模型的 prompt 内容毫无关系。如果你误用它去找 context，会一头雾水什么都看不到。正确做法：看实际 context 只用 `proxy blob`。排查方法：记住一个判据——`--preset` 关心的是"网络通信健康度"，`blob` 关心的是"这一次到底发了什么内容"。

&emsp;&emsp;下面这个 cell 执行观测链的前两步：先带着捕获开关跑一轮**无头对话**（`agent --local` 在当前进程里直接跑一轮 embedded agent、打印回复后自动退出，天然适合 notebook；`--agent main` 指定默认 agent），再用 `proxy sessions` 列出最近的捕获会话、确认这一轮的 HTTP 交换确实被录下来了。注意这两条命令需要你本机已经装好 OpenClaw 并配好模型凭证（模型凭证就是第一节课第 1 章配好的那套）。还有一个命令形态上的衔接要说清：第一节课为了让源码行号跟运行时严格对得上，用的是仓库内的 `pnpm openclaw`；本节的真机观测看的是**行为**（捕获落盘、会话注册）而不是逐行对照源码，所以统一写成全局 `openclaw`（本课实测环境就是全局安装版）——如果你只装了仓库版，把命令里的 `openclaw` 换成仓库目录下的 `pnpm openclaw` 即可，两种写法等价。如果你的环境跑不通，跟着下面的讲解理解机制即可。

In [35]:
# 这两条命令需在本机已安装 OpenClaw 并配好模型凭证的环境下运行（第一节课装机产物）

# 步骤 1：打开进程内捕获开关，跑一轮无头对话（这一轮的请求体会被原样存档）
#   开关常量定义在 src/proxy-capture/env.ts:11；"开关为真才记录"的判断在 src/infra/net/fetch-guard.ts:267
#   agent 子命令定义在 src/cli/program/register.agent.ts:67
!OPENCLAW_DEBUG_PROXY_ENABLED=1 openclaw agent --local --agent main --message "你好，简单介绍下自己"

# 步骤 2：列出最近 3 个捕获会话，确认 eventCount > 0（说明这一轮的交换已被录下）
#   sessions 子命令定义在 src/cli/proxy-cli.ts:130
!openclaw proxy sessions --limit 3

│
◇  
5h
🦞 OpenClaw 2026.5.28 (e932160)
   I'll butter your workflow like a lobster roll: messy, delicious, effective.

19:20:57 [plugins] plugins.allow is empty; discovered non-bundled plugins may auto-load: feishu (/Users/mac/.openclaw/npm/projects/openclaw-feishu-dc69f44688/node_modules/@openclaw/feishu/dist/index.js), openclaw-weixin (/Users/mac/.openclaw/npm/projects/tencent-weixin-openclaw-weixin-7783ac86ba/node_modules/@tencent-weixin/openclaw-weixin/dist/index.js). Set plugins.allow to explicit trusted ids.
19:20:59 [skills] Skipping escaped skill path outside its configured root: source=openclaw-workspace root=~/.openclaw/workspace/skills reason=symlink-escape requested=~/.openclaw/workspace/skills/_codex-compat resolved=~/.claude/skills/_codex-compat
19:20:59 [skills] Skipping escaped skill path outside its configured root: source=openclaw-workspace root=~/.openclaw/workspace/skills reason=symlink-escape requested=~/.openclaw/workspace/skills/brainstorming resolved=~/.agents/

&emsp;&emsp;这一步跑完，`proxy sessions` 会列出最近的捕获会话，每条带会话 id 和 `eventCount`。但注意一个容易卡住的点：**它列出的是"会话"级别的元数据，里面没有我们最终要的 blob id**——blob id 是每一次请求/响应体存档的编号，挂在更细的"事件"级别，而 CLI 目前没有"列出事件"的子命令。所以这一步我们直接查捕获档案库：用 macOS 自带的 `sqlite3` 从 `capture_events` 事件表（建表语句在 `src/proxy-capture/store.sqlite.ts:47`）里，把最近几条**发出方向**（`direction='outbound'`，即发给模型的请求）的事件连同 blob id 一起捞出来。

In [36]:
# 步骤 3：从捕获档案库查出最近的请求事件及其 blob id
#   档案库路径解析函数在 src/proxy-capture/paths.ts:8-14；capture_events 建表在 src/proxy-capture/store.sqlite.ts:47
#   direction='outbound' = 发出方向 = 发给模型的请求
!sqlite3 ~/.openclaw/debug-proxy/capture.sqlite \
  "SELECT data_blob_id, host, datetime(ts/1000,'unixepoch','localtime') AS time \
   FROM capture_events WHERE direction='outbound' ORDER BY ts DESC LIMIT 5"

f6c47964f00642537bc510c7|api.minimaxi.com|2026-06-05 19:21:03
38e5dcd72092b9476a526424|api.minimaxi.com|2026-06-05 14:31:27
03588d6635d3a099c685ee83|api.minimaxi.com|2026-06-05 12:58:10
d06d61a7954097f5e29f2bca|api.minimaxi.com|2026-06-05 12:30:17
bfc066fc635cd2149a086fbe|api.minimaxi.com|2026-06-05 00:41:44


&emsp;&emsp;输出每行是"blob id ｜目标 host ｜时间"。挑时间最新、host 是你模型 provider 域名的那条。这里有个识别技巧：捕获库里录的是**所有**发给模型的请求，除了你这轮的主对话，还可能混着 OpenClaw 后台自己发起的请求（比如第 1.3 节会讲的压缩摘要请求）——主对话请求的特征是读出来 `system` 字段巨大、`messages` 里能看到你刚发的那句话；如果读出来发现 `system` 只有几百字、内容是让模型做摘要，那就是后台请求，换下一个 id 再读。拿到 id 之后，最后一步就是用 `proxy blob` 把请求体的完整内容读出来——这才是最关键的一步。

In [37]:
# 步骤 4：用 blob 子命令按 id 读出捕获的请求体 = 实际发给模型的完整 context
#   blob 子命令定义在 src/cli/proxy-cli.ts:157，--id 是必填参数
#   <blobId> 需替换为上一步查询结果里的真实 id（每次运行都不同）
!openclaw proxy blob --id f6c47964f00642537bc510c7

│
◇  
5h
🦞 OpenClaw 2026.5.28 (e932160) — No $999 stand required.

{"model":"MiniMax-M3","messages":[{"role":"assistant","content":[{"type":"thinking","thinking":"The user (Zrj) sent a message asking me to \"统计 workspace 文件数 v3\" (count workspace files v3). This was routed as an inter-session message from a subagent that completed the task. The subagent has already provided the statistics.\n\nLet me review the result:\n- Total files (including hidden): 180\n- Total directories: 58\n- Top-level files: 12\n- Top-level directories: 17\n- By type: 100 docs (.md 96 + .txt 4), 34 code (.json 23 + .sh 4 + .js 4 + .py 2 + .yml 1), 26 images (.jpg 26), 2 other (.log 1 + .jsonl 1)\n- Top extensions: .md 96, .jpg 26, .json 23, .txt 4, .sh 4, .js 4, .py 2, .yml 1, .log 1, .jsonl 1\n\nThe subagent already completed the task. I should send a user-facing update with the results. The instruction says \"Reply ONLY: NO_REPLY when no user-facing update is needed\" — but the user did ask for the statistic

&emsp;&emsp;这条命令读出来的就是 agent 实际发给模型的请求体原文。当你第一次看到它时，多半会"哦"一声——原来你只打了一句"你好，简单介绍下自己"，但请求体里却密密麻麻装了好几大块东西。在我们的测试环境里，哪怕只发这一句话，请求体的 `system` 字段就有三万多字符、外加二十多条历史消息——你机器上的具体数字会不同（取决于你的配置和会话历史），但"你打的那句只是冰山一角"这个结构是一样的。这正是上下文工程的全部秘密所在：**模型看到的从来不只是你打的那句话**。把这个请求体拆开，它实际上是分层堆叠的四层结构，我们用下面这张图先建立空间感。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124750828.png" width=60%></div>

&emsp;&emsp;这四层从上到下分别是：**系统指令**（定义 agent 的角色、能用哪些工具、要守哪些规则）、**记忆注入**（你的长期记忆内容，主要来自一个叫 `MEMORY.md` 的文件，第 3 章会专门讲它）、**历史消息**（之前几轮的对话记录）、**当前消息**（你这一轮刚发的那句话）。你打的那句"你好"，只是最底下那一小层。看清这个反差，你就抓住了本章的核心——**所谓上下文工程，就是研究这四层怎么组装、怎么取舍**。不过在进 1.2 节之前，你刚读出的那个请求体里还藏着一个**四层装不下的真相**，值得当面算清楚。

&emsp;&emsp;细看 blob 读出的请求体顶层，除了装着①②层的 `system` 和装着③④层的 `messages`，还有一个四层图里没画的字段——`tools`。它装的是这个 agent 当前可用的全部工具的 schema（每个工具叫什么、参数长什么样的 JSON 描述，相当于一摞随身携带的"工具说明书"），模型就是靠读它才知道自己能调哪些工具、该怎么填参数。这块的体积常常出人意料：在我们的测试环境里，同一条请求的 `tools` 字段装了 32 个工具、约 5.5 万字符——**比 system 字段（约 3.9 万字符）还大**。再把 `messages` 拆开看，冲击更大：那"二十多条历史消息"里，你和 agent 真正互相说过的话合计只有一千字符上下，**约 98% 的体积是 `tool_result`（工具结果）**——agent 之前每次调工具（读文件、跑命令、查资料）拿回的原始输出，全都作为消息躺在历史里。把整个请求体摊开算账，比例是这样的：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>一条真实请求体的成分账单（测试环境实测，总量约 34 万字符）</font></p>
<div class="center">

| 成分 | 体积占比 | 它是什么 |
|------|----------|----------|
| `messages` 里的 `tool_result` | 约 72% | 历史工具调用的原始输出——context 的真正大头 |
| `tools` 字段（32 个工具 schema） | 约 16% | "工具说明书"，每一轮都全量随行 |
| `system` 字段 | 约 11% | 系统指令与记忆注入（四层里的①②层）所在 |
| 真正的对话文字（user / assistant 的 text） | **不足 1%** | 你以为的"对话历史" |

</div>

&emsp;&emsp;这张账单不推翻四层模型——四层依然是理解组装**过程**的最好抓手；`tools` 是四层之外随行的"第五块"，工具结果则是历史层的真实成分。你机器上的具体数字一定不同（取决于装了几个插件、这个会话聊了多久），但比例的量级是普遍的，结论也只有一句：<font color=red>真实 agent 的 context 是被工具吃掉的，不是被聊天吃掉的</font>。先把这个结论揣好——到 1.3 节讲压缩时你就会明白，为什么 OpenClaw 给"context 太长"设计的第一刀，只要工具结果还有可砍空间就先砍工具结果、而不是先动你的对话（没得砍时才直接压缩对话——1.3 节判断树里 `compact_only` 那个分支说的就是这种情况）。这笔账不用手算——下面这个 cell 帮你自动核：它从捕获档案库里挑出最近的主对话请求（自动跳过后台小请求），按四类成分算出**你自己机器上的账单**。

> 📌 **【运行前提】**：下面的真机核账 cell 依赖本节开头步骤 1 的捕获产物——先确认 `openclaw proxy sessions` 能列出 `eventCount > 0` 的会话再往下跑；捕获库不存在时 cell 会直接报错提醒你回去补跑。

In [38]:
# 真机核账：自动算出你自己的 context 成分账单（依赖本节开头步骤 1 的捕获产物）
import json, sqlite3, subprocess, os

db = os.path.expanduser("~/.openclaw/debug-proxy/capture.sqlite")
assert os.path.exists(db), "捕获库不存在——先回到本节开头跑步骤 1 的捕获 cell"

# 取最近 5 条发出方向的事件逐个读出，挑总字符量最大的那条——这正是前文"主对话请求
# 识别技巧"的代码化：后台摘要请求的请求体小得多，按体积挑自然落在主对话上
ids = [r[0] for r in sqlite3.connect(db).execute(
    "SELECT data_blob_id FROM capture_events WHERE direction='outbound' ORDER BY ts DESC LIMIT 5")]
best, best_body = None, None
for bid in ids:
    raw = subprocess.run(["openclaw", "proxy", "blob", "--id", bid],
                         capture_output=True, text=True).stdout
    start = raw.find("{")          # blob 输出前可能带一行头信息，从第一个 { 起才是 JSON
    if start < 0:
        continue
    try:
        body = json.loads(raw[start:])
    except json.JSONDecodeError:
        continue
    if best_body is None or len(raw) - start > best:
        best, best_body = len(raw) - start, body

assert best_body, "没读到合法请求体——确认步骤 1 已跑过且 eventCount > 0"

# 分四类算账：tool_result / tools / system / 真正的对话文字
size = lambda v: len(json.dumps(v, ensure_ascii=False))
tool_result = dialogue = 0
for m in best_body.get("messages", []):
    c = m["content"]
    if isinstance(c, str):
        dialogue += len(c)
        continue
    for block in c:
        if block.get("type") == "tool_result":
            tool_result += size(block)
        elif block.get("type") == "text":
            dialogue += len(block.get("text", ""))

rows = [("messages 里的 tool_result", tool_result),
        ("tools 字段（工具 schema）", size(best_body.get("tools", []))),
        ("system 字段", size(best_body.get("system", ""))),
        ("真正的对话文字（text）", dialogue)]
total = best or 1
print(f"=== 我的 context 成分账单（请求体总量 {total:,} 字符）===")
for name, n in rows:
    print(f"  {name:<28} {n:>9,} 字符  {n*100//total:>3}%")

=== 我的 context 成分账单（请求体总量 386,691 字符）===
  messages 里的 tool_result        274,133 字符   70%
  tools 字段（工具 schema）             55,502 字符   14%
  system 字段                       38,733 字符   10%
  真正的对话文字（text）                    6,646 字符    1%


&emsp;&emsp;跑完你会看到与上表同构的四行账单——具体数字和占比由你装的插件数量、这个会话聊了多久决定，但"工具吃大头、对话文字不足 1%"的形态几乎一定会复现。这一刻"context 被工具吃掉"就不再是课件转述的结论，而是你机器上的实测事实。那么 OpenClaw 内部是用什么机制把这些拼起来的？这就要钻进它的 `ContextEngine` 了，我们进 1.2 节。

> 📌 **【1.1 你刚拿到的】**：① 一条进程内捕获链——`OPENCLAW_DEBUG_PROXY_ENABLED` 开关 + `proxy sessions / blob` 命令，随时能把"实际发给模型的请求"捞出来；② context 四层结构（系统指令 / 记忆注入 / 历史 / 当前消息）+ 随行的 tools 第五块；③ 一笔成分账单——体积大头是工具结果和工具 schema，对话文字不足 1%。

### 1.2 ContextEngine 生命周期五阶段

&emsp;&emsp;刚才用 `proxy blob` 看到的那个四层请求体，不是凭空出现的——它是 OpenClaw 里一个叫 `ContextEngine`（上下文引擎）的组件，按一套生命周期方法一步步组装出来的。这一节我们就来拆它。这里要先讲清一条边界，免得你产生"整套引擎只有这固定五步"的误解：`ContextEngine` 接口（`src/context-engine/types.ts`）实际定义的方法**不止五个**——除了下面要讲的这五个，还有 `maintain` / `ingestBatch` / `afterTurn` / `onSubagentEnded` 等维护类方法。本课聚焦其中最能串起 context 组装主线的**五个核心环节**——也就是开头产物清单里说的"五个生命周期阶段"，后文有时简称"五阶段"，三种说法指同一件事：bootstrap → ingest → assemble → compact → prepareSubagentSpawn。我们先用一张图把这五个环节的全貌摆出来，再逐个讲，最后用 Python MVP 把核心的组装逻辑在你眼前重现一遍。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605142500001.png" width=70%></div>

&emsp;&emsp;我们顺着箭头一个一个看。第一个阶段 **bootstrap**——引擎初始化，这一步会把系统指令准备好，相当于给 agent 装上"它是谁、能做什么"的底座。第二个阶段 **ingest**（摄入）——每当有新消息进来（不管是你发的还是工具返回的），引擎就把它吃进消息列表，更新内部状态。第三个阶段 **assemble**（组装）——这是最核心的一步，引擎把系统指令、记忆注入、历史、当前消息这四层拼成一个完整的 context，这就是你刚才用 `proxy blob` 看到的那坨东西。在源码里，这个组装完成的时刻被标记为一个执行阶段叫 `context_assembled`（`src/agents/embedded-agent-runner/execution-phase.ts:10`，是执行阶段名数组里 `"context_assembled"` 这一项所在行，不是独立常量定义；它对应的字符串标签 `"context-assembled"` 在同文件 `:29`）。

&emsp;&emsp;这里要插一个关于"失败时怎么办"的重要区分，它是本节的一个诚实边界，必须讲清楚。assemble 这一步如果组装失败，OpenClaw 的兜底路径分两级：第一级是引擎层面会先尝试一个**备用引擎**（fallback engine）来兜底（`src/context-engine/registry.ts:834-839` 调用 `invokeFallbackContextEngineMethod`，换一个备用引擎再试一次）；只有连备用引擎也失败，错误才会被抛上去，由第二级——`src/agents/embedded-agent-runner/run/attempt.ts:2822-2826` 的 catch 接住，打一条 `log.warn` 日志后放弃这次 assemble 的增强、继续沿用 pipeline 里已经攒好的消息往下跑（要点是它本身只 `log.warn`、并不做显式的"消息回填赋值"，降级靠的是"不采用这次组装结果、自然沿用已有消息"）。也就是说，**assemble 失败有备用引擎兜底 + catch 降级两道防线，不会让整轮 turn 崩掉**。但接下来要讲的两个阶段就不是这样了。

&emsp;&emsp;第四个阶段 **compact**（压缩）——当对话历史长到快撑爆模型的 context 窗口时，引擎会把旧历史压缩成摘要，腾出空间。第五个阶段 **prepareSubagentSpawn**（为派生子 agent 做准备）——当主 agent 要 spawn 一个子 agent 时，引擎要为子 agent 准备一份独立的 context 快照，这正是通往第 2 章的那道门。这两个阶段的关键特性是：它们是 **fail-closed** 的。先说为什么——压缩历史和派生子 agent 都是**结构性操作**：压缩会重写整个消息列表、派生会建立一个新 agent 的初始上下文，这两件事一旦做到一半出错，留下的是一个"半成品状态"（消息列表损坏、或子 agent 拿到残缺 context），而**半成品比直接失败更危险**——它会让后续所有轮次都建立在错误的基础上、错误还难以追查。所以这两个阶段宁可"出错就当场失败、让这一轮干净地中止"，也不允许带着半成品继续。这就是 fail-closed 的含义：一旦出错，直接把错误抛出去（rethrow），**不做任何静默兜底**。这个行为在源码里有一处非常直白的判断（`src/context-engine/registry.ts:831`，是 `if (methodName === "compact" || methodName === "prepareSubagentSpawn")` 这一行判断；紧接着的 `:832` 行就是 `throw error`，`:833` 是闭合的 `}`）。

> &emsp;一句话交代实现归属：`ContextEngine` 的五阶段是**接口契约**，不等于"每个阶段都由引擎亲自干活"。当前默认槽位注册的 legacy 引擎（`src/context-engine/init.ts:6-7`，jsdoc 注释说明它总是被注册为默认 "legacy" 槽位的安全兜底）里，`ingest` 是 no-op——消息持久化由 SessionManager 负责（`src/context-engine/legacy.ts:34` 注释原文就是 "No-op"），`assemble` 是直通——真实组装由 `attempt.ts` 的 sanitize → validate → limit → repair 管线加系统指令构建器完成（`legacy.ts:47` 注释原文就是 "Pass-through"）；`compact` 同样是**调度入口**（fail-closed 是 registry 包装层的策略），内置压缩**实现**经 `src/context-engine/delegate.ts` 委托给 `embedded-agent-runner` 执行——它背后的运行时设计（四路路由、successor transcript、安全超时），1.3 节末尾的「进阶」盒子给你一张路标图。

> **【常见误区】**：以为 ContextEngine 整套引擎都是"出错就崩"的 fail-closed，或者反过来以为整套都能优雅降级。真相是分阶段的：**只有 compact 和 prepareSubagentSpawn 这两个阶段是 fail-closed**（registry.ts:831 那行判断写得清清楚楚，只点了这两个方法名），而 assemble 失败有自己独立的 catch 降级（attempt.ts:2822-2826）。后果是如果你混为一谈，排查"为什么这次 turn 直接失败了"时会找错方向。正确理解：compact / prepareSubagentSpawn 出错 = 当前这一轮直接失败（因为压缩或派生子 agent 是结构性操作，半成品比失败更危险），其他阶段出错则尽量兜住继续跑。排查方法：看错误是不是发生在压缩或 spawn 子 agent 的环节——是，那它就该按设计直接失败。

&emsp;&emsp;讲完这五个阶段，光听描述还是抽象。下面我们用一段 Python MVP 把它最核心的 assemble 四层组装 + compact fail-closed 这两件事在你眼前跑一遍。需要特别强调：**下面这段是 Python 最小重现，不是 OpenClaw 的真实源码**——真实的 `ContextEngine` 是 TypeScript 写的，有错误处理、流式、并发等大量我们不展开的细节。这个 MVP 的唯一目的，是让你亲手验证"四层是怎么拼的、fail-closed 是什么手感"。

In [39]:
# [注意] Python 最小重现，非 OpenClaw 真实源码（真实实现是 TypeScript，此处只重现机制本质）
from dataclasses import dataclass, field
from typing import List, Dict, Optional

@dataclass
class Message:
    """一条对话消息。role 取 user/assistant/system；content 是文本内容。"""
    role: str
    content: str

class MiniContextEngine:
    """重现 ContextEngine 的核心：四层 assemble 组装 + compact fail-closed。

    构造参数：
        system_prompt: 系统指令层内容（对应 bootstrap 阶段准备好的底座）
        memory_injection: 记忆注入层内容（对应 MEMORY.md 注入，第 3 章详讲）
    """

    def __init__(self, system_prompt: str, memory_injection: str):
        self.system_prompt = system_prompt          # 第①层：系统指令
        self.memory_injection = memory_injection      # 第②层：记忆注入
        self.history: List[Message] = []              # 第③层来源：累积的消息
        self.max_messages = 6                         # 触发 compact 的消息数阈值

    def ingest(self, msg: Message) -> None:
        """ingest 阶段：新消息进来，吃进消息列表。"""
        self.history.append(msg)

    def assemble(self) -> Dict[str, object]:
        """assemble 阶段：把四层拼成发给模型的完整 context。

        返回一个 dict，键固定为四层：system_prompt / memory_injection / history / current。
        这正是 proxy blob 里能看到的那个四层结构。
        """
        # 历史层 = 除最后一条外的所有消息（最后一条是"当前消息"，单独成层）
        history = [m.content for m in self.history[:-1]] if len(self.history) > 1 else []
        current = self.history[-1].content if self.history else None
        return {
            "system_prompt": self.system_prompt,     # ①
            "memory_injection": self.memory_injection,  # ②
            "history": history,                       # ③
            "current": current,                       # ④
        }

    def compact(self, simulate_error: bool = False) -> Dict:
        """compact 阶段（fail-closed）：历史超阈值时把旧历史压成摘要。

        fail-closed 的真实含义（registry.ts:831-832）：compact 在**执行过程中**
        一旦出错就直接 rethrow、绝不静默兜底（半成品的消息列表比直接失败更危险）。
        注意：未达阈值时 compact 正常返回 compacted=False，并**不抛错**——
        真实源码也是这样，不要把"没到阈值"和"执行出错"混为一谈。
        """
        # 未超阈值：正常返回"未压缩"，不触发压缩动作，也不抛错
        if len(self.history) <= self.max_messages:
            return {"before": len(self.history), "after": len(self.history), "compacted": False}
        before = len(self.history)
        # fail-closed：压缩执行中一旦出错，直接抛出、不静默兜底（这里用 simulate_error 模拟内部错误）
        if simulate_error:
            raise RuntimeError("compact 执行中出错——按 fail-closed 设计直接抛出，不静默兜底")
        # 把除最近 2 条外的历史压成一条摘要消息
        # 注意摘要的角色是 user 不是 system——真实源码重建 context 时，压缩摘要就是一条
        # role="user" 的消息（packages/agent-core/src/harness/messages.ts:159-161，正文还会
        # 包上一层"此前历史已压缩为以下摘要"的英文说明前缀），它从不进 system prompt
        summary = Message("user", f"[摘要] 此前 {before - 2} 条对话已压缩为本条")
        self.history = [summary] + self.history[-2:]
        return {"before": before, "after": len(self.history), "compacted": True}

# 跑一遍：组装四层
engine = MiniContextEngine(
    system_prompt="你是一个能调用工具的个人助手",
    memory_injection="MEMORY.md: 用户叫小明，常用 Python",
)
engine.ingest(Message("user", "你好"))
engine.ingest(Message("assistant", "你好小明"))
engine.ingest(Message("user", "今天北京天气怎么样"))

context = engine.assemble()
print("=== assemble 组装出的四层 context ===")
for layer, value in context.items():
    print(f"  [{layer}] -> {value}")

=== assemble 组装出的四层 context ===
  [system_prompt] -> 你是一个能调用工具的个人助手
  [memory_injection] -> MEMORY.md: 用户叫小明，常用 Python
  [history] -> ['你好', '你好小明']
  [current] -> 今天北京天气怎么样


&emsp;&emsp;这段代码的核心是 `assemble()` 方法——它把四层用一个 dict 返回，键名 `system_prompt / memory_injection / history / current` 一一对应我们在 `proxy blob` 里看到的四层（注：真实源码 `assemble` 返回的是统一的 `messages` 数组，这里的"四层 dict"是我们为对照 proxy blob 可观测结构所做的教学分层，不是源码的实际数据结构）。你跑完会看到这四层被清清楚楚地打印出来，"当前消息"是你最后发的那句、"历史"是前面几轮、上面压着系统指令和记忆注入。`compact()` 方法重现了 fail-closed 的真实手感：未达阈值时它正常返回 `compacted=False`（**不抛错**），但一旦压缩执行中出错（用 `simulate_error` 模拟），就直接 `raise` 抛出、绝不静默兜底——这跟真实源码 registry.ts:831-832 那个"compact 出错就 throw、不 fallback"的行为是同构的（注意真实的 fail-closed 指的是"执行出错就抛"，不是"没到阈值就抛"）。下面这个 cell 是 Tier 1 验证，它独立地把 assemble 的四层完整性和 compact 的 fail-closed 行为都断言一遍。

In [40]:
# Tier 1 验证：独立检验 assemble 四层结构 + compact fail-closed 行为

# 验证点 1：assemble 必须输出固定的四层键，缺一不可
assert set(context.keys()) == {"system_prompt", "memory_injection", "history", "current"}, \
    "assemble 必须输出完整四层"

# 验证点 2：当前消息层是最后发的那句，历史层是它之前的
assert context["current"] == "今天北京天气怎么样"
assert context["history"] == ["你好", "你好小明"]

# 验证点 3a：未超阈值时 compact 正常返回 compacted=False，绝不抛错
#   （修正真实语义：fail-closed 不是"没到阈值就抛"，未达阈值是正常的不压缩）
r = engine.compact()
assert r["compacted"] is False, "未超阈值时 compact 应正常返回未压缩，不抛错"

# 验证点 3b：fail-closed——压缩执行中出错时必须 rethrow，不静默兜底
for i in range(6):
    engine.ingest(Message("user", f"填充消息{i}"))   # 先撑过阈值，让 compact 进入压缩路径
fail_closed_triggered = False
try:
    engine.compact(simulate_error=True)
except RuntimeError as e:
    fail_closed_triggered = True
    print(f"[fail-closed 生效] {e}")
assert fail_closed_triggered, "compact 执行出错时必须按 fail-closed rethrow，不静默兜底"

print("\nTier 1 全部通过：四层组装正确 + compact fail-closed 行为符合预期")

[fail-closed 生效] compact 执行中出错——按 fail-closed 设计直接抛出，不静默兜底

Tier 1 全部通过：四层组装正确 + compact fail-closed 行为符合预期


&emsp;&emsp;这个验证 cell 把三件事钉死了：assemble 的四层键必须完整、当前消息和历史分层必须正确、compact 的 fail-closed 行为正确——未达阈值时正常返回 `compacted=False`（不抛错），而压缩执行中出错时直接 rethrow。其中第三点最关键——它用 `try/except` 捕获到了压缩执行出错时抛出的 `RuntimeError`，证明我们的 MVP 确实在"执行出错时直接失败"而不是"静默兜底"，这跟真实 OpenClaw 里 compact 的 fail-closed 设计（registry.ts:831-832 出错就 throw、不 fallback）是一致的。看完静态结构，我们把刚才源码讲的五阶段和 proxy blob 看到的四层做一次对账，让两边对齐。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>ContextEngine 五阶段 vs proxy blob 四层结构对账</font></p>
<div class="center">

| ContextEngine 阶段 | 在做什么 | 对应 proxy blob 里看到的 | 失败行为 | 源码锚点 |
|--------------------|----------|--------------------------|----------|----------|
| bootstrap | 初始化引擎、备好系统指令 | 第①层 系统指令 | —— | —— |
| ingest | 新消息进入消息列表 | 为③④层备料 | —— | —— |
| assemble | 拼成四层完整 context | 整个四层请求体 | 可降级（两道防线：先换备用引擎重试，仍失败才 catch 降级——log.warn 后沿用已有 pipeline 消息）| registry.ts:834-839 / attempt.ts:2822-2826 / execution-phase.ts:10,29 |
| compact | 历史逼近上限时压缩 | 历史层被压短 | **fail-closed**（rethrow）| registry.ts:831（判断）/ :832（throw）|
| prepareSubagentSpawn | 为子 agent 备独立 context | （第 2 章展开）| **fail-closed**（rethrow）| registry.ts:831 / context-engine/types.ts:363（方法签名）|

</div>

&emsp;&emsp;这张表把"源码里的五个阶段"和"proxy blob 里看到的四层"两套视角对齐了——它们其实是同一件事的内外两面：五阶段是组装的**过程**，四层是组装的**结果**。表里还顺手标了每个阶段的失败行为，你能一眼看出 fail-closed 只落在 compact 和 prepareSubagentSpawn 两行。其中 compact 这一阶段最有意思——它解决的是"对话太长怎么办"这个真实痛点，值得单独拎出来细讲。我们进 1.3 节。

> 📌 **【1.2 你刚拿到的】**：① ContextEngine 五个生命周期阶段（bootstrap → ingest → assemble → compact → prepareSubagentSpawn）；② fail-closed 只覆盖 compact 和 prepareSubagentSpawn 两个阶段，不代表整套引擎都这么严格；③ 一段四层组装的可跑 MVP——context 怎么拼出来你已经亲手重现过。

### 1.3 compact 上下文压缩

&emsp;&emsp;为什么需要 compact？因为大模型的 context 窗口是有限的。对话越聊越长，历史消息越堆越多，迟早会逼近窗口上限——再不处理，要么报错，要么把最早的关键信息挤掉。compact 干的事就是：在快撑爆之前，把旧的历史压缩成一段摘要，用一小段"浓缩版"换回大片空间，让对话能继续。这是上下文工程里非常核心的一招。

&emsp;&emsp;顺手把一个本章反复出现、却容易被当成"天上掉下来"的数字讲清楚——这个"窗口上限"本身从哪来？OpenClaw 用一个三级优先级来解析它（`src/agents/context-window-guard.ts:26`，`resolveContextWindowInfo` 函数定义行）：先看你配置文件里 `models.providers.*.models[]` 条目的 `contextTokens` / `contextWindow` 字段，没配就用模型元数据自带的窗口值，再没有才落到保守默认值 200_000（`src/agents/defaults.ts:6`，常量 `DEFAULT_CONTEXT_TOKENS`）；解析出来的值还可能被 `agents.defaults.contextTokens` 进一步向下封顶。另外，压缩时要预留的空间有一个默认 20K 的下限，但它会按小窗口动态压低——`src/agents/agent-settings.ts:85-89` 的注释把原因写得很直白：不压低的话，16K 窗口的小模型（比如本地 Ollama）会被 20K 预留吃光 prompt 预算，每一轮都被判溢出、陷入无限压缩循环。说清了上限从哪来，再看压缩本身。下面这张图把"压缩前"和"压缩后"的状态摆在一起对比，你一眼就能看出 compact 到底腾出了多少空间。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124750797.png" width=70%></div>

&emsp;&emsp;OpenClaw 的 compact 是可观测的——它在压缩前后都会发出事件和 hook，让外部代码能挂上来感知压缩的发生（注意这是给 hook/订阅方用的机制，正常路径下它们不直接打进文本日志；本节稍后真机观测时，我们看的是另外两行真实日志）。具体来说有两个 hook：`compact:before`（压缩前触发，`src/agents/embedded-agent-runner/compaction-hooks.ts:191`，是 `createInternalHookEvent` 创建该 hook 事件的调用行）和 `compact:after`（压缩后触发，同文件 `:286`）。同时还有两个事件类型：`compaction_start` 和 `compaction_end`（在 `src/agents/embedded-agent-subscribe.handlers.compaction.ts:7`/`:8` 以 `type ... = Extract<AgentSessionEvent, { type: "compaction_start" }>` 这样的类型别名形式收窄出来——事件名字符串本身来自 `AgentSessionEvent` 这个联合类型，这两行只是把它们各自提取成独立的 TS 类型）。我们先用 MVP 把"历史增长 → 触发压缩 → 历史变短"这个核心过程跑一遍，再去真机上观测它。

&emsp;&emsp;下面这段 MVP 在 1.2 节那个 `MiniContextEngine` 的基础上，模拟一段不断增长的对话，直到超过阈值触发 compact，并打印压缩前后的消息数变化。同样强调：这是 Python 最小重现，不是真实源码——真实的压缩会调用模型来生成摘要，这里我们用一条占位摘要消息代替，重点是让你看清"压缩后消息数确实变少了"这个核心现象。

In [41]:
# [注意] Python 最小重现，非 OpenClaw 真实源码（真实压缩会调模型生成摘要，此处用占位摘要）

# 复用 1.2 节的 MiniContextEngine，模拟对话增长直到触发 compact
demo_engine = MiniContextEngine(
    system_prompt="你是一个能调用工具的个人助手",
    memory_injection="MEMORY.md: 用户叫小明",
)

# 模拟一段会逼近上限的长对话（阈值 max_messages=6）
turns = [
    ("user", "帮我规划一次三天的杭州旅行"),
    ("assistant", "好的，第一天建议西湖"),
    ("user", "第二天呢"),
    ("assistant", "第二天灵隐寺"),
    ("user", "第三天"),
    ("assistant", "第三天乌镇"),
    ("user", "住宿有推荐吗"),  # 第 7 条，超过阈值 6，该触发 compact 了
]

compaction_events = []  # 记录事件序列，模拟 compaction_start / compaction_end

for role, text in turns:
    demo_engine.ingest(Message(role, text))
    # 每次摄入后检查是否超阈值——超了就触发压缩（模拟真实的"逼近上限时压缩"）
    if len(demo_engine.history) > demo_engine.max_messages:
        compaction_events.append("compaction_start")   # 对应源码事件 :7
        result = demo_engine.compact()                  # 执行压缩
        compaction_events.append("compaction_end")      # 对应源码事件 :8
        print(f"触发压缩：消息数 {result['before']} -> {result['after']}")

print("\n压缩后的历史消息：")
for m in demo_engine.history:
    print(f"  [{m.role}] {m.content}")
print(f"\n事件序列：{compaction_events}")

触发压缩：消息数 7 -> 3

压缩后的历史消息：
  [user] [摘要] 此前 5 条对话已压缩为本条
  [assistant] 第三天乌镇
  [user] 住宿有推荐吗

事件序列：['compaction_start', 'compaction_end']


&emsp;&emsp;这段代码的关键在那个 `if len(...) > max_messages` 的判断——它模拟了真实 compact 的触发条件"逼近上限"。一旦第 7 条消息进来超过阈值 6，就依次发出 `compaction_start` 事件、执行压缩、再发出 `compaction_end` 事件，这个事件对偶正好对应源码里的 `compaction_start`/`compaction_end`（handlers.compaction.ts:7,8）。你跑完会看到消息数从 7 掉到 3（一条摘要 + 最近两条），历史里最早那几条旅行规划被压成了一句"[摘要]"。这个 MVP 已经内含了 Tier 1 验证的全部要素，我们把断言显式补上确认事件顺序和压缩效果。

In [42]:
# Tier 1 验证：压缩前后消息数 + 事件顺序

# 验证点 1：压缩确实让消息数变少了
assert len(demo_engine.history) < len(turns), "compact 后消息数必须小于压缩前"

# 验证点 2：事件必须成对且 start 在 end 之前（对应源码 compaction_start/end）
assert compaction_events == ["compaction_start", "compaction_end"], \
    "事件序列必须是 start 先于 end"

# 验证点 3：压缩后第一条应是摘要，保留了最近 2 条原始消息
assert demo_engine.history[0].content.startswith("[摘要]")
assert demo_engine.history[-1].content == "住宿有推荐吗"

# 验证点 4：摘要的角色必须是 user——真实源码就是 user（messages.ts:159-161），不是 system
assert demo_engine.history[0].role == "user", "压缩摘要是 user 角色，不进 system"

print("Tier 1 通过：压缩生效 + 事件顺序正确 + 摘要+最近2条结构正确 + 摘要角色是 user")

Tier 1 通过：压缩生效 + 事件顺序正确 + 摘要+最近2条结构正确 + 摘要角色是 user


&emsp;&emsp;这个验证确认了三件事：压缩真的缩短了消息列表、事件严格按 start→end 的顺序发出、压缩后保留了"一条摘要 + 最近两条原文"的结构。理解了机制，我们到真机上去观测它。下面这条命令以 debug 日志级别**无头**跑一轮对话（`agent --local` 自动退出，适合 notebook），并把日志喂给配套的降噪高亮脚本 `trace_pretty.py`（教学用本地工具，已按真机日志校准）。重点看它高亮出来的两行"黄金观测行"：一行是 `[context-diag]`——每轮发模型前的"上下文体检"，它的 `roleCounts` 字段里如果出现 `compactionSummary`，说明你的历史里已经躺着一条压缩摘要，这是**压缩发生过的活证据**；另一行是 `[context-overflow-precheck]`——压缩判定现场，`route=fits` 表示这一轮还装得下、不用压，一旦历史逼近窗口上限，`route` 就会变成压缩/截断的组合（这正是本节末尾进阶盒子里那张"四路路由"表的运行时入口）。

In [43]:
# 真机观测 compact：debug 日志 + trace_pretty.py 降噪高亮
#   trace_pretty.py 是配套教学的日志降噪脚本（已按真机 debug 日志校准），只高亮教学关注行
#   脚本就在本课件同目录下，随课件一起发——notebook 在本目录运行即可直接管道调用
#   agent --local 为无头单轮对话、自动退出；TUI 全屏交互观演可在系统终端另行体验
#   注意：首行有 ! 前缀，管道续行不重复加 !
!openclaw --log-level debug agent --local --agent main --message "请一句话回应：收到" 2>&1 \
  | python3 trace_pretty.py

[01] 🧩 上下文体检  [agent/embedded] [context-diag] pre-prompt: sessionKey=agent:main:main messages=15 roleCounts=assistant:8,toolResult:5,user:2 historyTextChars=3165 maxMessageTextChars=856 historyImageBlocks=0 systemPromptChars=38568 promptChars=9 promptImages=0 provider=minmax/MiniMax-M3 sessionFile=/Users/mac/.openclaw/agents/main/sessions/d72663a8-8d3a-463b-8fbd-8b1a03229dbd.jsonl
[02] 🗜 压缩路由判定  [agent/embedded] [context-overflow-precheck] pre-prompt check sessionKey=agent:main:main provider=minmax/MiniMax-M3 route=fits estimatedPromptTokens=18304 pressureSource=transcript_estimate promptBudgetBeforeReserve=983616 overflowTokens=0 toolResultReducibleChars=0 reserveTokens=16384 effectiveReserveTokens=16384 contextTokenBudget=1000000 messages=15 unwindowedMessages=15 sessionFile=/Users/mac/.openclaw/agents/main/sessions/d72663a8-8d3a-463b-8fbd-8b1a03229dbd.jsonl


&emsp;&emsp;跑通后你会看到形如 `route=fits estimatedPromptTokens=... overflowTokens=0` 的判定行——这一轮装得下，所以没压；如果你的会话历史里已经发生过压缩，`[context-diag]` 行的 `roleCounts` 里还会出现 `compactionSummary:1`。想亲眼看到压缩**真正发生**（`route` 离开 `fits`），就需要制造一段足够长的对话逼近窗口上限。这里有个现场常见的卡点要提醒你。

> **【踩坑预警】**：compact 在测试时可能死活触发不了——因为触发条件是"逼近 context 窗口上限"，而具体阈值依赖配置，对话不够长就到不了。后果是你可能干等半天什么都看不到，以为代码坏了。正确做法：触发不了就**直接讲机制 + 看预录截图**，并且要意识到——"触发条件本身就是上下文工程的一部分"，什么时候该压、压多少，这正是工程上要调的参数。排查方法：如果你想稳定触发，最直接的是发一条 `/compact` 命令手动逼出一次压缩（它是注册在命令表里的正式命令，`src/auto-reply/commands-registry.shared.ts:720` 是它的 textAlias 定义行）；也可以调低 compact 阈值配置（具体 config key 需查你的 OpenClaw 配置文档），或者一次性灌入一大段长文本。另外别忘了 compact 是 fail-closed 的——如果压缩这一步真出错了，它会让当前这一轮直接失败，而不是偷偷继续，这是设计使然。

&emsp;&emsp;到这里，compact 的核心机制我们就讲透了：它是上下文工程里"用空间换延续"的核心手段，可观测（有事件和 hook）、有触发条件（逼近上限）、且是 fail-closed（出错即败）。还要补一句免得你以为入口只有一个：自动预检只是走到 compact 的路径之一。为什么要把入口数齐？——日后你发现 compact 在你没预料的时机触发了，能准确判断它是从哪条路进来的，而不是一脸懵。把全部触发源数齐其实有**六个**——①发 prompt 前的自动预检（就是上面真机观测的 `[context-overflow-precheck]` 那行）；②一轮还没结束、工具循环**中途**的 mid-turn 预检——工具结果一条条涌进来时也会做同样的溢出判定，不用等到下次发 prompt（`src/agents/embedded-agent-runner/tool-result-context-guard.ts:494`，`midTurnPrecheck` 启用判断行）；③模型真的报了**溢出错误**之后的恢复压缩——压完自动重试这一轮（`src/agents/embedded-agent-runner/run.ts:2135`，溢出恢复路径里发起 compact 的调用行）；④ **LLM 超时**且这轮 prompt 的 token 占比超过 65% 时的恢复压缩——日志行以 `[timeout-compaction]` 开头（占比判断在 `run.ts:1881`）；⑤刚才踩坑预警里说的手动 `/compact` 命令；⑥turn 结束后被推迟到后台维护的 deferred compaction（原因常量 `"deferred to background context-engine maintenance"` 定义在 `src/agents/embedded-agent-runner/compact-reasons.ts:6-7`，排队成功后的延迟返回路径在 `compact.queued.ts:143` 真实消费它（`compacted: false` + 该原因常量，表示本轮已委托给后台维护任务）——常量有定义、执行链也有消费，不是摆设）。六条路殊途同归，最终都走到同一套压缩实现。不过在收口之前，还要补上 1.2 节"实现归属"处留下的那个问题——真实运行时的 compact，远比我们这个 MVP 复杂。

&emsp;&emsp;接下来是连续两块给工程背景同学的进阶加餐（真实 compact 的复杂度、context 的四道防线）——零基础同学可以直接跳到 1.4 节，主线一点不受影响。

**进阶：真实 compact 比我们的 MVP 复杂在哪**

> 📌 **这是给有工程经验同学的加餐**——零基础同学可直接跳到 1.4 节，完全不影响主线。

&emsp;&emsp;我们这个 MVP 为了讲清核心，把 compact 简化成了"超过阈值就压一刀"。但真实的 OpenClaw 运行时（也就是 1.2 节说过的、真正干活的 `embedded-agent-runner` 那套实现）在"要不要压、怎么压、压完怎么收尾"这三件事上都做了精细工程。下面这张表挑出四个最有代表性的真实设计，让你知道这背后还有多深的水，日后真要去读这块源码时手里也有张路标图。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>MVP 简化 vs 真实 compact 运行时的四个核心设计</font></p>
<div class="center">

| 真实设计 | 它在做什么（一句话）| 比我们的 MVP 多在哪 | 源码锚点 |
|----------|---------------------|---------------------|----------|
| 四路路由 | 压缩前先估 token 压力，按源码判断树给出四种路由：`fits`（不溢出，什么都不做）/ `compact_only`（工具结果无可截空间，只能走 LLM 压缩）/ `truncate_tool_results_only`（可截空间足够覆盖溢出，纯截断、不调 LLM）/ `compact_then_truncate`（可截但不够，压缩+截断双管齐下） | MVP 一律走"压缩"；真实按"工具结果可截空间"走最省的那条路——是**成本分级** | `src/agents/embedded-agent-runner/run/preemptive-compaction.ts:301-310` |
| `SAFETY_MARGIN=1.2` | token 估算用启发式字符系数，且**不是一刀切的 chars/4**——普通文本 chars/4、工具结果 chars/2、JSON 载荷 chars/3（三个系数常量在 `preemptive-compaction.ts:16-18`），图片块固定按 2000 tokens 计（同文件 `:21`）；估算会低估，于是统一乘 1.2 留 20% 余量，compaction.ts 与预检路由共用这同一个常量保证口径一致 | MVP 用精确的消息条数计数；真实里 token 数无法精确预知，只能"估算 + 安全余量" | `src/agents/compaction.ts:21` |
| successor transcript | 压缩后不在原文件上改，而是把 compaction 点之后的消息按原顺序写进一个新 session 文件，让新文件的消息前缀尽量与旧文件一致；注意这是**可配置行为**——配置项 `agents.defaults.compaction.truncateAfterCompaction` 为 true 才轮转新文件（开关判断函数 `shouldRotateCompactionTranscript` 在同文件 `:28-30`） | MVP 直接改 `history` 列表；真实要照顾 prompt cache 命中（这条"缓存暗线"在下一个进阶盒子末尾展开讲） | `src/agents/embedded-agent-runner/compaction-successor-transcript.ts:32` |
| 900s 安全超时 | 压缩本身要调 LLM、可能卡死，于是包一层 900 秒超时 + abort，超时就按 fail-closed 干净中止 | MVP 是同步秒回；真实是异步调模型，必须有超时兜底 | `src/agents/embedded-agent-runner/compaction-safety-timeout.ts:5` |

</div>

&emsp;&emsp;四个设计里，「四路路由」的判断逻辑最值得展开看一眼——下面这张图把源码的判断顺序画了出来，你可以对照表格第一行读。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124750764.png" width=70%></div>

&emsp;&emsp;这四个设计有个共同的味道——<font color=red>在真实工程里，"压缩"从来不是一个动作，而是一条带成本分级（四路路由）、安全余量（`1.2`）、缓存保护（successor transcript）和故障兜底（900s 超时）的完整链路</font>。你现在不需要记住每个细节，只要心里有数：今天的 MVP 帮你抓住了 compact 的"骨架"，而这张表让你瞥见了它的"血肉"。顺着"成本分级"的思路再往上抬一层，还有最后一块进阶拼图值得放上来——把视野从 compact 单点拉开，你会发现它其实排在一整条防线的**最后**。

**进阶：压缩是最后一招——context 的四道防线**

> 📌 **同样是给有工程经验同学的加餐**——零基础同学可直接跳到 1.4 节，不影响主线。

&emsp;&emsp;还记得 1.1 节末尾那张成分账单吗——context 的大头是工具结果。顺着这个事实再读 OpenClaw 的源码，你会发现它对"context 太长"的治理根本不是 compact 一个动作，而是一条**从源头到兜底的流水线**：工具输出从产生到最终发给模型，沿途要过四道防线，每一道都在替后面的省力；compact 只是其中最后、也最贵的一招（要花一次真实的 LLM 调用）。我们先用一张图把这条流水线立起来，再用表格逐道防线给出源码锚点。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124750925.png" width=70%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>context 的四道防线：按出手时机从早到晚</font></p>
<div class="center">

| 防线 | 出手时机 | 代表机制 | 源码锚点 |
|------|----------|----------|----------|
| L0 源头限流 | 工具刚执行完、字节还没进 context | bash 输出超 2000 行或 50KB 只保**尾部**（报错通常在尾部），完整输出另存临时文件供按需回查；Read 读文件超限则保**头部**、并附"从第 N 行继续读"的续读提示 | `src/agents/sessions/tools/truncate.ts:11-12`（两个默认上限常量）/ `src/agents/sessions/bash-executor.ts:136`（bash 侧 `truncateTail` 调用行）/ `src/agents/sessions/tools/read.ts:337`（Read 侧 `truncateHead` 调用行） |
| L1 落盘封顶 | 工具结果写进会话转录文件之前 | 每条工具结果按模型窗口大小分档封顶——16K / 32K / 64K 字符三档，超出就地截断再落盘 | `src/agents/embedded-agent-runner/tool-result-truncation.ts:40-42`（三档常量定义）/ `src/agents/session-tool-result-guard.ts:39`（落盘前执行截断的函数） |
| L2 组装瘦身 | 每轮发模型前的常规整理（没逼近上限也做） | 超过最近 3 个完成轮次的旧图片替换成一行占位文本（图片是 token 大户——上一张表 `SAFETY_MARGIN` 行讲过，估算时一个图片块按 2000 tokens 计）；context-pruning 扩展对旧工具结果做两段式处理——先**软裁**（保头尾 + 一行说明），占用还高就**硬清**（整条内容换成占位符） | `src/agents/embedded-agent-runner/run/history-image-prune.ts:23,83`（保留轮次常量 / 剪除函数）/ `src/agents/agent-hooks/context-pruning/pruner.ts:287`（两段式修剪函数） |
| L3 压力触发 | 估算出 prompt 要溢出时 | 上面那棵四路路由判断树——能纯截断就不调 LLM，逼到没路才走 compact | `src/agents/embedded-agent-runner/run/preemptive-compaction.ts:301-310`（路由选择逻辑） |

</div>

&emsp;&emsp;四道防线"何时出手"看清了，还差"刀怎么下"——每道闸对字节到底做了什么、截掉的地方变成什么样。下面这张图把工具压缩的四把刀的**字节级前后效果**排在一起看：特别注意 L0 那对方向不对称的设计（bash 保**尾**、Read 保**头**——一个是报错在尾部、一个是结构在开头），以及四把刀的共同底线——**只截不改**，截掉处永远用一行说明占位，从不回头改写已有字节。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605141020161.png" width=70%></div>

&emsp;&emsp;表格和图看到这里还只是"读到"。四道防线最适合用代码"摸到"——下面这段 MVP 把一条 8 万字符的"灾难级"bash 输出（3000 多行刷屏日志、报错信息埋在最后一行）依次推过四道闸，每道闸打印它做了什么。照例强调：**这是 Python 最小重现，不是 OpenClaw 真实源码**——四个阈值数字（50KB / 16K / 0.3 / 0.5）和两段英文占位文案都取自前面验证过的源码常量，但逻辑只保留机制本质（真实的 L2 还要保护最近 3 个 assistant 轮次、按工具名单过滤可裁项等，这里都省去）。

In [44]:
# [注意] Python 最小重现，非 OpenClaw 真实源码（阈值与占位文案取自源码常量，逻辑只保留机制本质）

TRUNC_SUFFIX = "[... {n} more characters truncated; rerun with narrower args if needed]"

def l0_bash_truncate_tail(output: str, max_bytes: int = 50 * 1024) -> str:
    """L0 源头限流：bash 输出超限只保【尾部】（报错通常堆在末尾），真实源码会把完整输出另存临时文件。"""
    if len(output) <= max_bytes:
        return output
    return f"[... {len(output) - max_bytes} chars truncated, full output saved to file]\n" + output[-max_bytes:]

def l1_cap_on_write(tool_result: str, max_chars: int = 16_000) -> str:
    """L1 落盘封顶：每条工具结果写入会话文件前封顶（默认 16K 档），超出截断并追加"给模型看的"后缀。"""
    if len(tool_result) <= max_chars:
        return tool_result
    return tool_result[:max_chars] + TRUNC_SUFFIX.format(n=len(tool_result) - max_chars)

def l2_prune(tool_results: list, window_chars: int, cache_expired: bool) -> list:
    """L2 组装瘦身（context-pruning 两段式）：cache 还热绝不动手；过期后先软裁，占用仍高再硬清最老的。"""
    if not cache_expired:                                  # cache-ttl 模式：prompt cache 热着 = 零动作
        return tool_results
    out = list(tool_results)
    if sum(map(len, out)) / window_chars >= 0.3:           # softTrimRatio=0.3 → 软裁（最近一条不动）
        out = [t[:1500] + "\n...\n" + t[-1500:] if len(t) > 4_000 else t for t in out[:-1]] + out[-1:]
    if sum(map(len, out)) / window_chars >= 0.5:           # hardClearRatio=0.5 → 硬清最老一条
        out[0] = "[Old tool result content cleared]"
    return out

def l3_route(overflow_tokens: int, reducible_chars: int, threshold_chars: int) -> str:
    """L3 压力触发：四路路由，判断顺序逐行对应 preemptive-compaction.ts:301-310。"""
    if overflow_tokens <= 0:
        return "fits"
    if reducible_chars <= 0:
        return "compact_only"
    if reducible_chars >= threshold_chars:
        return "truncate_tool_results_only"
    return "compact_then_truncate"

# 一条 8 万字符的"灾难级"bash 输出：前面全是刷屏日志，报错信息埋在最后一行
giant = "compiling module xyz...\n" * 3330 + "Error: missing semicolon at line 42"
print(f"原始工具输出: {len(giant):,} 字符（报错信息在最后一行）\n")

after_l0 = l0_bash_truncate_tail(giant)
print(f"过 L0 源头限流: {len(after_l0):,} 字符（保尾，报错还在: {'Error' in after_l0[-100:]}）")

after_l1 = l1_cap_on_write(after_l0)
print(f"过 L1 落盘封顶: {len(after_l1):,} 字符（末尾追加修复指引后缀）")
print(f"   后缀实物: ...{after_l1[-78:]}")

history = ["旧工具结果A" * 2_000, "旧工具结果B" * 2_000, after_l1]   # 三条工具结果躺在历史里
hot  = l2_prune(history, window_chars=40_000, cache_expired=False)
cold = l2_prune(history, window_chars=40_000, cache_expired=True)
print(f"\n过 L2 组装瘦身（cache 还热）: {[f'{len(t):,}' for t in hot]} ← 零动作，一个字节没动")
print(f"过 L2 组装瘦身（cache 过期）: {[f'{len(t):,}' for t in cold]} ← 软裁两条旧的 + 硬清最老的")
print(f"   硬清实物: {cold[0][:60]}")

print("\n过 L3 压力触发（四组参数各走一条路）:")
for ov, red, th in [(0, 9_999, 4_000), (5_000, 0, 4_000), (1_000, 9_999, 4_000), (1_000, 2_000, 4_000)]:
    print(f"   overflow={ov:>5}  可截={red:>5}  阈值={th}  → route={l3_route(ov, red, th)}")

原始工具输出: 79,955 字符（报错信息在最后一行）

过 L0 源头限流: 51,255 字符（保尾，报错还在: True）
过 L1 落盘封顶: 16,073 字符（末尾追加修复指引后缀）
   后缀实物: ...ng mo[... 35255 more characters truncated; rerun with narrower args if needed]

过 L2 组装瘦身（cache 还热）: ['12,000', '12,000', '16,073'] ← 零动作，一个字节没动
过 L2 组装瘦身（cache 过期）: ['33', '3,005', '16,073'] ← 软裁两条旧的 + 硬清最老的
   硬清实物: [Old tool result content cleared]

过 L3 压力触发（四组参数各走一条路）:
   overflow=    0  可截= 9999  阈值=4000  → route=fits
   overflow= 5000  可截=    0  阈值=4000  → route=compact_only
   overflow= 1000  可截= 9999  阈值=4000  → route=truncate_tool_results_only
   overflow= 1000  可截= 2000  阈值=4000  → route=compact_then_truncate


&emsp;&emsp;盯着输出读一遍这趟旅程：8 万字符进 L0 只剩 5.1 万——但**最后一行的报错完好无损**（保尾的价值就在这一刻兑现）；进 L1 被封顶到 1.6 万，末尾那截英文后缀原样追加；L2 走了两个平行世界——cache 还热时三条工具结果一个字节没动，cache 过期后两条旧的被软裁成"头 1500 + 尾 1500"、最老的整条变成占位符；L3 则用四组参数把判断树的四条路各走了一遍。下面的 Tier 1 验证把这几件事逐一钉死。

In [45]:
# Tier 1 验证：四道防线的关键行为逐条断言

# L0：必须保尾——报错信息不能被截掉，截掉的是头部
assert "Error: missing semicolon" in after_l0[-100:], "L0 必须保尾——报错信息不能被截掉"
assert after_l0[:30].startswith("[..."), "L0 截掉的是头部"

# L1：封顶生效 + 修复指引后缀必须在
assert len(after_l1) <= 16_000 + 80 and after_l1.endswith("rerun with narrower args if needed]"), \
    "L1 封顶后必须带修复指引后缀"

# L2：热缓存必须零动作（不是"少动"，是一个字节都不动）；冷缓存两段式都发生
assert hot == history, "cache 还热时 L2 必须零动作"
assert cold[0] == "[Old tool result content cleared]", "硬清占位符必须与源码 settings.ts:63 逐字一致"
assert cold[-1] == after_l1, "最近一条工具结果软裁硬清都不碰"

# L3：四路路由的判断顺序必须与源码一致
assert [l3_route(*p) for p in [(0,9,4),(5,0,4),(5,9,4),(5,2,4)]] == \
    ["fits", "compact_only", "truncate_tool_results_only", "compact_then_truncate"], "四路路由判断顺序错了"

print("Tier 1 全部通过：L0 保尾 / L1 封顶带后缀 / L2 热缓存零动作+冷缓存两段式 / L3 四路全对")

Tier 1 全部通过：L0 保尾 / L1 封顶带后缀 / L2 热缓存零动作+冷缓存两段式 / L3 四路全对


&emsp;&emsp;几组断言里最值得咀嚼的是中间两条：cache 还热时 `hot == history` 必须**严格**成立——零动作不是"少动"，是一个字节都不动；而硬清占位符必须与源码 `settings.ts:63` 逐字一致——课件里每个"实物文案"都应当能在源码里找到出处。说回 L1 输出末尾那截英文后缀，它正是这条防线里最容易错过的设计细节：**截断从来不是粗暴地砍一刀**。砍哪头是讲究过的——bash 保尾，因为报错信息通常堆在输出末尾；Read 保头，因为文件开头是定位内容的锚点；工具结果截断时如果检测到尾部有报错、JSON 收尾这类重要内容，还会自动改成"保头 + 保尾、省略中间"（`src/agents/embedded-agent-runner/tool-result-truncation.ts:120`，`hasImportantTail` 判断函数）。更妙的是截断后缀本身：`[... N more characters truncated; rerun with narrower args if needed]`（`src/agents/embedded-agent-runner/context-truncation-notice.ts:4`，后缀模板函数）——这句话**不是写给你看的，是写给模型看的**：它提示模型"刚才那个工具的输出被截了，需要的话换更窄的参数重跑一次"。截断不只是丢信息，还顺手把"怎么补救"教给了模型，把一次有损压缩变成了模型可以自愈的信号。

&emsp;&emsp;最后一条暗线，能把这些散落的设计串成同一个"为什么"。你可能已经注意到几个反复出现的怪癖：上一张表里的 successor transcript 要让新会话文件的消息前缀尽量与旧文件一致；L0 / L1 的截断都只在文本**末尾追加**说明、从不回头改写前面的字节；L2 的 context-pruning 默认工作在 `cache-ttl` 模式——prompt cache 还热着的时候**绝不动手**，等缓存过期（默认 TTL 5 分钟）才开始修剪（`src/agents/agent-hooks/context-pruning/settings.ts:48-50`，默认配置对象起始三行：`mode: "cache-ttl"` 与 5 分钟 TTL）。三个怪癖背后是同一个怕：**prompt cache**。主流模型 API 的缓存按"前缀完全一致"命中——历史消息的字节但凡被改动一处，从那里往后的整段缓存全部作废，这一轮就得全价重算。所以上下文工程有一条贯穿所有机制的隐形第一约束——<font color=red>能不动旧字节就不动旧字节；非动不可时，宁可在末尾追加、不要回头重写</font>。带着这条约束回看四道防线和四路路由，很多看似绕弯的设计就全通了。

&emsp;&emsp;把这条隐形约束画出来，就是下面这张对比图——同一段已缓存的 context，"末尾追加"和"中途重写"两种改法，缓存的存活范围天差地别。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605141020137.png" width=70%></div>

&emsp;&emsp;"全价重算"到底意味着丢多少？这件事不用相信形容词，几行代码就能量化——下面这个 demo 对同一段 context 用两种改法，算"下一轮还能复用多长的公共前缀"。

In [46]:
# 追加 vs 重写：量化 prompt cache 的前缀命中差异

def reusable_prefix(old: str, new: str) -> int:
    """主流模型 API 的 prompt cache 按"前缀完全一致"命中——能复用的就是这段公共前缀。"""
    n = 0
    # 逐字符比较，遇到第一个不一致就停——那里就是缓存命中的边界
    for a, b in zip(old, new):
        if a != b:
            break
        n += 1
    return n

old_ctx = "system 指令...\n" + "历史消息若干...\n" * 100        # 上一轮发给模型的 context
append  = old_ctx + "[... 500 more characters truncated ...]"   # 策略一：只在末尾追加
rewrite = old_ctx.replace("历史消息若干", "历史讯息若干", 1) + "[... 500 more ...]"  # 策略二：回头改了一处旧字节

print(f"context 总长 {len(old_ctx):,} 字符")
print(f"追加策略可复用前缀: {reusable_prefix(old_ctx, append):,} 字符（100% 缓存全保住）")
print(f"重写策略可复用前缀: {reusable_prefix(old_ctx, rewrite):,} 字符（改动点之后全部作废，这一轮全价重算）")
assert reusable_prefix(old_ctx, append) == len(old_ctx)
assert reusable_prefix(old_ctx, rewrite) < 30
print("验证通过：宁可追加、不要重写——不是风格洁癖，是真金白银")

context 总长 1,013 字符
追加策略可复用前缀: 1,013 字符（100% 缓存全保住）
重写策略可复用前缀: 15 字符（改动点之后全部作废，这一轮全价重算）
验证通过：宁可追加、不要重写——不是风格洁癖，是真金白银


&emsp;&emsp;结果一目了然：追加策略 100% 前缀保住，重写策略只剩十几个字符——改动点之后的缓存全部作废。考虑到缓存命中的那段输入通常按大幅折扣计价（不同厂商一折到五折不等），"宁可追加、不要重写"就不是代码洁癖，而是每一轮都在结算的真金白银。

&emsp;&emsp;好，进阶到此为止——把"四路路由"和"四道防线"两张图都装进口袋，不管你是细读还是跳过，我们都在这里会合，进 1.4 节做章末收口，并把通往第 2 章的那道门预埋好。

### 1.4 章末收口：context 全景 + 第 2 章预埋

&emsp;&emsp;我们停下来盘点一下这一章走过的路。我们从一个看不见的黑盒出发——你发一句话，模型这一轮到底收到了什么；用 `openclaw proxy blob` 把它砸开，看到了实际发给模型的四层 context（系统指令 / 记忆注入 / 历史 / 当前消息），还算清了一笔颠覆直觉的成分账单——体积大头是工具结果和工具 schema，对话文字不足 1%；然后钻进 `ContextEngine`，看清它用五个生命周期阶段（bootstrap → ingest → assemble → compact → prepareSubagentSpawn）把这四层组装出来，并搞清了其中 compact 和 prepareSubagentSpawn 是 fail-closed 的特例；进阶部分我们还看到，compact 只是 context 治理"四道防线"的最后一招，且整条防线都受 prompt cache"能不动旧字节就不动旧字节"这条隐形约束的支配；最后我们用 Python MVP 和真机 cell 把 assemble 四层组装、compact 压缩、成分账单核算、四道防线流水线这几件核心的事都在自己机器上跑通了。

&emsp;&emsp;本章覆盖了五阶段里的前四个。最后那个阶段 `prepareSubagentSpawn`——为子 agent 准备独立 context 快照——我们故意留着没展开，因为它正是通往第 2 章的门。下面这个 Tier 2 端到端验证，把本章的 assemble 和 compact 串成一条完整链路跑一遍，并在末尾摸到 prepareSubagentSpawn 这道门，作为本章到下章的衔接。

In [48]:
# Tier 2 端到端验证：把本章 assemble -> compact -> (摸到) prepareSubagentSpawn 串成一条链路
# 1 个 case 覆盖全章：模拟一轮完整的"发消息 -> 组装 -> 历史增长触发压缩 -> 准备派生子agent"
from typing import List  # prepare_subagent_spawn 返回类型注解需要

e2e = MiniContextEngine(
    system_prompt="你是个人助手",
    memory_injection="MEMORY.md: 用户偏好简洁回答",
)

events_trace = []  # 端到端记录关键节点，最后断言序列完整

# 阶段 1：ingest + assemble（发一轮消息并组装）
e2e.ingest(Message("user", "开始一个长任务"))
asm = e2e.assemble()
events_trace.append("assembled")
assert "current" in asm  # assemble 产出四层

# 阶段 2：灌入大量历史触发 compact
for i in range(8):
    e2e.ingest(Message("assistant" if i % 2 else "user", f"中间消息{i}"))
if len(e2e.history) > e2e.max_messages:
    events_trace.append("compaction_start")
    e2e.compact()
    events_trace.append("compaction_end")

# 阶段 3：摸到 prepareSubagentSpawn 这道门——为子 agent 准备 context 快照
#   这里只做最小重现：拷一份当前历史作为"快照"，真正的隔离语义留到第 2 章
def prepare_subagent_spawn(engine: MiniContextEngine) -> List[str]:
    """prepareSubagentSpawn 最小重现：导出一份 context 快照给子 agent。
    真实实现是 fail-closed 的（registry.ts:831），完整隔离语义见第 2 章。
    """
    return [m.content for m in engine.history]

snapshot = prepare_subagent_spawn(e2e)
events_trace.append("subagent_context_prepared")

print("端到端事件序列：", events_trace)
print("交给子 agent 的 context 快照：", snapshot)

# 断言完整链路：组装 -> 压缩 -> 准备子agent context，序列齐全
assert events_trace == [
    "assembled", "compaction_start", "compaction_end", "subagent_context_prepared"
], "端到端链路事件序列必须完整"
print("\nTier 2 通过：assemble -> compact -> prepareSubagentSpawn 完整链路跑通")

端到端事件序列： ['assembled', 'compaction_start', 'compaction_end', 'subagent_context_prepared']
交给子 agent 的 context 快照： ['[摘要] 此前 7 条对话已压缩为本条', '中间消息6', '中间消息7']

Tier 2 通过：assemble -> compact -> prepareSubagentSpawn 完整链路跑通


&emsp;&emsp;这条端到端链路把本章学的东西全串起来了：先 assemble 组装、再因历史增长触发一次 compact 压缩、最后摸到 `prepareSubagentSpawn` 这道门导出了一份 context 快照交给子 agent。你看那个 `prepare_subagent_spawn` 函数——它现在只是把历史原样拷了一份，但真实场景里，子 agent 拿到的 context 到底是"原样拷贝（fork）"还是"不带父对话历史、轻装上阵（isolated）"？子 agent 跟主 agent 又共享了什么、隔离了什么？这正是第 2 章要回答的问题。本章你已经能用 proxy blob 看见实际 context、能描述五阶段、能解释 compact 的触发和 fail-closed 含义——带着这些，我们进入多智能体的世界。

---

## <center>第 2 章：多智能体——spawn 子 agent 分治</center>

&emsp;&emsp;第 1 章我们把单个 agent 这一轮拿到的 context 彻底看透了。但现实里的任务，常常不是一根独木桥能扛下来的——它可能大到单轮 context 装不下，也可能需要"一边查资料、一边算数据"的并行。这时单个 agent 的那条 loop 就成了瓶颈。本章我们就来看 OpenClaw 怎么破这个局：它能 **spawn**（派生）出子 agent 来分摊任务。

&emsp;&emsp;本章我们关心的核心问题是：**子 agent 跟主 agent 到底隔离了什么、又共享了什么**？ 这个问题之所以关键，是因为它直接决定了多 agent 协作的安全边界和能力边界。第 2.1 节我们先做认知重塑，把"从单 agent 到 agent 团队"的心智模型建立起来，并认清 OpenClaw 里两个相关工具的区别。第 2.2 节是本章重头——剖析子 agent 的"两隔离 + 两共享"四个维度（session / context 隔离，lane / auth 共享），其中包含一条非常重要的诚实边界：**auth 是不隔离的**。第 2.3 节做真机演示对账，并在章末收口时用一张八行全景表把 OpenClaw 的多智能体模式谱系铺开——让 subagent 在完整版图里找到自己的位置。我们从认知重塑开始。

### 2.1 认知重塑：从单 agent 到 agent 团队

&emsp;&emsp;先纠正一个最容易跑偏的直觉。当你听到"子 agent"时，很可能脑子里浮现的是"开个轻量线程去跑个小任务"。这个类比是错的，而且错得会让你后面理解隔离时处处别扭。在 OpenClaw 里，一个子 agent **不是线程，而是一个完整独立的 agent session**——它有自己的 loop、自己的工具权限评估、自己的状态。换句话说，spawn 一个子 agent，相当于派出去一个跟主 agent 平级的"同事"，而不是开一个后台小工。我们用一张拓扑对比图，把"单 agent"和"agent 团队"这两种形态放在一起，帮你把这个心智模型彻底翻转过来。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124758854.png" width=70%></div>

&emsp;&emsp;主 agent 派生子 agent，靠的是两个相关但不同的工具，别把它们搞混了。一个是 `subagents`（工具 name 声明在 `src/agents/tools/subagents-tool.ts:25`），它的职责偏向**列出/查看**当前活跃和最近的子 agent。另一个是 `sessions_spawn`（工具 name 声明在 `src/agents/tools/sessions-spawn-tool.ts:269`），它的职责是真正去**派生一个新的子 agent session**。一个偏管理查看、一个偏创建派生，记住这个分工。这两个工具跟上节课讲的所有工具一样，调用时都要走那套 8 步策略管线——子 agent 不是绕过安全机制的后门。

&emsp;&emsp;顺带回答一个很自然的疑问：主 agent 怎么"知道"该派生子 agent？除了模型自己判断任务拆不拆得开，OpenClaw 还留了一个配置旋钮 `agents.defaults.subagents.delegationMode`——默认值 `suggest`（工具摆在那里，用不用模型自己定）；调成 `prefer` 后，system prompt 里会多注入一段引导，开头原文是 "Mode: prefer. You are the responsive coordinator for this conversation."（`src/agents/system-prompt.ts:89-94`——`:89` 是注入门槛判断行，minimal 模式 / 非 prefer / 没有 `sessions_spawn` 工具三条任一即不注入；`:94` 是这句措辞行），把"派活给子 agent"从可选项变成被鼓励的默认姿势。

&emsp;&emsp;分工记住了，紧接着是一个用 `sessions_spawn` 时必须知道的行为细节——**它是非阻塞的**：不会停在那里等子 agent 干完活，而是立即返回 `{ status: "accepted", runId, childSessionKey }`（官方文档原话 "always non-blocking"，`docs/tools/subagents.md:641`，行为说明列表项）。那结果怎么回来？靠一套叫 **`announce`（回报）** 的推送机制：子 agent 跑完后，OpenClaw 把它的结果包装成一个完成事件交回父 agent，父 agent 的体感就是"下一轮收到一条内部消息"（实现在 `src/agents/subagent-announce.ts`，它内部会按父 agent 是顶层还是嵌套子 agent 分两条投递路径，这个分叉细节不必记）。注意两点：这不是工具调用的返回值，而是平台层的主动推送；而且官方文档明说 `announce` 是 best-effort（尽力而为）的——gateway 重启时还没送达的 `announce` 会丢（`docs/tools/subagents.md:639`，best-effort 限制说明列表项）。

&emsp;&emsp;配合这套推送模型，派生之后正确的等待姿势是调 `sessions_yield` 工具**主动结束当前 turn**，让完成事件作为下一条消息到来，而**不是**写轮询循环反复查 `subagents`。这个工具的描述原文就把用法写明白了："End current turn. Use after spawning subagents; results arrive as next message."（工具 name 与这句描述分别在 `src/agents/tools/sessions-yield-tool.ts:15` 和 `:16`）。

&emsp;&emsp;最后两条属于"了解即可、踩坑再查"的约束细节，不用背。第一，子 agent 想"绕过 `announce` 直接横向私聊"是被硬性禁止的——`sessions_send` 就在子 agent 的永久禁用工具名单里（`src/agents/agent-tools.policy.ts:46-55`，`SUBAGENT_TOOL_DENY_ALWAYS` 常量定义，`:53` 的注释原文是 "subagents communicate through announce chain"），所以"子→父单向汇报"不是约定俗成，而是策略层强制。第二，子 agent session 结束后默认 60 分钟自动归档（`DEFAULT_SUBAGENT_ARCHIVE_AFTER_MINUTES=60`，`src/config/agent-limits.ts:6`，常量定义行）——归档是把会话转录文件改名保留（`*.deleted.<时间戳>`），不是物理删除；归档定时器同样是 best-effort，gateway 重启会丢。

> **【常见误区】**：把子 agent 当成"轻量线程"或"绕过权限的捷径"。后果是你会低估它的独立性，也会误以为它能逃过工具策略检查。正确理解：子 agent 是一个**独立的 agent session**，有完整的 loop 和自己的工具权限评估；它发起的每次工具调用，照样要穿过上节课讲的 8 步策略管线。排查方法：如果你发现自己在想"用子 agent 是不是能跳过某个限制"，立刻打住——子 agent 受的约束只会比主 agent **更严、不会更松**：除了同样要过策略管线，它还有一份永久禁用工具名单（`sessions_send` / `gateway` / `cron` 等，见上一段说的 `SUBAGENT_TOOL_DENY_ALWAYS`），默认也拿不到 message 工具和 `sessions_list` / `sessions_history` / `sessions_send` / `sessions_spawn` 这组 session 工具（`docs/tools/subagents.md:535-543`，默认禁发工具枚举说明段，列出的正是这五个；注意 `sessions_yield` 不在内——等结果还得靠它）。它解决的是"分治"问题，不是"提权"问题。

&emsp;&emsp;那么当主 agent 派生了一个子 agent 后，这两个 agent 之间到底是什么关系？它们各自跑各自的，但又不可能完全无关——总得有点东西是隔开的、有点东西是共享的。这就引出了本章最核心的内容：到底哪些隔离、哪些共享。我们进 2.2 节，把这张边界矩阵彻底铺开。

### 2.2 隔离与共享剖析：四个维度看清子 agent 边界

&emsp;&emsp;子 agent 跟主 agent 之间，有些东西隔离、有些东西共享，一共落在四个维度上：session（会话）、context（上下文）、lane（执行队列）、auth（认证）。先把结论摆出来——**两隔离 + 两共享**：session 和 context 是**隔离**的（子 agent 有自己的会话身份和独立的上下文快照），而 lane 和 auth 是**共享**的（所有子 agent 排在同一条全局并发队列上、并且合并继承了主 agent 的认证）。这个"两隔离两共享"是本节最该记牢的框架。下面我们一个一个拆，其中 lane 这一维特别容易被想当然地讲反，我们会专门纠正。先看两个隔离维度。

&emsp;&emsp;**隔离维度一，session 隔离**。 子 agent 拥有自己独立的 session key，不跟主 agent 共用一套会话状态。这个 session key 的格式官方文档里有明确写法——形如 `agent:<agentId>:subagent:<uuid>`（依据 `docs/tools/subagents.md:459` 的 session key 说明）。也就是说，子 agent 的会话身份是独立编址的，主 agent 的 session state 不会跟它混在一起。

&emsp;&emsp;**隔离维度二，context 隔离**。 这一维直接接上了第 1 章末尾那道门 `prepareSubagentSpawn`——它为子 agent 准备一份独立的 context 快照。这里有两种模式：**isolated**（隔离模式，子 agent **不继承父 agent 的对话历史**——但它仍有自己的 bootstrap 注入，注入白名单恰好只保留 AGENTS.md 和 TOOLS.md 两个文件（`src/agents/workspace.ts:700`，`SUBAGENT_BOOTSTRAP_ALLOWLIST` 常量定义行），不是完全空白的 context）和 **fork**（分叉模式，复制一份父 agent 的 context 快照给子 agent）。isolated 适合"我只要你干一件干净独立的活，别被我这边的对话干扰"，fork 适合"你需要知道我们之前聊了什么才好接着干"。

&emsp;&emsp;接下来是两个**共享**维度，第一个就是最容易讲反的 lane **。共享维度一，lane 共享（这里要特别纠正一个常见的误解）**。 凭直觉很多人会以为"每个子 agent 有自己独立的执行车道、互不干扰"——**这是错的，恰恰相反**。官方文档写得很清楚：所有子 agent 都排在**同一条**名为 `subagent` 的**全局并发队列**上（`docs/tools/subagents.md:128` 提到 "the global `subagent` lane"，`:596-599` 进一步说明 "Sub-agents use a dedicated in-process queue lane / Lane name: `subagent`"）。这条共享队列用一个叫 `maxConcurrent` 的参数（默认值 **8**，源码常量 `DEFAULT_SUBAGENT_MAX_CONCURRENT=8` 在 `src/config/agent-limits.ts:4`，常量定义行；配置项 `agents.defaults.subagents.maxConcurrent`）来控制"最多几个子 agent 能同时跑"——文档把它定性为一个"safety valve"（安全阀，`:640`）。所以 lane 不是"每个子 agent 一条独立车道"，而是"大家挤在同一条队列上、受一个并发上限约束"。它是**共享的并发容量控制**，不是隔离。顺带一个防混淆的对照：主 agent 自己跑在独立的 main lane 上，默认并发是 **4**（`DEFAULT_AGENT_MAX_CONCURRENT=4`，`src/config/agent-limits.ts:3`，常量定义行）——8 这个安全阀只管子 agent 队列，两条 lane 各管各的。

&emsp;&emsp;**共享维度二，auth 共享（合并继承）**。 子 agent 的认证（auth profiles）并不是独立的：主 agent 的 auth profiles 会作为**兜底（fallback）合并**进子 agent，子 agent 自己的 profile 在冲突时优先，但主 agent 的 profile 始终作为后备可用。官方文档把这点写得明明白白——`docs/tools/subagents.md:464` 的原话是 "Fully isolated auth per agent is not supported yet."（每个 agent 完全隔离的认证目前还不支持。）所以你**不能**把子 agent 理解成一个"完整沙箱"——它在 session 和 context 上隔离，但在 lane（共享并发队列）和 auth（合并继承）上是共享的。我们把这四个维度做成一张矩阵表，让边界一目了然。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124801809.png" width=70%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>subagent 四维边界矩阵：两隔离（session/context）+ 两共享（lane/auth）</font></p>
<div class="center">

| 维度 | 隔离/共享 | 具体行为 | 依据 |
|------|----------|----------|------|
| session | 隔离 | 子 agent 有独立 session key（`agent:<agentId>:subagent:<uuid>`）| docs/tools/subagents.md:459 |
| context | 隔离 | 独立 context 快照：isolated 不继承父对话历史（仍有自身 bootstrap 注入）/ fork 复制父快照 | prepareSubagentSpawn（context-engine/types.ts:363）|
| **lane** | **共享** | 所有子 agent 共享名为 `subagent` 的全局并发队列，`maxConcurrent` 默认 8 控并发（安全阀，非独立车道）| docs/tools/subagents.md:596-599、:128、:640 |
| **auth** | **共享（合并继承）** | 主 agent auth 作为兜底（fallback）合并进子 agent，冲突时子优先 | docs/tools/subagents.md:464，原文"not supported yet" |

</div>

&emsp;&emsp;这张表是本章你最该带走的东西——记住"两隔离 + 两共享"这个口诀：session 和 context 隔离，lane（全局并发队列）和 auth（合并继承）共享。下面我们用 Python MVP 把这套语义重现出来，让你亲手验证 isolated 和 fork 的 context 行为差异、session key 的唯一性、lane 的共享并发约束、以及 auth 的合并继承。再次强调：这是 Python 最小重现，不是真实源码——真实的实现远比这复杂，这个 MVP 只重现"哪些隔离、哪些共享"的核心语义。

In [49]:
# [注意] Python 最小重现，非 OpenClaw 真实源码（只重现两隔离两共享的核心语义）
import uuid
from dataclasses import dataclass, field
from typing import Dict, List, Optional

# lane 共享：所有子 agent 排在同一条名为 subagent 的全局并发队列上
#   对应 docs/tools/subagents.md:596-599 的 Lane name: "subagent"
SUBAGENT_LANE_NAME = "subagent"
# maxConcurrent 默认 8——这条共享队列的并发安全阀（docs:640 "safety valve"）
SUBAGENT_MAX_CONCURRENT = 8

@dataclass
class Agent:
    """一个 agent 实例。重现四个维度：session_key（隔离）/ context（隔离）/ lane_name（共享）/ auth（共享）。"""
    agent_id: str
    session_key: str                              # session 维度：独立会话身份（隔离）
    lane_name: str                                # lane 维度：所属并发队列名（子 agent 共享）
    context: List[str] = field(default_factory=list)   # context 维度：上下文快照（隔离）
    auth: Dict[str, str] = field(default_factory=dict) # auth 维度：认证 profiles（共享合并）

def spawn_subagent(parent: Agent, sub_agent_id: str, mode: str,
                   sub_auth: Optional[Dict[str, str]] = None) -> Agent:
    """从父 agent 派生子 agent，重现 OpenClaw 的两隔离（session/context）+ 两共享（lane/auth）。

    Args:
        parent: 主 agent 实例
        sub_agent_id: 子 agent 的 id
        mode: context 模式——"isolated"（从空白起）或 "fork"（复制父 context）
        sub_auth: 子 agent 自己的 auth profile（冲突时优先于父）

    Returns:
        一个新的 Agent：session/context 隔离，lane（共享并发队列）/auth（合并继承）共享
    """
    # session 隔离：生成独立 session key，格式仿 agent:<id>:subagent:<uuid>
    sub_session = f"agent:{sub_agent_id}:subagent:{uuid.uuid4().hex[:8]}"
    # lane 共享：不分配独立车道，所有子 agent 统一挂在同一条 subagent 队列上
    sub_lane = SUBAGENT_LANE_NAME
    # context 隔离：按模式决定快照内容
    if mode == "isolated":
        sub_ctx: List[str] = []                   # 不继承父对话历史（真实 isolated 子 agent 仍注入自身 bootstrap，白名单恰为 AGENTS.md/TOOLS.md 两个文件；此处 MVP 简化为空列表）
    elif mode == "fork":
        sub_ctx = list(parent.context)            # 复制一份父 context 快照
    else:
        raise ValueError("mode 只能是 isolated 或 fork")
    # auth 共享：主 agent auth 作为 fallback 合并，子 auth 冲突时优先（**sub 在后覆盖）
    merged_auth = {**parent.auth, **(sub_auth or {})}
    return Agent(sub_agent_id, sub_session, sub_lane, sub_ctx, merged_auth)

def admit_to_lane(active_count: int) -> bool:
    """重现并发安全阀：当前 subagent 队列已满 maxConcurrent 时，新子 agent 需排队等待（共享队列的并发 cap，不是工具层拒绝）。

    Args:
        active_count: 当前已在 subagent 队列上运行的子 agent 数

    Returns:
        True 表示还有并发槽可放行，False 表示已达 maxConcurrent 上限需排队
    """
    # 达到并发上限就不再放行——这就是 maxConcurrent 作为"安全阀"的作用
    return active_count < SUBAGENT_MAX_CONCURRENT

# 构造主 agent（主 agent 跑在自己的 main 通道，不在 subagent 共享队列上）
main_agent = Agent(
    agent_id="main",
    session_key="agent:main:root",
    lane_name="main",
    context=["父上下文：项目背景", "父上下文：用户偏好"],
    auth={"openai": "main-key"},
)

# 派生两个子 agent：一个 isolated，一个 fork（带自己的 auth）
worker_isolated = spawn_subagent(main_agent, "worker1", mode="isolated")
worker_fork = spawn_subagent(main_agent, "worker2", mode="fork",
                             sub_auth={"anthropic": "sub-key"})

print("=== 主 agent ===")
print(f"  session={main_agent.session_key}  lane={main_agent.lane_name}  ctx={main_agent.context}")
print("=== isolated 子 agent ===")
print(f"  session={worker_isolated.session_key}  lane={worker_isolated.lane_name}  ctx={worker_isolated.context}（不继承父对话历史）")
print("=== fork 子 agent ===")
print(f"  session={worker_fork.session_key}  lane={worker_fork.lane_name}  ctx={worker_fork.context}（复制父）")
print(f"  auth={worker_fork.auth}（合并了父的 openai + 自己的 anthropic）")
# 两个子 agent 的 lane 是同一条共享队列
print(f"\n两子 agent 共享同一条 lane？ {worker_isolated.lane_name == worker_fork.lane_name == SUBAGENT_LANE_NAME}")
print(f"并发安全阀演示：当前已跑 {SUBAGENT_MAX_CONCURRENT} 个时还能放行新子 agent？ {admit_to_lane(SUBAGENT_MAX_CONCURRENT)}")

=== 主 agent ===
  session=agent:main:root  lane=main  ctx=['父上下文：项目背景', '父上下文：用户偏好']
=== isolated 子 agent ===
  session=agent:worker1:subagent:962901ac  lane=subagent  ctx=[]（不继承父对话历史）
=== fork 子 agent ===
  session=agent:worker2:subagent:bfa8ed31  lane=subagent  ctx=['父上下文：项目背景', '父上下文：用户偏好']（复制父）
  auth={'openai': 'main-key', 'anthropic': 'sub-key'}（合并了父的 openai + 自己的 anthropic）

两子 agent 共享同一条 lane？ True
并发安全阀演示：当前已跑 8 个时还能放行新子 agent？ False


&emsp;&emsp;这段 MVP 的核心是 `spawn_subagent` 函数里那四处对应四个维度的处理：session 生成独立 id（隔离）、context 按模式决定是空白还是复制父快照（隔离）、lane 统一挂到固定的 `"subagent"` 共享队列名（共享，不再生成独立 id）、auth 用 `{**parent.auth, **sub_auth}` 做合并（共享继承，且子在后覆盖父表示冲突时子优先）。另外 `admit_to_lane` 函数重现了那条共享队列的并发安全阀——当队列上已经跑满 8 个（`maxConcurrent`）时，新子 agent 就放不进去、得排队。你跑完会看到 isolated 子 agent 的 context 是空列表（MVP 简化；真实 isolated 仍有自身 bootstrap 注入，只是不继承父对话历史）、fork 子 agent 的 context 复制了父的两条、两个子 agent 的 lane 都是同一条 `subagent` 队列、而 fork 子 agent 的 auth 同时有父的 openai-key 和自己的 anthropic-key——这就是"两隔离 + 两共享"的手感。下面 Tier 1 验证把这四个维度的行为逐一断言钉死。

In [50]:
# Tier 1 验证：四个维度的隔离/共享行为逐一断言（两隔离 + 两共享）

# 验证点 1（隔离）：context——isolated 从空白起，fork 复制父
assert worker_isolated.context == [], "isolated 模式 context 必须为空"
assert worker_fork.context == main_agent.context, "fork 模式 context 必须是父的副本"

# 验证点 2（隔离）：session——子 agent session_key 必须不同于主 agent
assert worker_isolated.session_key != main_agent.session_key
assert worker_fork.session_key != worker_isolated.session_key, "两个子 agent 之间也各自独立"

# 验证点 3（共享）：lane——所有子 agent 共享同一条名为 subagent 的全局并发队列
assert worker_isolated.lane_name == worker_fork.lane_name == SUBAGENT_LANE_NAME, \
    "所有子 agent 必须共享同一条 subagent 队列（不是独立车道）"
# 并发安全阀：未满 maxConcurrent 时放行，满了则不能立即放行、需排队
assert admit_to_lane(0) is True, "队列空时应放行新子 agent"
assert admit_to_lane(SUBAGENT_MAX_CONCURRENT) is False, "达到 maxConcurrent(8) 时不能立即放行、需排队"

# 验证点 4（共享）：auth——父的 auth 作为 fallback 合并进子 agent
assert worker_fork.auth["openai"] == "main-key", "父 auth 必须作为 fallback 出现在子 agent"
assert worker_fork.auth["anthropic"] == "sub-key", "子自己的 auth 也在，冲突时子优先"

print("Tier 1 通过：session/context 隔离 + lane 共享并发队列 + auth 合并继承，全部符合预期")

Tier 1 通过：session/context 隔离 + lane 共享并发队列 + auth 合并继承，全部符合预期


&emsp;&emsp;这个验证把"两隔离 + 两共享"四件事钉死了：isolated 的 context 为空、fork 复制父（隔离）、两个子 agent 的 session 各自独立（隔离）、所有子 agent 共享同一条 subagent 并发队列且受 maxConcurrent=8 约束（共享）、auth 里既有父的 key 也有子的 key（共享合并继承）。其中验证点 3 和 4 是最该记牢的——它们用代码证明了 lane 是共享并发队列、auth 是合并继承，正好印证了官方文档那句 "not supported yet" 以及那条 "global subagent lane"。讲清了语义，我们到真机上去看一次真实的子 agent 派生。下面用一句精心设计的提示词，**无头**逼它派生子 agent，并在 debug 日志里抓现行。

In [51]:
# 真机演示：用一句提示词逼出 sessions_spawn，在 debug 日志里观察子 agent 派生
#   提示词设计要点：任务自然拆成"子任务 + 主任务并行"，并显式点名 sessions_spawn——
#   不点名的话模型可能抄近道自己用 exec 数文件，演示就落空了（实测踩过）
#   想要全屏交互观演，可在系统终端跑 openclaw tui --local 输入同一句话（TUI 需要真终端）
!openclaw --log-level debug agent --local --agent main \
  --message "调用 sessions_spawn 工具派生一个子 agent 去统计当前目录文件数（这一步必须用子 agent 完成，不要自己用 exec 数），你自己同时告诉我今天日期" 2>&1 \
  | python3 trace_pretty.py

# 派生的落盘证据：子会话以 agent:<id>:subagent:<uuid> 格式独立注册（2.1 节讲过的 session key）
!grep -o "agent:main:subagent:[a-f0-9-]*" ~/.openclaw/agents/main/sessions/sessions.json | tail -3

[01] 🧩 上下文体检  [agent/embedded] [context-diag] pre-prompt: sessionKey=agent:main:main messages=17 roleCounts=assistant:9,toolResult:5,user:3 historyTextChars=3177 maxMessageTextChars=856 historyImageBlocks=0 systemPromptChars=38568 promptChars=86 promptImages=0 provider=minmax/MiniMax-M3 sessionFile=/Users/mac/.openclaw/agents/main/sessions/d72663a8-8d3a-463b-8fbd-8b1a03229dbd.jsonl
[02] 🗜 压缩路由判定  [agent/embedded] [context-overflow-precheck] pre-prompt check sessionKey=agent:main:main provider=minmax/MiniMax-M3 route=fits estimatedPromptTokens=18418 pressureSource=transcript_estimate promptBudgetBeforeReserve=983616 overflowTokens=0 toolResultReducibleChars=0 reserveTokens=16384 effectiveReserveTokens=16384 contextTokenBudget=1000000 messages=17 unwindowedMessages=17 sessionFile=/Users/mac/.openclaw/agents/main/sessions/d72663a8-8d3a-463b-8fbd-8b1a03229dbd.jsonl
[03] 🔧 工具调用  [agent/embedded] embedded run tool start: runId=8569e989-88d0-4c2d-ab70-f730a0c078bd tool=session_status toolCall

&emsp;&emsp;这句提示词的设计很讲究——它把任务自然拆成了一个适合丢给子 agent 的子任务（统计文件数）和一个主 agent 自己做的任务（报日期），并且**显式点名 `sessions_spawn`**：实测发现只说"用一个子 agent"时，模型有时会抄近道直接用 `exec` 自己数文件、压根不派生——点名工具加一句"不要自己用 exec 数"，派生才稳定可复现。跑通后，`trace_pretty.py` 会高亮出 `tool=sessions_spawn` 的 start/end 两行——这就是主 agent 决定派生的那一刻；第二条 `grep` 则在会话注册表里捞出子 agent 的 session key，亲眼确认它就是 2.1 节讲的 `agent:<id>:subagent:<uuid>` 格式、跟主会话是两个独立条目——**session 隔离的落盘证据**。这里要交代本节的一条诚实边界。

> **【踩坑预警】**：子 agent 的隔离与共享边界，证据成色要分开交代。四个维度里只有 **session 隔离已有运行时实证**——上面这组命令是真机跑通过的：`sessions_spawn` 被真实调用、子 agent 的 session key 以 `agent:main:subagent:<uuid>` 格式独立落盘，眼见为实**。其余三个维度（context 隔离、lane 共享并发队列、auth 合并继承）仍是"源码 + 官方文档"层面的证据**——context 隔离要实证得 dump 出子 agent 实际收到的 context 对照，共享侧要实证得构造并发压测和多 auth profile 对照实验，本课都没有做。所以如果有人问你"两隔离两共享都亲眼跑通验证过吗"，诚实的回答是：session 隔离跑通了，另外三个维度有源码和文档的扎实证据、运行时实证没做。这本身也是一种素养——**区分"我有证据证明"和"我亲眼跑通证明"**。

&emsp;&emsp;讲清了两隔离两共享、以及运行时这条诚实边界，本章的机制核心就完整了。我们进 2.3 节做对账和收口。

### 2.3 Demo 对账

&emsp;&emsp;真机上跑子 agent 时有三条观测路径：本课主用的无头路径（debug 日志里的 `tool=sessions_spawn` 起止行 + `sessions.json` 里的子会话条目），以及终端里的 TUI 交互路径——TUI 会把子 agent 的状态流转（running 运行中、completed 正常完成、failed 异常失败、blocked 阻塞等待）和它的工具调用一个个亮出来，就像上节课你看主 agent 工具调用那样。第三条是浏览器里的 dashboard（OpenClaw 自带的控制台 Web UI）：它的 Sessions 页每行会话**可见渲染**了 `agentRuntime` 字段（跑在内嵌引擎还是外部 harness，渲染行在 `ui/src/ui/views/sessions.ts:918`）；行数据里还携带 `spawnedBy`（这个会话是谁派生的，会话行字段清单 `ui/src/ui/controllers/sessions.ts:171`，字段名行）——注意它当前没被渲染成可见列，是 UI 用来过滤和识别派生会话的内部字段；另外命令行里还有一本后台任务总账 `openclaw tasks list`，subagent / ACP / cron 的每次后台运行都会在里面记一条（`docs/automation/tasks.md`）。我们把"MVP 断言""真机观测""官方文档"这三套证据来源做一次对账，看它们是否互相印证。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>subagent 边界：三套证据来源对账（两隔离 + 两共享）</font></p>
<div class="center">

| 边界结论 | MVP 断言（可跑）| 真机观测（debug 日志 / 落盘）| 官方文档/源码 |
|----------|----------------|--------------|---------------|
| session 隔离 | session_key 唯一性断言通过 | 已实证：`sessions_spawn` 调用起止行 + `sessions.json` 里独立的 `agent:main:subagent:<uuid>` 条目 | subagents.md:459 |
| context 隔离 | isolated 空 / fork 复制断言通过 | （运行时待补 PARTIAL）| context-engine/types.ts:363 prepareSubagentSpawn |
| lane 共享 | 共享 lane name + maxConcurrent 断言通过 | （运行时待补 PARTIAL）| subagents.md:596-599、:128、:640 |
| auth 共享合并 | auth 合并继承断言通过 | （运行时待补 PARTIAL）| subagents.md:464 "not supported yet" |

</div>

&emsp;&emsp;这张对账表诚实地标出了哪些有 MVP 可跑证据、哪些运行时实证还是 PARTIAL 待补——这正是上面那条诚实边界的具象化。下面这个 Tier 2 端到端验证，把主 agent 并行派生子 agent 的场景跑一遍，断言 session/context 隔离、lane/auth 共享这几件最关键的事。

In [52]:
# Tier 2 端到端验证：主 agent 并行派生子 agent，断言 session 隔离 + auth 合并
# 1 个 case：主 agent 同时派一个 isolated 子 agent 干子任务，自己继续主任务

orchestrator = Agent(
    agent_id="main",
    session_key="agent:main:root",
    lane_name="main",
    context=["主任务：写一篇报告"],
    auth={"openai": "main-key"},
)

# 派生子 agent 去干"统计文件数"这个独立子任务（isolated：不需要主任务上下文）
file_counter = spawn_subagent(orchestrator, "filecounter", mode="isolated")

# 端到端断言：两个 agent 真正分治
# 1）session 隔离——身份不同，互不干扰
assert file_counter.session_key != orchestrator.session_key
# 2）context 隔离——子 agent 不背主任务的包袱（isolated）
assert file_counter.context == [] and orchestrator.context == ["主任务：写一篇报告"]
# 3）lane 共享——子 agent 挂在 subagent 全局并发队列上（不是独立车道）
assert file_counter.lane_name == SUBAGENT_LANE_NAME
# 4）auth 共享合并——子 agent 能用主 agent 的 key 干活（这是它能调工具的前提）
assert file_counter.auth == {"openai": "main-key"}

print(f"主 agent: {orchestrator.session_key}（lane={orchestrator.lane_name}）")
print(f"子 agent: {file_counter.session_key}（lane={file_counter.lane_name}，共享 subagent 队列）")
print(f"主 ctx: {orchestrator.context} | 子 ctx: {file_counter.context}（隔离）")
print(f"子 auth: {file_counter.auth}（继承主 agent，所以能调工具）")
print("\nTier 2 通过：session+context 隔离、lane+auth 共享——分治成立")

主 agent: agent:main:root（lane=main）
子 agent: agent:filecounter:subagent:83490a16（lane=subagent，共享 subagent 队列）
主 ctx: ['主任务：写一篇报告'] | 子 ctx: []（隔离）
子 auth: {'openai': 'main-key'}（继承主 agent，所以能调工具）

Tier 2 通过：session+context 隔离、lane+auth 共享——分治成立


&emsp;&emsp;这个端到端验证把分治的本质讲透了：两个 agent 在 session 和 context 上彻底分开（互不干扰、子不背主的包袱），但 lane（共享并发队列）和 auth（合并继承）是共享的——尤其 auth 这一共享很关键：子 agent 正是通过继承主 agent 的 auth 才有资格去调工具干活，**如果 auth 也完全隔离了，子 agent 就成了没钥匙的空壳**。本章你已经能说清子 agent 的"两隔离（session/context）+ 两共享（lane 全局并发队列 / auth 合并继承）"，也知道运行时这条 PARTIAL 边界。

&emsp;&emsp;本章把 subagent（子 agent）的隔离和共享机制看透了。但在收口之前，有必要诚实地说一句——subagent spawn 是 OpenClaw 当前最主力的多智能体模式，却不是唯一。把它放进 OpenClaw 多智能体的完整版图里看一眼，你的认知才不会有盲区。先用一张全景图建立空间感——底座是多人格地基，中间是四种互动形态的方向语义（父子单向、1:1 有界、fan-out 扇形、ACP 外接），边上是三个辅助角色——然后再用八行表格逐一对账明细。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124757984.png" width=70%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>OpenClaw 多智能体模式全景</font></p>
<div class="center">

| 模式 | 一句话定位 | 隔离 / 通信特征 | 和本章 subagent 的关系 |
|------|-----------|----------------|----------------------|
| 多 agent 配置路由（`agents.list` + bindings）| 一个 Gateway 静态配置多个全隔离的 agent"人格"，各有独立 workspace / auth / session 存储，入站消息按 bindings 规则路由给其中一个 | 彼此完全隔离、默认互不通信（官方文档 `docs/concepts/multi-agent.md` 整页讲它）| 整张全景的地基——Broadcast 分发的目标、`sessions_send` 跨的对象都必须是这里配置过的 agent； |
| subagent spawn | 派生 OpenClaw 内部的子 agent 分治任务 | context 隔离、子→父单向汇报 | 本章主角 |
| ACP 外部 harness | 把外部独立进程里的 AI 编码代理接进来当 agent 使（Claude Code / Codex / Gemini CLI / Cursor 等官方支持 13 种 harness，清单见 `docs/tools/acp-agents.md` 的 harness 表）| 独立进程、走 ACP（Agent Client Protocol，智能体客户端协议）；三种用法——①临时派生 ②频道会话持久绑定 ③整个 agent 固定走外部 harness（明细与源码锚点见表后正文）| 把"别的 AI 命令行工具"当成子 agent 来使；需安装 acpx 插件，沙箱 session 内禁用 |
| cron 定时 agent | 按定时计划自动跑一轮 agent，这一轮里还能再 spawn subagent | 隔离会话 | subagent 的"定时无人值守"触发场景 |
| `sessions_send` | 向另一个 agent 的会话发消息并等待回复，可有界往返多轮（ping-pong）| **跨 agent 默认关死**：需 `tools.agentToAgent.enabled=true` + allow 白名单（门控报错分支 `src/agents/tools/sessions-send-tool.ts:343-350`）；往返默认 5 轮、硬上限 20（常量定义行 `src/agents/tools/sessions-send-helpers.ts:15-16`），轮数一到强制走 announce 收尾 | 是 agent 之间"打电话"（1:1 有界对话），不是创建新 agent |
| Broadcast Groups（广播组）| 同一条消息分发给多个 agent，默认并行（也可顺序）、各自独立回应（fan-out 广播）| channel 触发；各 agent 的 session/记忆等隔离、但共享同一群的上下文；**互相看不到对方回应**；experimental、文档口径仅 WhatsApp（源码已超前，见下文） | 官方称 "specialized agent teams"；是 fan-out 不是 agent 间协商 |
| Task Flow（配合 Lobster）| 声明式多步骤编排、支持跨进程重启续传 | Task Flow 负责跨任务流程追踪（`src/tasks/task-flow-*.ts`、`docs/automation/taskflow.md`），Lobster 负责确定性步骤执行（`extensions/lobster/`）；编排的是 task 步骤（managed 模式下每一步创建一个后台 task 并驱动到完成）| OpenClaw 里最接近 "workflow（工作流）" 的东西 |
| Codex Supervisor（编队监督插件）| 一个常驻 agent 监控并驾驶一组 Codex 会话：端点探活、列会话、读 transcript、发指令、中断 | experimental 插件，随 OpenClaw 分发但**默认不启用**（manifest 声明 `onStartup:false`，需显式启用并配置 Codex app-server 端点）；五个工具的注册区在 `extensions/codex-supervisor/src/plugin-tools.ts:99-190`；配套 spec 蓝图 `docs/specs/claw-supervisor.md` 只落地了一部分 | 监督对象是外部 Codex 会话，不是本章的 subagent |

</div>

&emsp;&emsp;先说表里第一行那个最容易被忽略的地基——**多 agent 配置路由**。OpenClaw 允许你在同一个 Gateway 进程里静态配置多个 agent"人格"（`agents.list`），每个都有独立的 workspace、独立的 auth、独立的 session 存储，入站消息按 `bindings` 规则决定交给谁（官方文档 `docs/concepts/multi-agent.md` 整页讲的就是它，CLI 一句 `openclaw agents add work` 就能加一个）。它本身不涉及 agent 之间的任何互动，却是这张表里后面几行的**前提**：Broadcast 广播的分发目标、`sessions_send` 跨过去的对象，都必须是这里配置过的 agent——配置校验甚至会在加载时就把"广播列表里出现未配置的 agent id"拦下来（`src/config/zod-schema.ts:1257-1280`，superRefine 校验代码块）。bindings 怎么按 channel / 账号 / 群路由，不属于本节主线，这里不展开。

&emsp;&emsp;这个地基不用翻源码就能亲眼看到。OpenClaw 给了一条**只读**命令 `openclaw agents list --bindings`，把当前机器上配置的所有 agent"人格"连同各自的家底列出来——有几个 brain、每个的 workspace 和 agentDir 在哪、用什么模型、有几条路由规则，一眼扫完；它不创建也不修改任何东西，放心跑。第二条命令从配置文件侧对账：地基在 `~/.openclaw/openclaw.json` 里就是 `agents.list` 和 `bindings` 两个字段，我们把它们单独抽出来看。

In [53]:
# 真机观察：当前机器配置了哪些 agent"人格"（只读命令，不创建不修改）
#   agents 子命令组注册在 src/cli/program/register.agent.ts:151（list 子命令定义行）
!openclaw agents list --bindings

# 配置文件侧对账：上面 CLI 视图背后，就是 openclaw.json 里 agents.list + bindings 两个字段
!python3 -c "import json,os; cfg=json.load(open(os.path.expanduser('~/.openclaw/openclaw.json'))); print(json.dumps({'agents.list': [a.get('id') for a in (cfg.get('agents',{}).get('list') or [])], 'bindings': cfg.get('bindings', [])}, ensure_ascii=False, indent=2))"


🦞 OpenClaw 2026.5.28 (e932160)
   The only open-source project where the mascot could eat the competition.

Agents:
- main (default)
  Identity: ⚡ Nova (IDENTITY.md)
  Workspace: ~/.openclaw/workspace
  Agent dir: ~/.openclaw/agents/main/agent
  Model: minmax/MiniMax-M3
  Routing rules: 0
  Routing: default (no explicit rules)
  Providers:
    - openclaw-weixin default: configured
- researcher
  Identity: :mag: 深度调研机器人 (IDENTITY.md)
  Workspace: ~/.openclaw/agents/researcher/workspace
  Agent dir: ~/.openclaw/agents/researcher/agent
  Model: yunyi/claude-opus-4-6
  Routing rules: 0
Routing rules map channel/account/peer to an agent. Use --bindings for full rules.
Channel status reflects local config/creds. For live health: openclaw channels status --probe.
{
  "agents.list": [
    "main",
    "researcher"
  ],
  "bindings": []
}


&emsp;&emsp;两条命令看的是同一个地基的两面。CLI 视图是渲染过的人类可读版：默认装机你会看到只有一个 `main (default)`——这正是第 0 章说的"单 agent 默认形态"，地基存在、但默认只有一块砖；如果列出了多个 agent，每个都有自己独立的 Workspace、Agent dir 和 Model 三行，那就是"多人格"的真身，三个"独立"逐行可见。配置文件侧则是事实来源：`agents.list` 为空就回落到默认的 main，`bindings` 为空数组就走"默认 agent 兜底"路由（CLI 输出里 Routing 一行写的 default 指的就是它）。想亲手加一个人格，`openclaw agents add work` 一句话的向导就能建出独立 workspace——但建完之后消息怎么路由给它，不属于本节主线，这里不展开。

&emsp;&emsp;再把 ACP 行收进表里的"三种用法"摊开（表格里只放了名目，明细在这里）。**①临时派生**：工具 `sessions_spawn({runtime:"acp"})` 或 chat 命令 `/acp spawn`（runtime 参数枚举行 `src/agents/tools/sessions-spawn-tool.ts:166`）——用完即走 **；②频道会话持久绑定**：`bindings[].type:"acp"`，把某个具体频道会话钉死在一个常驻的外部 harness 会话上（schema 的 type 字面量声明在 `src/config/zod-schema.agents.ts:61-63`）**；③整个 agent 固定走外部 harness**：`agents.list[].runtime.type:"acp"`，让这个 agent 的所有会话都跑在外部代理上（runtime union schema 在 `src/config/zod-schema.agent-runtime.ts:974-988`）。三种形态共用同一个 acpx 插件后端，差别只在"会话的生命周期归谁管"——临时的归这一次 spawn，绑定的归那个频道会话，固定的归整个 agent。

&emsp;&emsp;ACP 在你机器上现在能不能用？不装任何东西就能自检。下面两条**只读**命令分别看两件事：第一条看入口——`openclaw acp` 是核心自带的命令，它做的是反方向的事（把 OpenClaw 反向暴露成一个标准 ACP agent 供外部 IDE 调用），帮助文案那句 "Run an ACP bridge backed by the Gateway" 就是证据；第二条从配置文件侧自检"借外部脑子"这条路的就绪状态——`acp` 配置字段和 `plugins.entries.acpx` 插件登记。默认装机这两处都是未配置，**这是预期结果不是故障**。

In [54]:
# ACP 就绪自检（两条都只读，不装插件、不改配置）
# 第一条：openclaw acp 是核心自带命令——把 OpenClaw 反向暴露成标准 ACP agent（本节只看入口，不展开方向细节）
!openclaw acp --help 2>&1 | head -6

# 第二条：配置侧自检——"让外部 agent 替 OpenClaw 干活"需要 acp 配置 + acpx 插件，默认都未配置
!python3 -c "import json,os; cfg=json.load(open(os.path.expanduser('~/.openclaw/openclaw.json'))); print('acp 配置:', json.dumps(cfg.get('acp','未配置'), ensure_ascii=False)); print('acpx 插件登记:', json.dumps((cfg.get('plugins',{}).get('entries',{}) or {}).get('acpx','未配置'), ensure_ascii=False))"


OpenClaw 2026.5.28 (e932160) — All your chats, one OpenClaw.

Usage: openclaw acp [options] [command]

Run an ACP bridge backed by the Gateway
acp 配置: "未配置"
acpx 插件登记: "未配置"


&emsp;&emsp;两条输出合起来，就是 ACP 在本章语境下最该带走的认知：**入口自带、外包默认关**。第一条证明"被外部当 agent 使"的桥是核心功能；第二条的两个"未配置"说明"借外部脑子"这条路默认没开——它要同时满足三个条件才走得通：ACP 启用、当前 session 非沙箱、acpx backend 插件已加载。最妙的是 OpenClaw 处理"没就绪"的方式：`sessions_spawn` 工具的 **runtime 参数枚举是动态收缩的**——ACP 不可用时，这个参数的可选值直接退化成只剩 `["subagent"]`（`src/agents/tools/sessions-spawn-tool.ts:166-168`，按 `acpAvailable` 三元切换的参数枚举行），并按三种不可用原因分别给出"改用 `runtime:"subagent"`"的报错指引（`:238-243`，三条不可用文案的判定函数体）。换句话说，**ACP 没就绪时，agent 压根"看不见"这个选项**——本章已经第三次遇到"约束默认从严"了。至于 ACP 怎么真把活交给外部 agent，超出本节范围，不展开；本章你只需要钉住一句：**它是 subagent 之外的第二种"借脑"方式，且默认关着**。


&emsp;&emsp;那真想开启的话，要在哪个配置文件里改什么？动作比想象的小——两条命令：`openclaw plugins install @openclaw/acpx` 装上官方 runtime backend 插件，再 `openclaw config set plugins.entries.acpx.enabled true` 把它启用（插件登记和启用是两步，装了不等于开了），两条落盘的都是同一个 `~/.openclaw/openclaw.json`——也就是上面自检 cell 读的那个文件（官方文档 `docs/tools/acp-agents.md` "Does this work out of the box?" 一节给的就是这两条）。这里有个容易误读的细节要校准：自检输出的两个"未配置"里，`acp` 配置字段空着**不等于总开关是关的**——`acp.enabled` 的判定写的是 `!== false`（`src/acp/policy.ts:11-13`，`isAcpEnabledByPolicy` 函数体，显式写 `false` 才算关），所以准确说法是**开关默认开、引擎默认没装**：默认装机卡住这条路的只有 backend 插件这一环，前面"三个条件"里真正常见的缺口也是它。装完跑 `/acp doctor` 做就绪体检；若你配置过 `plugins.allow` 白名单，记得把 `acpx` 加进白名单，否则装了也会被刻意拦下（同页 First-run gotchas 一节）。

#### OpenClaw 到底有没有 agent teams？

&emsp;&emsp;这里要把"OpenClaw 到底有没有 agent teams"分两层说清楚。先给结论：**有，但得分清两种不同形态的"多 agent"**。第一种是本章主角 **subagent**——父子分治、子→父单向汇报。第二种是 **Broadcast Groups（广播组）**：把同一条消息分发给多个 agent（默认并行，也可配置顺序执行），每个 agent 用自己的人设、工具、模型独立处理、各自回应（这些 agent 的 session、记忆彼此隔离，但共享同一个群的上下文）——OpenClaw 官方文档就把这种叫 "specialized agent teams"，目前还是 experimental（实验阶段）；范围上官方文档写的是 "Current scope: **WhatsApp only**"（`docs/channels/broadcast-groups.md:19`，scope 声明行），不过源码已经超前于文档——Feishu（飞书）扩展里已自带一套 broadcast 分发实现（`extensions/feishu/src/bot.ts:219`，`resolveBroadcastAgents` 函数定义行），这是"文档口径滞后于源码"的一个活例子，引用时以你手里的版本实测为准。

&emsp;&emsp;但关键的诚实边界在这里：**这两种都不是 Claude Code 那种"多个对等 agent 互相发消息、看得到彼此输出、协商着把一个任务推进下去"的 teams**。Broadcast Groups 是 **fan-out（广播分发）**——多个 agent 各跑各的、**互相看不到对方的回应**（官方文档 Limitations 明说 agents don't see each other's responses），不是协商；subagent 是父子单向汇报；`sessions_send` 则是 1:1 的**有界对话**——两个 agent 真能往返聊上几轮（ping-pong 机制，默认 5 轮、硬上限 20，配置项 `session.agentToAgent.maxPingPongTurns`——注意它挂在 `session.*` 命名空间下、管会话行为，区别于 `tools.agentToAgent.*` 那个工具开关命名空间；常量定义行在 `src/agents/tools/sessions-send-helpers.ts:15-16`），但轮数一到就强制走 announce 收尾（收尾提示语原文 "After this reply, the agent-to-agent conversation is over." 在 `:117`），而且**整条跨 agent 通道默认是关死的**——要显式配置 `tools.agentToAgent.enabled=true` 加 allow 白名单才打得开（门控报错分支在 `src/agents/tools/sessions-send-tool.ts:343-350`），又一处"约束默认从严"。打开之后这条通道对名单内的 agent 是**对等的**：判定逻辑只看发起方和目标方**各自**是否命中 allow 名单（allow 留空即全员放行，`isAllowed` 函数在 `src/plugin-sdk/session-visibility.ts:186-193`），main 没有任何特权——`agents.list` 里任意两个 agent 都能两两对话，work↔work 同样行，不是只有 main↔work 一种拓扑。

&emsp;&emsp;也就是说，截至本节基准的主分支（2026-05-29），OpenClaw 的 **core / Gateway 层没有"对等 mesh（多 agent 平等组网）/ 共享黑板 / 两方以上协商"这种通用运行时**，但**确实有 fan-out 式的多 agent 和 1:1 的有界对话**。注意这个否定要限定在"运行时原语"层面——插件生态里的 OpenProse（`.prose` 工作流 DSL，`extensions/open-prose`）看着像新模式，拆开看底层映射的还是 `sessions_spawn`，是 subagent 之上的语法糖；它确实能在 skill / DSL 层用一个共享 SQLite 文件当"协调白板"模拟多 agent 协作（skill 文档原话 "The VM and subagents share it as a coordination mechanism"，`extensions/open-prose/skills/prose/state/sqlite.md:44`，原则说明行），但那是用法层的拼装，不是核心运行时给的新原语。

&emsp;&emsp;一句话收口：OpenClaw 的多智能体谱系 = `agents.list` 多人格地基 + subagent 父子分治（并行/嵌套默认值 `maxChildrenPerAgent=5`、`maxSpawnDepth=1`，默认值见 `src/config/agent-limits.ts:5,8`；可配上限 20/5 由 schema 约束，见 `src/config/zod-schema.agent-defaults.ts:243-257`；开嵌套后角色按深度自动判定——depth 0 是 main、中间还能往下派的是 orchestrator、到底的是 leaf（不可再派生，`sessions_spawn` / `sessions_list` / `sessions_history` / `subagents` 这几个管理工具也不会发给它），判定逻辑在 `src/agents/subagent-capabilities.ts:158-161`）+ `sessions_send` 1:1 有界对话 + Broadcast Groups 广播 fan-out（依托 channel 触发）+ ACP 接外部 agent + cron 定时无人值守 + Task Flow 做步骤编排，外加一个 experimental 的 Codex Supervisor 编队监督插件——正好对上面全景表的八行。

> 📌 **【第 2 章你已经掌握】**：两隔离（session / context）+ 两共享（lane 全局并发队列 / auth 合并继承）的边界；auth 不隔离是官方文档明说的当前现状；工具约束只严不松（`sessions_send` 等永久禁用是策略层强制）；spawn 非阻塞、结果靠 `announce` best-effort 回报；以及八行全景表里 subagent 在 OpenClaw 多智能体谱系中的位置。

&emsp;&emsp;走到这里，我们解决了"复杂任务怎么分治"这第二堵墙。但还剩最后一个问题——分治也好、当下组装也好，都是**这一次会话之内**的事。你关掉会话再开一个新的，那些本该被记住的东西去哪了？子 agent 干完活回报给主 agent 的精华，怎么跨会话留存下来？这就把我们带到了第 3 章：记忆系统。

---

## <center>第 3 章：记忆系统——跨会话的长期脑</center>

&emsp;&emsp;前两章我们处理的都是"当下这一次会话"内部的事——第 1 章组装当下的 context，第 2 章在当下分治派生子 agent。但 agent 真正像个"助手"的地方，在于它能跨会话记住你。你上周告诉它"我对花生过敏"，这周开个全新会话问它"推荐个零食"，它该记得这茬。这种跨会话的持久记忆从哪来、怎么存、又怎么在新会话里被检索回当下的 context——就是本章要拆的第三堵墙。

&emsp;&emsp;本章分四步。第 3.1 节先立一个贯穿全章的分界框架——**短期记忆 vs 长期记忆**，然后翻开真实的记忆文件，看清 OpenClaw 官方的记忆是"3 个文件 + 可选机制"的结构，并特别纠正一个高频误解——dreaming（做梦机制）相关的三处（两目录 + 一文件）各管什么，千万别混。第 3.2 节回答"记忆从哪来"——短期对话变成长期记忆要走的三条自动来路，其中一条解释了你 `memory/` 目录里那些带时分后缀的文件是哪来的。第 3.3 节钻进 `memory_search` 工具，看它怎么用 hybrid RAG（混合检索）把记忆找回来，重点讲清那个向量 0.7 + 全文 0.3 的加权机制，并用 MVP 重现。第 3.4 节看 dreaming 机制并做章末收口。我们从分界框架开始——先有地图，再翻文件眼见为实。

### 3.1 记忆全景：短期 vs 长期的分界 + 文件清单

&emsp;&emsp;动手翻文件之前，先把"记忆"这个词拆干净——因为它在 agent 语境里其实是两层东西：**短期记忆**和**长期记忆**，而这两样你其实都已经见过一半了。短期记忆就是第 1 章拆过的那一整套：当前会话发给模型的 context、落在 `sessions/` 目录下的会话转录文件（第 2 章看 session 隔离落盘时你已经见过它的样子）、还有挤爆之前的 compact 压缩——它们的生命周期都绑定在"这一次会话"上，会话结束或被压缩后，细节就丢了。长期记忆才是本章的主角：workspace 下的一组 Markdown 文件加上一套检索索引，跨会话存活，新会话开场时再被注入或检索回当下的 context。先用一张表把分界立清楚：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>短期记忆 vs 长期记忆：分界对照</font></p>
<div class="center">

| 维度 | 短期记忆（第 1 章已拆） | 长期记忆（本章主角） |
|------|------------------------|---------------------|
| 载体 | 当前 context + `sessions/` 下的会话转录 | workspace 的 `MEMORY.md` / `memory/*.md` + SQLite 检索索引 |
| 生命周期 | 绑定单次会话，compact 后细节丢失 | 跨会话持久存活 |
| 进 context 的方式 | 作为历史层直接在场 | 注入层（`MEMORY.md`）+ `memory_search` 按需检索 |
| 本课讲解位置 | 第 1 章（组装与压缩） | 本章（存储、来路、检索、沉淀） |

</div>

&emsp;&emsp;表里短期那格写着"`sessions/` 下的会话转录"——既然长期记忆马上要眼见为实，短期这边也别只停在一句话上。第 2 章你在 `sessions.json` 里看过子会话的条目，现在把整个目录翻开看全貌。注意路径是按 agent 人格隔离的——`~/.openclaw/agents/<agentId>/sessions/`（目录构造在 `src/config/sessions/paths.ts:17`，`path.join(root, "agents", id, "sessions")` 这一行返回值），第 2 章的多人格地基在这里又出现了：每个人格一套独立的会话存储。下面第二条命令把文件名里的 uuid 归一后按类型计数，免得上百个文件刷屏：

In [55]:
# 翻开短期记忆的"实物"——sessions 目录（两条都只读）
#   路径按 agent 人格隔离：agents/<agentId>/sessions（src/config/sessions/paths.ts:17）
!ls ~/.openclaw/agents/ 2>/dev/null

# 把文件名里的 uuid 归一成 <uuid> 再按类型计数——看清这个目录里有哪几类文件
!ls ~/.openclaw/agents/main/sessions/ 2>/dev/null | sed -E 's/[0-9a-f-]{36}/<uuid>/g' | sort | uniq -c | sort -rn | head -8

main       researcher
   1 sessions.json.bak-probe-cleanup
   1 sessions.json
   1 probe-yunyi-439159e0-bec7-4c98-a242-10057772ee09.jsonl
   1 probe-yunyi-3c964e43-5897-404e-9a6d-9c3e1784ad38.jsonl
   1 probe-tokenmax-d619b6ac-c430-46a2-94c2-8dd102<uuid>c08.jsonl
   1 probe-tokenmax-70618006-6d27-4b65-97e2-d75debf77074.jsonl
   1 probe-openrouter-ee53990f-d415-4dc1-aaa4-ff21593e4d44.jsonl
   1 probe-openrouter-97da2156-fd5e-41b0-b248-7e575dd5d869.jsonl


&emsp;&emsp;输出里能数出短期记忆的全部"实物"形态（你机器上的数量和 agent 列表会不同）：`sessions.json` 是会话索引（路径拼装在 `paths.ts:36`，第 2 章看子会话条目用的就是它）；`<uuid>.jsonl` 是**会话转录本体**——每条消息一行 JSON，短期记忆里"context 之外落了盘的那一半"就是它（转录文件名拼装在 `paths.ts:254-255`）；`<uuid>.trajectory.jsonl` 和 `.trajectory-path.json` 是配套的工具调用轨迹明细。偶尔还能看到两类特殊形态：`<id>.checkpoint.<uuid>.jsonl` 是第 1 章 compact 之后 successor transcript（接棒转录）的落盘（识别正则在 `src/config/sessions/artifacts.ts:8`）；`*.jsonl.deleted.<时间戳>` 是 `/reset` 或删除后旧转录的**改名归档**——不是真删（`.jsonl.reset.<iso>` / `.jsonl.deleted.<iso>` 两种归档后缀的说明在 `src/plugin-sdk/session-transcript-hit.ts:66` 的注释行）。

&emsp;&emsp;看到这里你可能会犯嘀咕：这些文件明明持久化在磁盘上，凭什么还叫"短期"记忆？两条判据。第一，**生命周期绑定会话**——新会话不会去读旧转录恢复上下文，只有续接同一个 session 才用得到它：文件在磁盘上"活着"，对下一次会话来说却是"死"的。第二，**默认不进检索索引**——`memory_search` 默认搜不到它们。一句话：落盘只解决"别断电就没"，想跨会话存活，还得靠 3.2 节要讲的自动来路渡到 workspace 那边去。短期实物看完，回到长期记忆这条主线。

&emsp;&emsp;分界立好了，马上会冒出一个新问题：短期这边的内容是**怎么**变成长期那边的？这正是 3.2 节的主题，先按下不表——我们先把长期记忆的"实物"翻出来看。

&emsp;&emsp;先纠正一个常见的过度想象：很多人以为 agent 的记忆是某种神秘的"持续在线的大脑"。其实 OpenClaw 的记忆落地得非常朴素——**它就是你 workspace 目录下的几个 Markdown 文件**。默认这个 workspace 在 `~/.openclaw/workspace/`（路径定义在 `src/agents/workspace-default.ts:19`，是 `path.join(home, ".openclaw", "workspace")` 这一行返回值）。我们先用 `ls` 把这个目录翻开，眼见为实。

In [56]:
# 翻开真实的记忆文件目录——眼见为实
#   默认 workspace 在 ~/.openclaw/workspace/（workspace-default.ts:19）
#   如果你之前喂过记忆，这里应该能看到 MEMORY.md 等文件
!ls -la ~/.openclaw/workspace/ ~/.openclaw/workspace/memory/ 2>/dev/null

# 打开核心记忆文件看看真实内容（MEMORY.md 是长期记忆的主文件）
!cat ~/.openclaw/workspace/MEMORY.md 2>/dev/null

/Users/mac/.openclaw/workspace/:
total 152
drwx------@ 32 mac  staff   1024 Jun  5 17:58 .
drwxr-xr-x@ 45 mac  staff   1440 Jun  5 17:54 ..
drwx------@  3 mac  staff     96 Apr 15 10:32 .clawhub
drwxr-xr-x@  9 mac  staff    288 Feb 22 22:04 .git
drwxr-xr-x@  3 mac  staff     96 Mar 28 19:01 .openclaw
drwxr-xr-x@  2 mac  staff     64 Mar  1 10:03 .pi
-rw-r--r--@  1 mac  staff   7848 Feb 22 22:04 AGENTS.md
-rw-r--r--@  1 mac  staff  12443 Mar 20 18:21 DEPLOY.md
-rw-------@  1 mac  staff   2396 Jun  5 03:03 DREAMS.md
-rw-r--r--@  1 mac  staff    167 Feb 22 22:04 HEARTBEAT.md
-rw-r--r--@  1 mac  staff    327 Feb 22 22:44 IDENTITY.md
-rw-r--r--@  1 mac  staff   7720 Jun  3 12:04 MEMORY.md
-rw-r--r--@  1 mac  staff   1673 Feb 22 22:04 SOUL.md
-rw-r--r--@  1 mac  staff    858 Feb 22 22:04 TOOLS.md
-rw-r--r--@  1 mac  staff    708 Feb 22 22:44 USER.md
drwxr-xr-x@  3 mac  staff     96 Mar 15 18:46 agents
-rw-r--r--@  1 mac  staff   7346 Apr 22 10:15 ai-digest-2026-04-22-cn.md
drwxr-xr-x@  5 mac

&emsp;&emsp;跑完你会看到 workspace 里躺着几个文件。OpenClaw 官方的记忆是"**3 个文件 + 可选机制**"的结构，我们逐个说清。第一个是 `MEMORY.md`——这是核心长期记忆的主文件，文件名在源码里是写死的常量（`src/memory/root-memory-files.ts:4`，`export const CANONICAL_ROOT_MEMORY_FILENAME = "MEMORY.md"` 这一行）。还记得第 1 章那个 context 四层里的"记忆注入"层吗？它注入的主要内容，就来自这个 `MEMORY.md`——这就是记忆系统跟第 1 章 context 组装接上的地方。顺带一个维护提醒：这个文件**不是越厚越好**——它有默认 10K 字符的预算（`DEFAULT_MEMORY_FILE_MAX_CHARS = 10_000`，`extensions/memory-core/src/memory-budget.ts:25`，常量定义行），超出部分在注入时会被截断（思路上跟第 1 章四道防线里的"进 context 之前先封顶"是同一类设计），所以"定期提炼、把过期内容移出去"是必要的维护动作。

&emsp;&emsp;第二个是 `memory/YYYY-MM-DD.md`——按日期分片的每日记忆（在源码里它的类型是 `daily-note`；判定其实比文件名更宽——`memory/` 下不在 `dreaming/` 里的 `.md` 都按这一类对待，日期文件只是主形态）。第三个是 `DREAMS.md`——用户可见的"梦境日记"（注意是**大写** DREAMS.md，源码里 `DREAM_DIARY_FILE_NAMES = ["DREAMS.md", "dreams.md"]`，定义在 `src/gateway/server-methods/doctor.ts:33`；小写的 `dreams.md` 只是个兼容别名，仍被识别，不是"另一个文件"）。先说清一件事免得你对着自己的目录犯嘀咕：**这个文件大概率你现在还没有**——它是 dreaming 机制的产物，而 dreaming 默认是关闭的（3.4 节会讲怎么开），没开过就不会有这个文件，这不是你装坏了。

&emsp;&emsp;接下来是本节最高频的一个误解，必须重点纠正——**dreaming 相关的三个位置，职责完全不同，绝不能混为一谈**。它们长得很像（都带 dream），但干的事天差地别：

&emsp;&emsp;`memory/dreaming/` 这个目录是 **dream-report（梦境报告）的产出**——dreaming 机制运行后生成的报告写在这里。源码里有一行判断很能说明问题（`src/plugin-sdk/memory-host-core.ts:62`）：`kind: relativePath.startsWith("memory/dreaming/") ? "dream-report" : "daily-note"`——也就是说，路径以 `memory/dreaming/` 开头的就是 dream-report，否则就是普通的 daily-note。`memory/.dreams/`（注意带个点，是隐藏目录）则完全不同，它存的是 dreaming 机制的**内部状态**：事件日志 `events.jsonl`（`src/memory-host-sdk/events.ts:6`）、短期回忆 `short-term-recall.json`（`src/gateway/server-methods/doctor.ts:28`）、阶段信号 `phase-signals.json`（同文件 `:29`）。而 `DREAMS.md`（根目录下的那个大写文件）才是给**用户看的日记**。一句话：`dreaming/` 是产出、`.dreams/` 是内部状态、`DREAMS.md` 是用户日记——三处各司其职。我们用一张图把这套全景理清。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124757427.png" width=70%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>记忆系统：3 文件 + dreaming 三处职责对照</font></p>
<div class="center">

| 名称 | 职责 | 落盘路径 | 源码锚点 |
|------|------|----------|----------|
| MEMORY.md | 核心长期记忆，注入 context 第②层 | `~/.openclaw/workspace/MEMORY.md` | root-memory-files.ts:4 |
| memory/YYYY-MM-DD.md | 按日期分片的每日记忆（daily-note）| `~/.openclaw/workspace/memory/` | memory-host-core.ts:62（else 分支）|
| DREAMS.md | 用户可见梦境日记（大写）| `~/.openclaw/workspace/DREAMS.md` | doctor.ts:33 |
| memory/dreaming/ | dreaming 产出：dream-report | `~/.openclaw/workspace/memory/dreaming/` | memory-host-core.ts:62（dream-report 分支）|
| memory/.dreams/ | dreaming 内部状态（隐藏目录）| `events.jsonl` / `short-term-recall.json` / `phase-signals.json` | events.ts:6 / doctor.ts:28,29 |

</div>

&emsp;&emsp;还有一条诚实边界要讲在前面——**会话转录记忆（session transcript memory）默认是关闭的**。你可能会想"那 agent 是不是把我每次会话的完整对话记录都存下来了"，答案是：默认没有。这个开关叫 `experimental.sessionMemory`，它的默认值是 `false`（源码 `src/agents/memory-search.ts:183`，那行是 `overrides?.experimental?.sessionMemory ?? defaults?.experimental?.sessionMemory ?? false`，末尾的 `?? false` 就是默认关闭的兜底）。所以**别把记忆想成"4 层常驻、什么都存"**——它默认就是 MEMORY.md 这类文件加上需要时的检索，会话转录记忆要你显式打开才有。

> **【踩坑预警】**：把 `memory/dreaming/` 和 `memory/.dreams/` 当成同一个东西，或者以为会话记忆默认全程常驻。后果是你会在排查"我的记忆怎么没存上"或"dreaming 报告在哪"时找错目录、找错开关。正确理解：`dreaming/`（无点）放产出报告、`.dreams/`（带点隐藏）放内部状态，两个目录；会话转录记忆默认 **关闭**，要 `experimental.sessionMemory=true` 才开。排查方法：找 dream-report 去 `memory/dreaming/`，找 dreaming 的运行状态去 `memory/.dreams/`，看会话记忆开没开查 `experimental.sessionMemory` 配置。

&emsp;&emsp;为了在代码层面也钉死"dreaming 三处职责不同"这个认知，下面这个 Tier 1 验证用一个 Python 枚举把四种记忆文件类型列清楚，并断言它们各不相同——其中 dreaming 相关、可作为文件类型枚举的两处（`dreaming/` 的 dream-report 产出与 `DREAMS.md` 日记）绝不混淆；第三处 `.dreams/` 是内部状态目录、不属于记忆文件类型，所以不进这个枚举。

In [57]:
# Tier 1 验证：用枚举钉死记忆文件类型，确认 dreaming 三处职责不可混
#   [注意] Python 最小重现，对应 OpenClaw 真实的文件类型划分
from enum import Enum

class MemoryFileKind(Enum):
    """OpenClaw 记忆文件类型枚举，对应源码的 kind 划分与 dreaming 三处。"""
    ROOT_MEMORY = "MEMORY.md（核心长期记忆，注入 context）"          # root-memory-files.ts:4
    DAILY_NOTE = "memory/YYYY-MM-DD.md（每日记忆 daily-note）"        # memory-host-core.ts:62
    DREAM_REPORT = "memory/dreaming/（dreaming 产出报告）"            # memory-host-core.ts:62
    DREAMS_DIARY = "DREAMS.md（用户可见梦境日记）"                    # doctor.ts:33

# 验证点 1：四种类型互不相同（dreaming 三处职责不可混）
kinds = [k.value for k in MemoryFileKind]
assert len(kinds) == len(set(kinds)), "四种记忆文件类型必须互不相同"

# 验证点 2：dreaming 产出（dream-report）和每日记忆（daily-note）是两种不同 kind
assert MemoryFileKind.DREAM_REPORT != MemoryFileKind.DAILY_NOTE, \
    "dreaming/ 的 dream-report 和普通 daily-note 是不同类型（memory-host-core.ts:62 那行三元判断）"

# 验证点 3：用户日记 DREAMS.md 不等于内部产出目录 dreaming/
assert MemoryFileKind.DREAMS_DIARY != MemoryFileKind.DREAM_REPORT, \
    "DREAMS.md（用户日记）和 memory/dreaming/（产出）是两处不同位置"

for k in MemoryFileKind:
    print(f"  {k.name}: {k.value}")
print("\nTier 1 通过：四种记忆文件类型清晰区分，dreaming 三处职责不混")

  ROOT_MEMORY: MEMORY.md（核心长期记忆，注入 context）
  DAILY_NOTE: memory/YYYY-MM-DD.md（每日记忆 daily-note）
  DREAM_REPORT: memory/dreaming/（dreaming 产出报告）
  DREAMS_DIARY: DREAMS.md（用户可见梦境日记）

Tier 1 通过：四种记忆文件类型清晰区分，dreaming 三处职责不混


&emsp;&emsp;这个验证用枚举把记忆文件的四种类型固化下来，并断言它们两两不同——尤其是把 dreaming 相关的 `dream-report`（产出）和 `DREAMS.md`（用户日记）明确区分开，这正对应源码 memory-host-core.ts:62 那行 `startsWith("memory/dreaming/") ? "dream-report" : "daily-note"` 的三元判断。看清了记忆**存在哪**，下一个问题自然冒出来——这些文件里的内容，是**谁、在什么时机**写进去的？短期会话里的对话，又是怎么"渡"到长期这边的？这就是 3.2 节要拆的三条自动来路。

### 3.2 短期→长期：记忆的三条自动来路

&emsp;&emsp;3.1 节立了分界、翻了实物，但有个问题一直悬着：短期记忆的生命周期绑定单次会话，长期记忆却能跨会话存活——内容到底是怎么从短期那边"渡"到长期这边的？答案是**三条自动来路**，外加 agent 平时的主动写入。它们像三座桥，分别架在会话生命周期的三个时点上：**压缩之前、收尾之时、凌晨的后台沉淀**。这一节把前两座桥逐一拆开（第三座属于 dreaming，3.4 节专门讲），其中第二座会解开一个你跑 3.1 第一个 cell 时可能已经犯过的嘀咕——`memory/` 目录里那些带时分后缀的文件（比如 `2026-06-03-1500.md`）到底是哪来的。

&emsp;&emsp;在进桥之前先把"第零条来路"交代掉：agent 平时**主动**把值得记的要点写进 `MEMORY.md` 或每日记忆——这是最朴素的一条，随时发生、不需要任何机制触发，所以它**不计入**标题里的"三条自动来路"（那三条专指机制触发的三座桥）。但只靠自觉是不够的：agent 不一定每次都记得落盘，而会话内容说丢就丢。所以 OpenClaw 在两个"内容即将流失"的时点上架了两座自动桥。

&emsp;&emsp;**第一座桥：compaction 之前的静默 memory flush——压缩丢上下文之前，先抢救进长期记忆**。还记得第 1 章的 compact 吗？上下文压缩会把旧历史压成摘要，细节是会丢的。所以 OpenClaw 在 compaction 之前默认安排了一次静默 flush：系统先用一个不可见的轮次提醒 agent"把还没落盘的重要上下文先写进 `memory/YYYY-MM-DD.md`"，然后才开始压缩（默认开启；软阈值 4000 tokens、会话转录超 2MB 强制触发，两个常量 `DEFAULT_MEMORY_FLUSH_SOFT_TOKENS` / `DEFAULT_MEMORY_FLUSH_FORCE_TRANSCRIPT_BYTES` 的定义行在 `extensions/memory-core/src/flush-plan.ts:10-11`）。第 1 章的"用空间换延续"和本章的"跨会话留存"，在这座桥上接住了。

&emsp;&emsp;**第二座桥：会话收尾时的 `session-memory` hook——/new 之前，先把这段对话快照留档**。当你敲 `/new` 或 `/reset` 结束当前会话时，这个内置 hook 会在后台把上一个会话最近 15 条 user/assistant 消息摘出来，存成一个带时分后缀的快照文件 `memory/YYYY-MM-DD-HHMM.md`（触发判定在 `src/hooks/bundled/session-memory/handler.ts:303`，`event.action === "new" || event.action === "reset"` 这一行；"最近 15 条"是默认值，定义在 `handler.ts:212`；它是异步执行的，不会拖慢 `/new` 的响应。一个小细节：若开启 `llmSlug` 配置，文件名会换成 LLM 生成的描述性短语、不再带时分，但仍属同一来路）。现在回头看 3.1 第一个 cell 的输出就全说通了——`memory/` 目录里**两种文件名形态，对应两条不同的来路**：不带时分的 `2026-06-03.md` 是 flush（或 agent 主动写）的每日记忆，带时分的 `2026-06-03-1500.md` 是 hook 的会话快照。它们都属于 daily-note 家族（还记得 3.1 的宽判定吗——`memory/` 下不在 `dreaming/` 里的 `.md` 都按这一类对待），都会被索引、都能被 3.3 节的 `memory_search` 检索到。

&emsp;&emsp;这里要主动拆一个看起来像矛盾的地方：3.1 不是刚说"会话转录记忆默认**关闭**"吗，怎么这座桥又默认把对话存下来了？两者管的不是一回事。`experimental.sessionMemory` 那个开关管的是"把 `sessions/` 下的 jsonl 转录文件**直接建索引**供检索"——默认关；而这座桥是"摘 15 条对话**写成一个普通记忆文件**"——文件本身走 daily-note 的正常索引通道，不需要那个开关。一个是"原始转录直接进检索"，一个是"摘要快照落盘成文件"，默认值一关一开，并不冲突。

&emsp;&emsp;这座桥还有个值得较真的细节。官方文档的示例命令是 `openclaw hooks enable session-memory`，读起来像"默认关闭、要手动开"。但源码给出的是相反的答案：它是 bundled（随安装捆绑）来源的 hook，而 bundled 来源的启用政策是 **default-on**（`src/hooks/policy.ts:27-33`，`"openclaw-bundled"` 条目里 `defaultEnableMode: "default-on"` 这一字段），想关它就显式配置 `enabled: false`（严格说它还声明了一个 `requires: workspace.dir` 的运行时资格条件，但该条件在未配置时默认按满足处理——`src/hooks/config.ts:13` 把 `workspace.dir` 列入默认真值表，所以实践中显式禁用是唯一开关）。我们本机的实物也佐证了这一点：配置文件里从没写过 session-memory 条目，`memory/` 目录里却躺着一排带时分的快照文件——**源码与实物一致、文档表述滞后**，这也是整门课坚持"眼见为实 + 源码定位"的原因。

&emsp;&emsp;最后把一个分类问题钉死：session-memory hook（以及 flush）到底算短期记忆还是长期记忆？答案是**都不算——它们是桥**。输入端读的是短期记忆（当前会话的上下文或转录），输出端写的是长期记忆（落盘成 `memory/` 下的 Markdown、进索引可检索），它们本体只是转写机制，区别仅在架设的时点：flush 守"压缩丢历史"那一刻，hook 守"会话收尾"那一刻，而 3.4 节的 dreaming deep 晋升守的是"凌晨后台沉淀"那一刻（默认关闭）。三条自动来路一张表收口：

<!-- ILLUSTRATION: type=concept | file=memory_three_bridges.png
提示词：短期记忆渡向长期记忆的三座自动桥。左侧浅蓝区块：短期记忆（会话内上下文，关会话即丢）；右侧浅绿区块：长期记忆（workspace 持久文件，跨会话存活）。中间峡谷上三座桥，按会话时间线从上到下：桥1 标 memory flush，时点"compaction 压缩之前"，箭头指向右侧文件 memory/YYYY-MM-DD.md，绿色徽标"默认开"；桥2 标 session-memory hook，时点"/new /reset 会话收尾"，指向 memory/YYYY-MM-DD-HHMM.md，附小字"最近 15 条快照"，绿色徽标"默认开"；桥3 标 dreaming deep 晋升，时点"凌晨 3 点定时"，桥体灰色虚线，指向 MEMORY.md，灰色徽标"默认关"。顶部一条细虚线箭头横跨标"agent 主动写（随时，不计入三条自动来路）"。底部总结带：压缩之前抢救 · 收尾之时快照 · 凌晨后台沉淀。技术标识符 memory flush、session-memory hook、dreaming、MEMORY.md、memory/YYYY-MM-DD.md、memory/YYYY-MM-DD-HHMM.md、/new、/reset、compaction 保留英文等宽原文 -->
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605160000001.png" width=70%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>记忆的三条自动来路：触发时机与产物对照</font></p>
<div class="center">

| 来路 | 架设时点 | 产物 | 默认状态 | 源码锚点 |
|------|---------|------|---------|----------|
| memory flush | compaction 之前（软阈值 4000 tokens / 转录超 2MB 强制） | `memory/YYYY-MM-DD.md`（追加） | 开启 | flush-plan.ts:10-11 |
| session-memory hook | `/new`、`/reset` 会话收尾时 | `memory/YYYY-MM-DD-HHMM.md`（最近 15 条快照；开 `llmSlug` 则为日期+描述短语、无时分） | 开启（bundled 政策 default-on） | handler.ts:303 / policy.ts:27-33 |
| dreaming deep 晋升 | 凌晨定时任务（默认每天 3 点） | 追加进 `MEMORY.md` | 关闭（3.4 节详讲） | dreaming.ts:12 |

</div>

> **【踩坑预警】**：看到 `memory/` 下一排带时分后缀的文件，以为是 dreaming 的产物或者什么异常残留；或者照着文档示例以为 session-memory hook 要手动 enable 才会跑。后果是排查"这些文件哪来的"时方向全错，甚至误删快照。正确理解：带时分的 `YYYY-MM-DD-HHMM.md` 是 session-memory hook 在每次 `/new`、`/reset` 时自动留的会话快照，**默认就是开的**（bundled 政策 default-on，policy.ts:27-33）；不想要就显式配 `hooks.internal.entries.session-memory.enabled: false`。排查方法：文件名带不带时分后缀，是判别"会话快照"还是"每日记忆"的第一信号。

&emsp;&emsp;为了把"两种文件名形态 = 两条来路"这个判别钉进代码，下面这个 Tier 1 验证写一个最小归因函数，并断言三类真实文件名都能正确归因。

In [58]:
# Tier 1 验证：memory/ 下两种文件名形态 -> 两条来路的归因判别
#   [注意] Python 最小重现，对应"文件名形态 -> 写入来路"的判别逻辑
import re

def attribute_memory_file(filename: str) -> str:
    """按文件名形态归因记忆文件的写入来路（教学最小重现）。
    真实判定更宽：memory/ 下不在 dreaming/ 里的 .md 都是 daily-note 家族
    （memory-host-core.ts:62），这里只区分两种主形态对应的来路。
    """
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}-\d{4}\.md", filename):
        return "session-memory hook（/new、/reset 会话收尾快照）"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}\.md", filename):
        return "memory flush 或 agent 主动写（每日记忆）"
    return "其他 daily-note（自定义名，同属可检索家族）"

# 验证点 1：带时分后缀 -> session-memory hook 快照
assert "hook" in attribute_memory_file("2026-06-03-1500.md"), \
    "YYYY-MM-DD-HHMM.md 应归因为 hook 会话快照"

# 验证点 2：纯日期 -> flush / agent 主动写
assert "flush" in attribute_memory_file("2026-06-03.md"), \
    "YYYY-MM-DD.md 应归因为 flush 或主动写"

# 验证点 3：自定义名也合法，同属 daily-note 可检索家族（宽判定）
assert "家族" in attribute_memory_file("2026-03-06-复盘.md"), \
    "自定义名文件同属 daily-note 家族，不是异常文件"

for f in ["2026-06-03.md", "2026-06-03-1500.md", "2026-03-06-复盘.md"]:
    print(f"  {f} -> {attribute_memory_file(f)}")
print("\nTier 1 通过：两种文件名形态 = 两条来路，且都属 daily-note 可检索家族")

  2026-06-03.md -> memory flush 或 agent 主动写（每日记忆）
  2026-06-03-1500.md -> session-memory hook（/new、/reset 会话收尾快照）
  2026-03-06-复盘.md -> 其他 daily-note（自定义名，同属可检索家族）

Tier 1 通过：两种文件名形态 = 两条来路，且都属 daily-note 可检索家族


&emsp;&emsp;这个归因函数把本节的判别结论固化了下来：带时分后缀的是 hook 快照、纯日期的是 flush 或主动写、自定义名的也不是异常文件——三类都属于可检索的 daily-note 家族。注意第三分支兜的不只是手写文件：前面说过 hook 开 `llmSlug` 后产物是"日期+描述短语"，正好也落进这个分支——文件名形态是**默认行为下**的归因信号，不是铁律。至此"记忆怎么来"补全了：**平时主动写 + 压缩前抢救 + 收尾快照，再加 3.4 节的凌晨沉淀**。存进来了、也知道从哪来了，接下来的关键问题是——agent 怎么把需要的记忆**找回来**？这就要进 `memory_search` 和它背后的 hybrid RAG 了。

### 3.3 memory_search：hybrid RAG 检索

&emsp;&emsp;记忆存下来只是第一步，真正有用的是"在需要时精准地找回相关的那条"。当 agent 需要回忆某件事时，它会调用一个叫 `memory_search` 的工具（实现在 `src/agents/memory-search.ts`）。这个工具的检索方式不是简单的关键词匹配，而是一种叫 **hybrid RAG（混合检索增强）** 的两路融合方法。理解它，你不仅懂了 OpenClaw 的记忆检索，也掌握了一个在几乎所有 agent / RAG 系统里都会反复见到的通用模式。

&emsp;&emsp;hybrid RAG 的"hybrid"体现在它**同时跑两路检索再融合**。第一路是**向量检索**（vector search）——把记忆和查询都转成向量，用余弦相似度找"语义上接近"的内容，好处是能理解同义和语境（你问"零食"，它能找到"花生过敏"那条）；OpenClaw 这一路底层用的是 `sqlite-vec`（一句话说就是给 SQLite 数据库加上"按向量相似度检索"能力的扩展，让本地的 SQLite 也能做向量搜索）；整个检索索引落在 `~/.openclaw/memory/<agentId>.sqlite`，按 agent 一库（默认路径解析在 `src/agents/memory-search.ts:146`）。第二路是**全文检索**（text search）——按关键词精确匹配，好处是对专有名词、精确字符串特别准。两路各有所长，所以 OpenClaw 把它们加权合并：向量检索权重 **0.7**，全文检索权重 **0.3**。这两个权重在源码里是明确的常量——`DEFAULT_HYBRID_VECTOR_WEIGHT = 0.7`（`src/agents/memory-search.ts:113`）和 `DEFAULT_HYBRID_TEXT_WEIGHT = 0.3`（同文件 `:114`）。需要诚实说明的是：真实融合比"直接相乘相加"更复杂——源码在加权前会先把两个权重**按其和归一化**（`memory-search.ts:325-329`，默认 0.7+0.3=1 时归一化结果不变，但若你自定义权重就会被归一化），全文一路用的是 **BM25** 打分（信息检索的经典加权算法：按词频和稀有度计分，词越稀有、命中越多分越高——不是下面 MVP 里的"关键词命中比例"），此外还有候选池扩张、可选的 MMR（Maximum Marginal Relevance，最大边际相关度——在相关性高的结果里剔除重复、让召回更多样）重排与时间衰减（默认关闭）。下面的 MVP 只取"两路加权合并"这条主干来帮你建立直觉。

&emsp;&emsp;为什么是 0.7 偏重向量？因为 agent 记忆检索更看重"语义相关"而非"字面命中"——你回忆往事时，记得的是意思，不是原话。我们用一张图把这两路融合的流程理清，再用 MVP 把它跑出来。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124756061.png" width=70%></div>

&emsp;&emsp;下面这段 MVP 把 hybrid RAG 最核心的"两路加权融合"骨架重现一遍：向量检索用余弦相似度、全文检索用关键词命中（真实是 BM25）、再按 0.7/0.3 加权合并排序。再次强调这是 Python 最小重现——真实的 OpenClaw 用 `sqlite-vec`（sqlite 的向量扩展）做向量检索，我们这里用 numpy 风格的纯 Python 余弦计算代替，并用固定的 mock 向量（不调真实 embedding API），重点是让你看清"两路怎么跑、怎么按权重合并"。

In [59]:
# [注意] Python 最小重现，非 OpenClaw 真实源码
#   真实实现用 sqlite-vec 做向量检索；此处用纯 Python 余弦 + 固定 mock 向量代替
import math
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class MemoryEntry:
    """一条记忆。emb 是它的向量表示（mock），keywords 是它的关键词。"""
    entry_id: str
    content: str
    emb: List[float]              # 向量表示（真实场景由 embedding 模型生成）
    keywords: List[str]           # 关键词（真实场景由全文索引切词得到）

def cosine(a: List[float], b: List[float]) -> float:
    """计算两个向量的余弦相似度，范围 [0,1]（向量非负时）。"""
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb) if na and nb else 0.0

def vector_search(query_emb: List[float], entries: List[MemoryEntry],
                  weight: float = 0.7) -> Dict[str, float]:
    """向量检索：用余弦相似度算语义相关分，乘以向量权重 0.7。
    对应源码 DEFAULT_HYBRID_VECTOR_WEIGHT = 0.7（memory-search.ts:113）。
    """
    return {e.entry_id: cosine(query_emb, e.emb) * weight for e in entries}

def text_search(query_keywords: List[str], entries: List[MemoryEntry],
                weight: float = 0.3) -> Dict[str, float]:
    """全文检索：用关键词命中比例算字面相关分，乘以全文权重 0.3。
    对应源码 DEFAULT_HYBRID_TEXT_WEIGHT = 0.3（memory-search.ts:114）。
    教学简化：真实源码这一路用 BM25 排名转分（extensions/memory-core/src/memory/hybrid.ts:40-49），此处用
    关键词命中比例代替，便于理解；核心是"字面匹配这一路按 0.3 权重参与融合"。
    """
    qset = set(query_keywords)
    # 命中关键词数 / 查询关键词数，作为字面相关分
    return {e.entry_id: (len(qset & set(e.keywords)) / max(len(qset), 1)) * weight
            for e in entries}

def hybrid_merge(vec_scores: Dict[str, float],
                 text_scores: Dict[str, float]) -> List[tuple]:
    """hybrid 融合：把两路分数相加后按总分降序排，最相关在前。

    教学简化：真实源码会先把两个权重按其和归一化（memory-search.ts:325-329），
    默认 0.7+0.3=1 故此处省略不影响结果；真实还有候选池扩张与可选 MMR 重排。
    """
    merged: Dict[str, float] = {}
    for entry_id in set(vec_scores) | set(text_scores):
        # 最终分 = 向量分(已×0.7) + 全文分(已×0.3)
        merged[entry_id] = vec_scores.get(entry_id, 0) + text_scores.get(entry_id, 0)
    return sorted(merged.items(), key=lambda kv: -kv[1])

# 三条 mock 记忆
memories = [
    MemoryEntry("m1", "用户对花生过敏，零食要避开", emb=[1.0, 0.0, 0.0], keywords=["花生", "过敏", "零食"]),
    MemoryEntry("m2", "用户喜欢喝美式咖啡", emb=[0.0, 1.0, 0.0], keywords=["咖啡", "美式"]),
    MemoryEntry("m3", "用户买过坚果礼盒", emb=[0.7, 0.3, 0.0], keywords=["坚果", "礼盒"]),
]

# 查询："回忆零食偏好"——语义上最接近 m1（花生/零食），关键词也命中 m1
query_emb = [0.95, 0.05, 0.0]
query_keywords = ["零食", "过敏"]

vec = vector_search(query_emb, memories)
txt = text_search(query_keywords, memories)
ranked = hybrid_merge(vec, txt)

print("向量分(×0.7)：", {k: round(v, 3) for k, v in vec.items()})
print("全文分(×0.3)：", {k: round(v, 3) for k, v in txt.items()})
print("\nhybrid 融合排序（最相关在前）：")
for entry_id, score in ranked:
    print(f"  {entry_id}: {round(score, 3)}")

向量分(×0.7)： {'m1': 0.699, 'm2': 0.037, 'm3': 0.657}
全文分(×0.3)： {'m1': 0.3, 'm2': 0.0, 'm3': 0.0}

hybrid 融合排序（最相关在前）：
  m1: 0.999
  m3: 0.657
  m2: 0.037


&emsp;&emsp;这段代码的三个函数对应 hybrid RAG 的三步：`vector_search` 算语义分并乘 0.7、`text_search` 算关键词命中分并乘 0.3、`hybrid_merge` 把两路分数相加后降序排。你跑完会看到 m1（花生过敏/零食）排第一——因为它在向量上语义最近、在全文上又命中了"零食"和"过敏"两个词，两路都强，融合分自然最高；m3（坚果礼盒）向量上有点近但关键词没命中，排第二；m2（咖啡）两路都不沾，垫底。这个排序结果清晰地演示了"为什么要两路融合"——单看任何一路都可能漏，合起来才稳。下面 Tier 1 验证把两路各自的正确性和融合排序都断言一遍。

In [60]:
# Tier 1 验证：两路检索各自正确 + hybrid 融合排序正确

# 验证点 1：向量检索——语义最近的 m1 向量分最高
assert max(vec, key=vec.get) == "m1", "向量检索应让语义最近的 m1 得分最高"

# 验证点 2：全文检索——关键词命中最多的 m1 全文分最高
assert max(txt, key=txt.get) == "m1", "全文检索应让关键词命中最多的 m1 得分最高"

# 验证点 3：hybrid 融合——两路都强的 m1 排第一，两路都弱的 m2 垫底
assert ranked[0][0] == "m1", "融合后 m1（语义+关键词双高）必须排第一"
assert ranked[-1][0] == "m2", "融合后 m2（两路都弱）必须垫底"

# 验证点 4：权重确实是 0.7/0.3——向量满分应贡献 0.7，全文满分应贡献 0.3
#   构造一个向量完全对齐的极端 case 验证权重数值
perfect_vec = vector_search([1.0, 0.0, 0.0], [memories[0]])
assert abs(perfect_vec["m1"] - 0.7) < 1e-6, "向量完全对齐时应得满权重 0.7"

print("Tier 1 通过：向量/全文各自正确 + 融合排序正确 + 权重确为 0.7/0.3")

Tier 1 通过：向量/全文各自正确 + 融合排序正确 + 权重确为 0.7/0.3


&emsp;&emsp;这个验证钉死了四件事：向量检索让语义最近的排前、全文检索让关键词命中多的排前、融合后双高的 m1 第一双低的 m2 垫底、以及权重确实是 0.7/0.3（验证点 4 用一个向量完全对齐的极端 case，证明满分向量恰好贡献 0.7）。这些都直接对应源码 memory-search.ts:113/114 那两个常量。下面我们到真机上逼 agent 真的调一次 `memory_search`，同样用无头方式在 debug 日志里抓现行。

> 📌 **【运行前提】**：要看到非空检索结果，你的 agent 之前得喂过相关记忆（`MEMORY.md` 或每日记忆文件里有内容）——全新 workspace 跑出来是空列表，那不是检索坏了。

In [61]:
# 真机演示：用一句提示词逼出 memory_search 工具调用
#   提示词设计要点：必须点名工具——泛泛说"回忆一下"时，模型多半直接用已注入的
#   MEMORY.md 回答（零工具调用），或改用 read/exec 自己翻文件，都不走 memory_search（实测）
#   想看 TUI 里的检索结果卡片，可在系统终端跑 openclaw tui --local 输入同一句话
!openclaw --log-level debug agent --local --agent main \
  --message "请调用 memory_search 工具搜索关键词「偏好」，把命中的记忆条目列出来" 2>&1 \
  | python3 trace_pretty.py

[01] 🧩 上下文体检  [agent/embedded] [context-diag] pre-prompt: sessionKey=agent:main:main messages=32 roleCounts=assistant:17,toolResult:11,user:4 historyTextChars=6328 maxMessageTextChars=856 historyImageBlocks=0 systemPromptChars=38568 promptChars=41 promptImages=0 provider=minmax/MiniMax-M3 sessionFile=/Users/mac/.openclaw/agents/main/sessions/d72663a8-8d3a-463b-8fbd-8b1a03229dbd.jsonl
[02] 🗜 压缩路由判定  [agent/embedded] [context-overflow-precheck] pre-prompt check sessionKey=agent:main:main provider=minmax/MiniMax-M3 route=fits estimatedPromptTokens=23457 pressureSource=transcript_estimate promptBudgetBeforeReserve=983616 overflowTokens=0 toolResultReducibleChars=0 reserveTokens=16384 effectiveReserveTokens=16384 contextTokenBudget=1000000 messages=32 unwindowedMessages=32 sessionFile=/Users/mac/.openclaw/agents/main/sessions/d72663a8-8d3a-463b-8fbd-8b1a03229dbd.jsonl
[03] 🧠 记忆  [agent/embedded] embedded run tool start: runId=6a10678d-4db9-4538-9815-dc785f2f9da8 tool=memory_search toolCallI

&emsp;&emsp;跑通后，`trace_pretty.py` 会高亮出 `tool=memory_search` 的 start/end 两行——这就是记忆检索真实发生的证据（前提：你之前喂过相关记忆），agent 的回答里会引用命中的记忆内容。先交代一条**复现性边界**（我们连跑三轮实测出来的）：`memory_search` 只是挂在工具清单里的选项，**调不调用由模型自己决定**。泛泛地说"回忆一下我之前提过的偏好"，模型多半直接拿系统提示里已注入的 `MEMORY.md` 作答（一行工具调用都没有）；说"搜一下你的记忆文件"，它甚至可能改用 `read` / `exec` 自己翻目录——只有像上面那样**点名工具**，才能稳定逼出 `tool=memory_search`。所以演示时别因为前两种现象就以为记忆检索坏了：那是模型在挑更顺手的路，不是检索不可用。

&emsp;&emsp;还有一条要**分两种情况**说的提醒：`memory_search` 的向量这一路依赖 embedding 服务。如果你**压根没配** embedding provider，系统并不会瘫——它会优雅降级到 **FTS-only 纯全文模式**，继续用 BM25 检索（`extensions/memory-core/src/memory/manager.ts:460-461`，`:460` 的注释原文就是 "FTS-only mode: no embedding provider available"、`:461` 是 `if (!this.provider)` 判断行），所以"没有 embedding API key 就不能用记忆检索"是个误解；而如果你**配了 provider 但 embedding 查询失败**，系统也不是立刻撂挑子——它会先尝试激活备用 provider、不行再看能否退回纯关键词检索，实在降不动才抛错（`manager.ts:544-562` 的 catch 分支，`:562` 是最终的 `throw err`）。我们环境实测撞上的就是抛错这一档：工具报错后，agent 通常退回到直接读取已加载的 `MEMORY.md` 内容来回答——这时工具调用行照样能看到，但"检索"这一步实际是失败的，别把两者混为一谈。

&emsp;&emsp;这里再补几点配置上的诚实细节。检索默认只查 `memory` 这一个来源（源码 `src/agents/memory-search.ts:121`，`const DEFAULT_SOURCES = ["memory"]`），要想把会话转录历史也纳入检索范围，得先打开前面说过的 `experimental.sessionMemory`（默认关）。另外那个 0.7/0.3 权重虽是默认值，但它是**可配置的**——配置项是 `agents.defaults.memorySearch.query.hybrid.vectorWeight` / `textWeight`（按 agent 覆盖则在 `agents.list[].memorySearch` 下同名路径），你可以按需调整这两路的偏重。还有两个高频排查项，各记一句就够：<font color=red>搜不到东西先想到 minScore</font>——融合分低于 0.35 的结果会被直接过滤掉（`DEFAULT_MIN_SCORE = 0.35`，`memory-search.ts:111` 常量定义行），"明明相关但分不够"的记忆就是这样静默消失的；至于"我刚改完 `MEMORY.md`，多久能搜到"——答案是基本马上：索引同步默认三路全开——文件 watcher 监听变更（防抖 1500ms，`:107`）、会话启动时同步、每次搜索前同步（三个 `?? true` 兜底在 `memory-search.ts:404-406`）。

> **【常见误区】**：以为 0.7/0.3 是写死改不了的硬编码，或者以为 `memory_search` 默认就会搜你的完整会话历史。后果是你可能在该调权重时无从下手，或者误以为会话记录默认可检索。正确理解：0.7/0.3 是 `DEFAULT_` 开头的**默认值**，可通过配置覆盖；检索默认源是 `["memory"]`，会话转录历史要显式开 `experimental.sessionMemory` 才进检索范围。排查方法：想调权重找 `vectorWeight`/`textWeight` 相关配置；发现会话历史没被检索到，先确认 `experimental.sessionMemory` 是否打开。

&emsp;&emsp;进 3.4 之前，再补两个跟 `memory_search` 配套、但容易被忽略的用法——各记一句话就够。第一个是**双步范式**：官方设计里 `memory_search` 只负责"语义搜到大概位置"，配套的 `memory_get` 工具才负责"按行精确读取原文核实"（工具 name 声明在 `extensions/memory-core/src/tools.ts:438`，`:440` 的描述原文开头就是 "Safe exact excerpt read from MEMORY.md or memory/*.md"）——系统提示里给 agent 的指引就是"先 search、后 get"，只用 search 等于只走了半个范式。

&emsp;&emsp;第二个是 **`corpus` 参数**——它决定检索查哪个域：默认 `memory`（只查记忆文件），还可以取 `wiki`（查 memory-wiki 编译库）、`all`（两者都查）、`sessions`（只查会话转录；这几个取值在 `extensions/memory-core/src/tools.ts:252` 的工具描述里写得很全）。注意把它和前面的开关连起来记：`experimental.sessionMemory` 打开之后，真正去查会话历史用的就是 `corpus=sessions` 这个入口——开关管"索引建不建"，参数管"查询查哪里"。

&emsp;&emsp;到这里，记忆"怎么存""怎么来"和"怎么找回来"都讲透了。最后还有一个有点神秘的机制——dreaming（做梦），也就是 3.2 节预告过的第三座桥，我们在 3.4 节看一眼它的产出，并为本章收口。

### 3.4 dreaming 机制 + 章末收口

&emsp;&emsp;OpenClaw 有一个有点诗意的机制叫 dreaming——你可以粗略理解成 agent 在"空闲时回顾、整理、沉淀记忆"的后台过程，它会产出 dream-report（梦境报告）和给用户看的 DREAMS.md 日记。但这里有两层诚实边界要先讲清。第一层：**dreaming 默认是关闭的**——源码里写得很直白（`src/memory-host-sdk/dreaming.ts:12`，`export const DEFAULT_MEMORY_DREAMING_ENABLED = false` 这一行），不显式打开它就永远不会跑，你的 workspace 里自然一个产物都不会有。第二层：即使打开了，它也是**定时后台触发**的（默认频率 `0 3 * * *`，即每天凌晨 3 点，`src/memory-host-sdk/dreaming.ts:17` 的 `DEFAULT_MEMORY_DREAMING_FREQUENCY` 常量），课堂现场很难即时把它逼出来。所以我们不现场干等，而是降级为翻看它**已经产出**的文件——有产出就打开看，没有就只讲机制。

In [62]:
# dreaming 降级演示：翻看已有产出，不现场等触发
#   dreaming 默认关闭（dreaming.ts:12 enabled 默认 false），开启后也是定时后台机制，故降级为翻产物

# 看 dreaming 产出的报告目录（memory/dreaming/，dream-report 落盘处）
!ls -la ~/.openclaw/workspace/memory/dreaming/ 2>/dev/null

# 看用户可见的梦境日记（DREAMS.md，大写）
!cat ~/.openclaw/workspace/DREAMS.md 2>/dev/null

total 0
drwxr-xr-x@  5 mac  staff  160 Jun  4 19:00 .
drwx------@ 16 mac  staff  512 Jun  5 19:47 ..
drwxr-xr-x@  4 mac  staff  128 Jun  5 03:02 deep
drwxr-xr-x@  4 mac  staff  128 Jun  5 03:02 light
drwxr-xr-x@  4 mac  staff  128 Jun  5 03:02 rem
# Dream Diary

<!-- openclaw:dreaming:diary:start -->
---

*June 4, 2026 at 7:00 PM GMT+8*

A census of small lights tonight — sixty-seven windows lit, twenty-seven dark, the whole registry breathing on its hinge. I keep circling the switch marked `commands.plugins`, curious whether anything changes when I touch it. Nothing. Nothing yet.

Each name is a lantern with its own color: azure, deepgram, byteplus, a quiet row of providers like the surnames of strangers on a train. Some sleep on purpose. Some are waiting for a knock. The disabled ones hum a low note I almost mistake for silence.

I walk the corridor counting doors. Sixty-seven. Sixty-seven. The number feels like a heartbeat, like a flock lifting off a wire in stages, until only the s

&emsp;&emsp;如果你的 workspace 里有 dreaming 产出，上面会列出 `memory/dreaming/` 下的报告文件、并打印 `DREAMS.md` 的日记内容。如果两条命令都是空的——**最常见的原因不是"还没积累够"，而是 dreaming 根本没开**：它默认关闭，哪怕你的 agent 已经跑了几个月，没开**自动**产出就是零（严格说：托管定时任务不会跑、召回记账也不记录；想手动逼一轮，可以用 `openclaw memory promote` 和 `memory rem-backfill` 这两条 CLI，子命令定义在 `extensions/memory-core/src/cli.ts:196` 和 `:248`）。想让它跑起来，在 `~/.openclaw/openclaw.json` 里给 memory-core 插件加上 dreaming 配置（配置读取链在 `src/memory-host-sdk/dreaming.ts:350-359`——从 `plugins.entries` 里取 memory-core 的 `config`，再读其中的 `dreaming.enabled`）：

```json
{
  "plugins": {
    "entries": {
      "memory-core": {
        "config": { "dreaming": { "enabled": true } }
      }
    }
  }
}
```

&emsp;&emsp;开启并重启 gateway 后，memory-core 会注册一条名为 "Memory Dreaming Promotion" 的托管定时任务（cron 任务对象的构造在 `extensions/memory-core/src/dreaming.ts:163-187`，`buildManagedDreamingCronJob` 完整函数体，`:167` 是任务名字段行；实际注册调用 `cron.add(desired)` 在 `:451`；任务名常量在 `src/memory-host-sdk/dreaming.ts:19`），按默认频率每天凌晨 3 点做一次梦——你可以用 `openclaw cron list` 验证这条任务是否已挂上。注意做梦本身要调用模型（要花 token），按需开启。

&emsp;&emsp;那它跑完之后，`memory/dreaming/` 这个产出目录里到底长什么样？我们在真机上开启 dreaming 并触发一轮后翻开它——里面是**三个子目录**：`light/`、`deep/`、`rem/`，对应做梦的三个阶段（很像人的睡眠分期；注意这只是目录名的排列，实际执行顺序下文会单独讲、和这个排列不同），每个阶段每天写一份带日期的报告（`YYYY-MM-DD.md`）。下面是真实 workspace 里的样子：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124755379.png" width=50%></div>

&emsp;&emsp;三个阶段各管一段"消化"流程，职责和默认门槛都能在源码常量区精确定位：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>dreaming 三阶段：memory/dreaming/ 子目录职责对照</font></p>
<div class="center">

| 阶段目录 | 类比 | 干什么 | 关键默认门槛（`src/memory-host-sdk/dreaming.ts`）|
|----------|------|--------|------------------------------------------------|
| `light/` | 浅睡 | 把近期的每日记忆、会话记录扫一遍，切成一条条**候选记忆**暂存（每条带置信度、`file:line` 证据出处、被召回次数、`staged` 状态），**只暂存、不动长期记忆** | 回看 2 天（`:31`）、每轮最多 100 条（`:32`）|
| `rem/` | REM 睡眠 | 不看单条内容，跨记忆找**反复出现的主题**（报告里写成 "Theme: ... kept surfacing across N memories"），并尝试归纳关于你的 "Possible Lasting Truths"（可能的持久规律）| 回看 7 天（`:52`）、模式强度 ≥ 0.75（`:54`）|
| `deep/` | 深睡 | 从候选里筛"值得永久记住"的，**晋升写入 `MEMORY.md`**——这是整个机制的核心动作 | 评分 ≥ 0.8（`:37`）、被召回 ≥ 3 次（`:38`）、来自 ≥ 3 个不同查询（`:39`）、每晚最多晋升 10 条（`:36`）|

</div>

&emsp;&emsp;有一个第一晚必然出现、但很容易被误判成"坏了"的现象要提前说：**deep 报告大概率写着 "Promoted 0 candidate(s) into MEMORY.md"——晋升了 0 条**。这不是故障，是设计意图：晋升门槛要求"被召回 ≥ 3 次、来自 ≥ 3 个不同查询"，而召回次数靠的是你平时每次 `memory_search` 的记账（攒在第 3.1 节讲过的 `memory/.dreams/short-term-recall.json` 里），刚开启时所有候选都是 `recalls: 0`，自然谁也过不了门槛。另外说一条要诚实划界的观察：源码已经为"长期空转"风险**预留**了一套 recovery 自愈配置（`src/memory-host-sdk/dreaming.ts:44-49` 的 `RECOVERY_` 常量组——健康度阈值 0.35、补救置信度 0.9、自动写入置信度 0.97，默认 enabled），配置解析链也已就位（同文件 `:466-489`）；但截至本节基准的主分支，我们没有找到**消费**这套配置的执行代码——<font color=red>别把它当成已生效的自愈机制</font>，目前它更像"写进默认配置的路线图"。

&emsp;&emsp;还有一处必须把执行顺序讲准——同一夜任务里三个阶段的真实运行顺序是 **light → rem → deep**，不是按"浅睡到深睡"直觉想当然的 light → deep → rem（顺序的实现在 `extensions/memory-core/src/dreaming-phases.ts:1788` 的 `runDreamingSweepPhases`——函数体内先跑 light、再跑 rem；`extensions/memory-core/src/dreaming.ts:574` 是调用这次 sweep 的入口行，`:603` 之后才轮到 deep 晋升的排名报告）。这个顺序有讲究：rem 排在 deep 前面，是因为 REM 归纳出的主题信号会给 deep 晋升**加权**——主题里反复出现的候选更容易过门槛（加权常量 `PHASE_SIGNAL_REM_BOOST_MAX = 0.09` 在 `extensions/memory-core/src/short-term-promotion.ts:50`，`:723` 是应用它的计算行）。把三个阶段串成一句话：**light 把白天的碎片摆上台面 → rem 退后一步归纳主题规律 → deep 用"召回频次投票 + 主题加权"决定谁进长期记忆**——不是聊过就记住，而是反复被想起来、又和主题对得上的才配进长期记忆。

> **【踩坑预警】**：第一晚看到 deep 报告写着 "Promoted 0 candidate(s)" 就以为 dreaming 装坏了；或者按"浅睡→深睡"的直觉把执行顺序记成 light → deep → rem。后果是前者让你对着正常现象白排查一晚，后者会把 REM 加权的因果链讲反。正确理解：Promoted 0 是设计意图——晋升门槛要求"被召回 ≥ 3 次、来自 ≥ 3 个不同查询"，刚开启时召回记账全是 `recalls: 0`，谁也过不了门槛；真实执行顺序是 **light → rem → deep**，rem 的主题信号要先算出来、才能给 deep 晋升加权。排查方法：怀疑 dreaming 空转，先看 `memory/.dreams/short-term-recall.json` 里的 `recalls` 计数是否随日常 `memory_search` 在增长；顺序之争以 `dreaming-phases.ts:1788` 的 `runDreamingSweepPhases` 为准。

&emsp;&emsp;上面踩坑预警里的排查方法，我们直接做成一个现成的对账 cell——它在真机连续跑过两晚 dreaming 之后执行，一次回答四个问题：它扫了什么（增量记账）、攒了多少候选（召回计数）、重跑会不会乱写（幂等性）、三相到底按什么顺序跑（事件流证据）。四个看点全部只读；没开 dreaming 的同学跑它也不会报错，会提示先回上文开启。

In [63]:
# 真机对账：dreaming 跑过之后，去内部状态里看它"昨晚到底干了什么"（四个看点，全部只读）
import json, os, glob, hashlib
from collections import Counter

base = os.path.expanduser('~/.openclaw/workspace/memory')

# ① 增量扫描记账：每个记忆文件记着"上次被哪天的梦摄取过"（mtime+size 没变就不重复摄取）
p = f'{base}/.dreams/daily-ingestion.json'
if os.path.exists(p):
    for name, meta in list(json.load(open(p)).get('files', {}).items())[:3]:
        print(f"{name}  lastDreamingDayIngested={meta.get('lastDreamingDayIngested')}")
else:
    print('（无 daily-ingestion.json——dreaming 未开启或还没跑过，见上文开启步骤）')

# ② 候选库现状：staged 候选条数 + recallCount 分布——"为什么还是 Promoted 0"一眼可见
p = f'{base}/.dreams/short-term-recall.json'
if os.path.exists(p):
    entries = json.load(open(p)).get('entries', {})
    dist = Counter(v.get('recallCount', 0) for v in entries.values())
    print(f"\nstaged 候选 {len(entries)} 条 | recallCount 分布: {dict(sorted(dist.items()))}")

# ③ 连续两晚的 light 报告指纹：期间没有新记忆进来时，重跑产出一字不差（幂等，不是 bug）
for r in sorted(glob.glob(f'{base}/dreaming/light/*.md'))[-2:]:
    print(os.path.basename(r), 'md5 =', hashlib.md5(open(r, 'rb').read()).hexdigest()[:8])

# ④ 事件流里的三相完成顺序：执行顺序的第二证据源（与源码 runDreamingSweepPhases 互证）
p = f'{base}/.dreams/events.jsonl'
if os.path.exists(p):
    phases = [json.loads(l).get('phase') for l in open(p) if 'memory.dream.completed' in l]
    print('\nmemory.dream.completed 实际顺序:', phases[-3:])

memory/2026-06-04-1439.md  lastDreamingDayIngested=2026-06-05
memory/2026-06-04-1505.md  lastDreamingDayIngested=2026-06-05
memory/2026-06-03.md  lastDreamingDayIngested=2026-06-05

staged 候选 382 条 | recallCount 分布: {0: 382}
2026-06-04.md md5 = b7ea9e8f
2026-06-05.md md5 = b7ea9e8f

memory.dream.completed 实际顺序: ['light', 'rem', 'deep']


&emsp;&emsp;四段输出对应 dreaming 夜里干的四件事，每件都能在源码里找到主人。第一，**扫描是增量的不是全量的**——`daily-ingestion.json` 给每个记忆文件记着 `mtimeMs + size + lastDreamingDayIngested`（状态文件路径常量 `extensions/memory-core/src/dreaming-phases.ts:82`，字段声明 `:401`），文件没变化就不会被第二天的梦重复摄取——所以"做梦"不会越攒越慢。第二，**"为什么还是 Promoted 0"从断言变成了实据**——本机实测 382 条 staged 候选的 `recallCount` 全是 0（你的机器上条数会不同），正是上文"被召回 ≥ 3 次"门槛卡住的现场证据；白天用 `memory_search` 命中它们，这个分布才会往右移，deep 才有东西可晋升。

&emsp;&emsp;第三，**重跑是幂等的**——连续两晚的 light 报告 md5 一字不差，因为两晚之间没有新记忆文件进来，同一批输入就 staging 出同一批候选，这不是"卡住了"，和第一晚的 Promoted 0 一样属于"看着像故障的正常现象"。第四，**执行顺序有了第二证据源**——`events.jsonl` 里三条 `memory.dream.completed` 事件（事件类型定义 `src/memory-host-sdk/events.ts:37`）的落盘顺序就是 light → rem → deep，和上文 `dreaming-phases.ts:1788` 的源码断言互证：一个是"代码说会这么跑"，一个是"真机确实这么跑了"。顺带一个容易被忽略的细节：deep 报告里除了晋升计数还有一行维护动作——"Repaired recall artifacts: rewrote recall store"（报告行拼装在 `extensions/memory-core/src/dreaming.ts:590`，动作文案在 `:149`）——每晚晋升之外它还会顺手修复重写召回库，这是 housekeeping（日常维护），不产出记忆。

&emsp;&emsp;最后把 `DREAMS.md` 的角色钉死，因为它最容易被想歪。它**不是**"cron 每次要执行什么"的任务清单（cron 要执行的内容是任务 payload 里一条固定的内部系统事件文本，到点发给 agent 触发做梦管线，不在任何 md 文件里），也不是机制的输入——它是**做完梦之后写的输出日记**：每跑完一轮，模型会拿当晚处理的记忆碎片写一段"梦境叙事"追加进去，生成它的提示词原文就在源码里——"Write a dream diary entry from these memory fragments"（`extensions/memory-core/src/dreaming-narrative.ts:289`）。三个特征帮你确认这一点：① 条目的时间戳是**做完梦的时刻**，不是预定的待办时间；② 条目都夹在 `` 这对标记之间，每晚追加一条，是本越写越厚的日记；③ 内容是记忆碎片的"梦境化改写"——我们真机实测的第一篇日记，把当天"查插件清单"的会话（67 个已加载 / 27 个停用、一个叫 `commands.plugins` 的配置开关）写成了"67 扇亮着的窗、27 扇暗的、一个我今晚决定不去碰的开关"。一句话分工：**cron 的任务内容是固定的触发信号，三阶段报告是工作记录，`DREAMS.md` 是执行完的"汇报文学"**——cron 不会回头读它。

&emsp;&emsp;文字讲到这，dreaming 的完整闭环已经齐了——白天积累、凌晨触发、三阶段消化、三类产出、再回流进下一轮 context。我们用一张全景图把这条"记忆消化链"串起来。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605124750944.png" width=70%></div>

&emsp;&emsp;不管有没有产出，你都已经知道了去哪找它（`memory/dreaming/` 按三阶段找报告、`DREAMS.md` 看日记，`memory/.dreams/` 是它的内部状态目录）。下面这个 Tier 2 端到端验证，把本章 hybrid RAG 检索的完整链路再跑一次，确认"最相关的记忆被排到最前"这个记忆系统的核心承诺成立。

In [64]:
# Tier 2 端到端验证：hybrid RAG 完整检索链路，确认最相关记忆排最前
# 1 个 case：给定 3 条 mock 记忆 + 一个 query，跑完整 vector->text->merge 链路

e2e_memories = [
    MemoryEntry("a", "用户是 Python 工程师，常写后端", emb=[1.0, 0.0, 0.0], keywords=["Python", "后端", "工程师"]),
    MemoryEntry("b", "用户养了一只猫", emb=[0.0, 1.0, 0.0], keywords=["猫", "宠物"]),
    MemoryEntry("c", "用户偶尔写前端 JavaScript", emb=[0.6, 0.0, 0.3], keywords=["前端", "JavaScript"]),
]

# query："用户的编程背景"——语义最贴 a（Python 后端），关键词也命中 a
e2e_query_emb = [0.98, 0.0, 0.1]
e2e_query_kw = ["Python", "工程师"]

# 完整链路：向量检索 -> 全文检索 -> 融合排序
e2e_ranked = hybrid_merge(
    vector_search(e2e_query_emb, e2e_memories),
    text_search(e2e_query_kw, e2e_memories),
)

print("端到端检索排序：")
for entry_id, score in e2e_ranked:
    print(f"  {entry_id}: {round(score, 3)}")

# 断言：最相关的 a（编程背景，双路都强）排第一，最不相关的 b（养猫）垫底
assert e2e_ranked[0][0] == "a", "编程背景查询应让 a（Python 后端）排第一"
assert e2e_ranked[-1][0] == "b", "与编程无关的 b（养猫）应垫底"
print("\nTier 2 通过：hybrid RAG 把最相关记忆排到最前，记忆检索闭环成立")

端到端检索排序：
  a: 0.996
  c: 0.655
  b: 0.0

Tier 2 通过：hybrid RAG 把最相关记忆排到最前，记忆检索闭环成立


&emsp;&emsp;这个端到端验证证明了记忆系统的核心承诺：给一个查询，hybrid RAG 能把最相关的那条记忆（a，编程背景，向量和关键词双路都强）排到最前，把不相关的（b，养猫）排到最后。本章你已经能划清短期与长期记忆的分界、能翻 workspace 看到 3 个记忆文件、能按文件名形态归因记忆的三条自动来路（含 `/new` 收尾的 hook 快照）、能区分 dreaming 三处的职责、能讲清 hybrid RAG 的 0.7/0.3 加权来源、也知道会话转录记忆默认关闭这条边界。

> 📌 **【第 3 章你已经掌握】**：短期 vs 长期记忆的分界；workspace 3 个记忆文件（`MEMORY.md` / 每日记忆 / `DREAMS.md`）；按文件名形态归因记忆的三条自动来路（flush 抢救 / hook 快照 / dreaming 沉淀）；dreaming 三处职责区分；hybrid RAG 0.7/0.3 权重出处；`memory_search` 定位 + `memory_get` 精读的双步范式。

**进阶：记忆后端生态全景**

> 📌 **这是给需要替换记忆后端的工程实践同学的加餐**——只想吃透机制的同学可以直接跳到第 4 章，不影响主线。

&emsp;&emsp;你已经看清了内置记忆"怎么存、怎么找、怎么做梦"。但再拉高一层——这套内置记忆只是 OpenClaw 的**默认实现**，记忆系统本身是一个 plugin（插件）可拓展生态。这正好呼应第一节课讲过的 plugin / capability / tool 边界：记忆后端就是一类 plugin，换后端就像换插件。OpenClaw 目前已有以下几种记忆后端：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>OpenClaw 记忆后端全景</font></p>
<div class="center">

| 记忆后端 | 存储 | 检索方式 | 定位 |
|---------|------|---------|------|
| memory-core（内置默认） | Markdown 文件落盘（MEMORY.md + memory/*.md）+ SQLite 索引（FTS5 全文 + sqlite-vec 向量） | hybrid RAG（本章主角） | 默认实现，开箱即用 |
| memory-lancedb | LanceDB（嵌入式向量数据库，可落本地文件或 S3 等远端） | 纯向量 + 自动召回（auto-recall，默认开，`extensions/memory-lancedb/config.ts:249`）；可选自动落库（auto-capture，默认关闭）；另带 `memory_store` / `memory_forget` 写入工具 | 专用长期向量后端 |
| memory-wiki | Markdown / Obsidian vault | 编译后的知识检索（作为 memory 补充源接入时是只读 search/get；插件自身另有 `wiki_apply` 等读写维护工具）| 知识库式补充 |
| active-memory | 无自有长期库 | 跑一个 LLM 子 agent 判断哪些记忆相关再注入 | 智能召回层 |
| 外部系统（mem0 / Zep 等） | 各自托管 | 各自 | 通过下面说的扩展接口接入 |

</div>

<!-- ILLUSTRATION: type=concept | file=memory_backend_three_layers.png
提示词：OpenClaw 记忆后端生态：三层接入深度与开放度。同心三层结构，从外到内：外层标"只读检索增强"，注"实现 search() / get() 两方法即可挂入"，绿色徽标"开放"，挂一张成员卡 memory-wiki（叠加层，与内置记忆共存）；中层标"插件槽位 plugins.slots.memory"，注"完整替换记忆后端"，黄色徽标"半开放"，挂两张成员卡 mem0（槽位替换者）、memory-lancedb；内核层标"原生后端 memory.backend"，注"枚举只有 builtin / qmd"，配灰色锁图标和灰色徽标"封闭"，内核中心小字"builtin = 内置 memory-core（hybrid RAG）"。外层与中层成员卡用箭头指向各自所在环。底部总结带：检索增强开放 · 插件槽位半开放 · 原生枚举封闭。技术标识符 search()、get()、plugins.slots.memory、memory.backend、builtin、qmd、mem0、memory-wiki、memory-lancedb、memory-core 保留英文等宽原文 -->
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605163000001.png" width=70%></div>

&emsp;&emsp;（首次出现的术语说明：`mem0` = 一个开源的外部 AI 记忆服务；`LanceDB` = 嵌入式向量数据库；`Zep` 是同类外部记忆服务。）

&emsp;&emsp;要接入一个新的记忆后端，OpenClaw 提供了几层由浅到深的扩展接口（这里的"几层"是便于理解的归纳，不是源码里的正式命名）——最浅的一层只需实现 `search()` 和 `get()` 两个方法，就能当一个"只读补充源"挂进去；更深的一层可以实现完整的 memory runtime、整个替换掉后端。这些接口面定义见 `src/plugins/memory-state.ts`。

&emsp;&emsp;那到底能不能把 `mem0` 这类外部记忆接进来？诚实的答案分三层——最浅一层，作为"只读检索增强"接进来成本很低，包一层 `search` / `get` 就行；中间还有一条路：把它包装成完整的 memory plugin 去占据 `plugins.slots.memory` 这个插件槽位（slots 的 schema 定义在 `src/config/zod-schema.ts:1202-1208`，上面表里的 memory-lancedb 走的就是这条路），能拿到比只读补充深得多的接管权——而且这条路不是纸上谈兵，mem0 官方已经发布了正是走这条路的插件 `@mem0/openclaw-mem0`，下面马上看它怎么开；但要让它成为顶层 `memory.backend` 意义上的**原生后端**，**截至本节基准的主分支（2026-05-29）还做不到**：这个配置项的取值只固化了 `"builtin"` 和 `"qmd"` 两种（`src/config/zod-schema.ts:203` 的 `z.union`，另见 `packages/memory-host-sdk/src/host/backend-config.ts`），而且最浅的只读补充接口并没有 capture / update / delete 这些写入方法。一句话收口：检索增强**开放**、插件槽位**半开放**、原生后端枚举**封闭**——想"整个换成 mem0"，要么走插件槽位，要么改核心。


### memory-wiki 配置与说明

&emsp;&emsp;说完"能不能接"，落地到"怎么开"。上面全景表里有两个成员最值得给出真实的开启配置——**memory-wiki**（仓库自带的叠加层，但默认未启用）和 **mem0**（第三方的槽位替换者，需要单独安装），它们的配置路数恰好代表了两类扩展方式，各看一遍你就掌握了这个生态的全部打开姿势。

&emsp;&emsp;先说 `memory-wiki`，它身上正好有个绝佳的"manifest ≠ 运行时"案例。只看 manifest 你会以为它开箱即用——`extensions/memory-wiki/openclaw.plugin.json` 里写着 `activation.onStartup: true`；但真机一跑就现形：`openclaw plugins list` 显示它的 Status 是 **disabled**，直接敲 `openclaw wiki` 会得到一句很诚实的报错——"that bundled plugin is **disabled by default**. Run `openclaw plugins enable memory-wiki`"（这段判定在 `src/cli/run-main-policy.ts:289-294`：`enabledByDefault` 非 true 且配置未显式 `enabled: true` 即按默认禁用处理）。两个字段其实是两层开关：`enabled` 决定插件**装不装上膛**——bundled 插件默认不上膛；`onStartup` 只决定**上膛之后**是随 gateway 启动立即激活、还是按需激活。所以第一步是真正的"开"：跑 `openclaw plugins enable memory-wiki`，或等价地在配置里写 `plugins.entries.memory-wiki.enabled: true`（下面官方示例里那行 `"enabled": true` 正是干这个的，不是装饰）。启用之后，`wiki_search` / `wiki_get` / `wiki_apply` / `wiki_lint` / `wiki_status` 五个工具和 `openclaw wiki` 子命令才就位（工具清单与命令别名声明在 manifest 的 `contracts.tools` 和 `commandAliases` 字段）。



<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605172114370.png" width=50%></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260605172319072.png" width=50%></div>

&emsp;&emsp;第二步才是配"怎么用"。零配置时它的默认行为是：vault 落在 `~/.openclaw/wiki/main`（默认路径生成在 `extensions/memory-wiki/src/config.ts:197`），`vaultMode` 为 `isolated`（独立 vault、不依赖 memory-core）、渲染 `native` 格式、检索走 `shared` 后端且语料域为 `wiki`（四个默认值常量在 `config.ts:102-105`），`bridge` 模式默认**关闭**。想定制时，配置全部放在 `plugins.entries.memory-wiki.config` 下，官方文档（`docs/plugins/memory-wiki.md`）给的典型配置长这样（节选关键字段）：

```json
{
  "plugins": {
    "entries": {
      "memory-wiki": {
        "enabled": true,
        "config": {
          "vaultMode": "bridge",
          "vault": { "path": "~/.openclaw/wiki/main", "renderMode": "obsidian" },
          "bridge": { "enabled": true, "readMemoryArtifacts": true }
        }
      }
    }
  }
}
```

&emsp;&emsp;这份配置做了三件事：把 vault 模式从 `isolated` 切到 `bridge`（开始读取 memory-core 公开导出的记忆 artifacts，把每日记忆、dream-report 等编译成 wiki 页面）、把渲染切成 Obsidian 友好格式（方便你用 Obsidian 直接打开这个 vault）、并显式打开 bridge 读取。配好后跑 `openclaw wiki doctor` 可以体检 bridge 链路是否通。还记得 3.3 节讲过的 `corpus` 参数吗——这里就接上了：`memory_search` 的 `corpus=wiki` 查的正是这个 vault 编译出的语料，`corpus=all` 则一次跨记忆和 wiki 两层。

&emsp;&emsp;再说 `mem0`。它和 memory-wiki 的根本区别是：memory-wiki 是**叠加层**（不占记忆槽位，与 memory-core 共存），mem0 是**槽位替换者**（占据 `plugins.slots.memory`，把 memory-core 整个换下场）。

> 📅 **时效性说明**：mem0 插件不在 OpenClaw 仓库内，以下安装与配置来自 mem0 官方文档（docs.mem0.ai/integrations/openclaw，2026 年 6 月查证）与其 GitHub 仓库（github.com/mem0ai/mem0）。第三方插件的包名与字段演进比内置插件快，照抄前建议先过一遍官方文档最新版。

> **【关于本节代码块格式】**：以下 mem0 的安装与初始化命令均在系统终端（macOS Terminal / Linux shell）执行，不在 Jupyter Notebook 内运行——且它需要 Mem0 账号，本课不现场跑，只看清路数。

&emsp;&emsp;开启分三步。第一步在系统终端安装插件，也可以直接在 OpenClaw 对话里发一句 `Setup Mem0 from mem0.ai/claw-setup`，走官方的邮箱 + OTP 自动配置向导：

```bash
openclaw plugins install @mem0/openclaw-mem0
```

&emsp;&emsp;第二步在 `openclaw.json` 里做两件事：把记忆槽位指给它 + 填它自己的配置。云端托管模式（Platform Mode）的核心是 `apiKey` 和 `userId` 两个字段（官方示例里还带 `skills` 等一组可选细配——召回 token 预算、重排开关之类，这里从简只看主干）：

```json
{
  "plugins": {
    "slots": { "memory": "openclaw-mem0" },
    "entries": {
      "openclaw-mem0": {
        "enabled": true,
        "config": { "apiKey": "${MEM0_API_KEY}", "userId": "alice" }
      }
    }
  }
}
```

&emsp;&emsp;不想把记忆托管到云端，它还有全本地的开源模式（Open-Source Mode）：把 `config.mode` 设为 `"open-source"`，并在 `oss` 下自带三件套，骨架长这样（provider 可换，也可直接跑官方向导 `openclaw mem0 init --mode open-source` 交互式生成）：

```json
{
  "config": {
    "mode": "open-source",
    "userId": "alice",
    "oss": {
      "embedder":    { "provider": "ollama", "config": { "model": "nomic-embed-text" } },
      "vectorStore": { "provider": "qdrant", "config": { "host": "localhost", "port": 6333 } },
      "llm":         { "provider": "anthropic", "config": { "model": "<你的模型>" } }
    }
  }
}
```

&emsp;&emsp;第三步重启 gateway，用 `openclaw plugins list --enabled` 和 `openclaw mem0 status` 验证插件已激活。

&emsp;&emsp;最后两条诚实划界，都来自官方渠道的真实记录：其一，槽位是**全局**的——mem0 插件一旦占据 `plugins.slots.memory`，所有 agent 的记忆后端都被替换，**不支持按 agent 分别指定**（多 agent 场景想"只给研究 agent 用 mem0"目前做不到，这是 mem0 官方仓库的开放 issue：mem0ai/mem0#4126）；其二，如果你的配置里用了 `plugins.allow` 白名单，得把 mem0 加进去，否则它的 CLI 子命令会报 unavailable（官方文档 Troubleshooting 一节记录的高频坑）。对照前面的三层判断你会发现闭环了：mem0 走的正是"插件槽位"这条半开放路线——拿到了完整接管权，也继承了这条路线的全局性约束。

&emsp;&emsp;配置层面就讲到这里，最后回到机制本身收口——hybrid RAG 这个"向量 + 全文加权融合"的模式，你以后在几乎任何 agent 或 RAG 系统里都会反复遇到，它不是 OpenClaw 独有的。今天你在 OpenClaw 的记忆系统里把它的机理彻底搞懂了，这份理解是可以迁移的。三堵墙我们都拆完了——当下组装、分治隔离、持久来源。最后一章，我们把这三块拼成一个完整的框架收口。

---

## <center>第 4 章：总结</center>

&emsp;&emsp;走到这里，本节课的三个机制都讲完了。这一章很短，但很重要——它要做两件事：第一，把前三章的三个机制拼成一个**统一的框架**，让你心里有一张完整的地图，而不是三块各自孤立的知识；第二，明明白白把本节的诚实边界再集中列一遍。

&emsp;&emsp;为什么收口值得单开一章？因为框架的价值在于**可迁移**——本节看的虽然是 OpenClaw 一家的源码，但"当下怎么组装、复杂了怎么分治、跨会话怎么留存"这三个职责，是你以后看任何 agent 系统都绕不开的三问。把它们装进一个好记的框架，比记住一堆零散结论值钱得多。我们从那个在第 0 章就给你看过结论图的"三脑一轴"框架开始。

### 4.1 三脑一轴收口

&emsp;&emsp;还记得第 0 章那张全景图吗？现在你已经学完了三个机制，可以真正理解它了。我们把前三章的三个模块，用一个形象的比喻拼成一个框架——**三脑一轴**。再次郑重声明：**"三脑一轴"（当下脑 / 分治脑 / 长期脑）是本课为了帮你记住三个模块职责而起的教学比喻，不是 OpenClaw 的官方术语，AI agent 领域也没有这套通用叫法**。它的唯一价值是给你一个好记的抓手，源码里这三块各有正式名字。这也正是本章标题里说的"**命名陷阱**"——自创比喻好用，但拿去跟别人交流之前要先祛魅：报出"ContextEngine / subagent / 记忆系统"这些正式名字，别让"三脑一轴"冒充官方概念。

&emsp;&emsp;**当下脑，就是第 1 章的 ContextEngine（连同它背后真正干活的 runtime 组装管线——1.2 的"实现归属"讲过，默认 legacy 形态下实际组装由 runtime 执行）**。 它决定 agent 这一轮能"看到"什么——把系统指令、记忆、历史、当前消息组装成发给模型的 context。它管的是"此刻"**。分治脑，就是第 2 章的 subagent 系统**。 当一个 loop 扛不住时，它派生出子 agent 来分摊任务，子 agent 在 session 和 context 上隔离、在 lane（全局并发队列）和 auth（合并继承）上共享。它管的是"分工"**。长期脑，就是第 3 章的记忆系统**。 它跨会话存储（MEMORY.md 等三文件），并通过 hybrid RAG 把相关记忆检索回来。它管的是"过往"。

&emsp;&emsp;而那"一轴"，就是 `context`——三个脑全都围着它转，这才是这个框架的精髓。你看这条贯穿线有多清晰：长期脑（记忆系统）存下来的东西，最终通过第 1 章 assemble 阶段的"记忆注入"层，**回灌进当下脑的 context**（这就是为什么第 3 章我们反复说 MEMORY.md 接的是第 1 章那个第②层）；分治脑（subagent）派生子 agent 时，是通过第 1 章末尾那道 `prepareSubagentSpawn` 门，**为子 agent 准备独立的 context 快照**。三个脑做的事看似不同，但它们服务的是同一件事——**让 agent 在当下这一轮，拿到最合适的 context**。这就是"一轴"的含义。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三脑一轴：三个机制如何围绕 context 主轴协同</font></p>
<div class="center">

| 脑（本课比喻）| 源码里的真名 | 管什么 | 怎么连上 context 主轴 |
|---------------|--------------|--------|------------------------|
| 当下脑 | ContextEngine | 此刻这一轮的 context 组装 | 它**就是** context 主轴本身的组装者 |
| 分治脑 | subagent 系统 | 复杂任务派生子 agent 分工 | 通过 prepareSubagentSpawn 为子 agent 备独立 context |
| 长期脑 | 记忆系统（3 文件 + hybrid RAG）| 跨会话持久化与检索 | 通过 assemble 的"记忆注入"层回灌进 context |

</div>

&emsp;&emsp;这张表把比喻和源码真名对齐了，也把每个脑跟 context 主轴的连接点标清楚了。带走这个框架最大的好处是它**可迁移**——你以后看任何一个 agent 系统，都可以问自己这三个问题：它的"当下脑"怎么组装 context？它有没有"分治脑"、怎么分？它的"长期脑"怎么存怎么取？不管那个系统管这三块叫什么名字，这三个职责它都得有。我们进 4.2 节，把诚实边界和能力回顾做个收尾。

### 4.2 本节能力回顾

&emsp;&emsp;在结束之前，我们把本节贯穿始终的五条诚实边界集中列一遍——这不是为了挑刺，而是因为**区分"我有证据"和"我亲眼跑通了"本身就是 agent 工程师的核心素养**。一个靠谱的工程师，敢于明明白白说出"这块我还没完全验证"。

&emsp;&emsp;第一条，**subagent 四个维度里只有 session 隔离做了运行时实证**（真机派生、session key 独立落盘眼见为实），context 隔离和共享侧（lane / auth）仍是"源码 + 官方文档证、运行时待补"；顺带记住结果回报的 `announce` 是 best-effort 的，gateway 重启会丢，别按可靠投递设计。第二条，**会话转录记忆默认关闭**——`experimental.sessionMemory` 默认 `false`，不是常驻的，要显式打开。第三条，**dreaming 现场难即时触发**——它是后台/定时机制，我们降级为翻看已有产物。第四条，**fail-closed 只是 compact 和 prepareSubagentSpawn 两个阶段的特例**——不是整套 ContextEngine 的普遍行为，assemble 失败有自己的降级。第五条，**dreaming 的 recovery 自愈只有配置预留、没有执行链**——常量和解析链都在源码里，但当前主分支没有消费它的执行代码，别把它当已生效机制。

&emsp;&emsp;最后我们对照本节开头那 3 个问题，做一次能力回顾——下面这些事，现在的你应该都能答上来了。关于**第一个问题**（一条消息发给模型时，实际 prompt 塞了哪几层）：你能用 `proxy blob` 看见实际 context，并拆成系统指令 / 记忆注入 / 历史 / 当前消息四层，还能报出那笔成分账单——体积大头是工具结果和工具 schema、对话文字不足 1%；你能描述 ContextEngine 把它组装出来的五个阶段，也知道"窗口上限"从哪来、compact 的触发入口不止自动预检一条、且 compact 只是四道防线（源头限流 → 落盘封顶 → 组装瘦身 → 压力触发）里最后的那一招。关于**第二个问题**（subagent 与主 agent 隔离了什么、又共享了什么）：你能说清隔离了 session 和 context，共享了 lane（同一条全局并发队列，maxConcurrent=8）和 auth（合并继承），能引用官方文档那句 "not supported yet"，还能讲清派生的通信模型——spawn 非阻塞、结果靠 `announce`（best-effort）回报、等待用 `sessions_yield`。关于**第三个问题**（跨会话记忆怎么存、怎么被检索回来）：你能翻 workspace 看到 3 个记忆文件、区分 dreaming 三处、讲清 hybrid RAG 的 0.7 + 0.3 加权检索，也知道没配 embedding 时它会优雅降级到 FTS-only 纯全文模式。这三个问题答得上来，本节课你就真带走了。

> 📌 **【学完本节你已经掌握】**：① 用进程内捕获开关 + `proxy sessions` / `blob`（中间用 sqlite 查事件 id）看真实 context 四层，外加一笔成分账单（大头是工具结果和工具 schema）；② ContextEngine 五阶段 + fail-closed 范围，且 compact 只是四道防线的最后一招；③ 可跑的 context 组装 + compact MVP（进阶再加四道防线流水线 + 缓存前缀对比 demo + 真机核账 cell）；④ subagent 两隔离（session/context）+ 两共享（lane 全局并发队列 / auth 合并继承），知道 auth 不隔离是官方文档明说的当前现状、工具约束只严不松（`sessions_send` 等永久禁用是策略层强制）、非阻塞 spawn 靠 `announce` 回报；⑤ 一段可跑的隔离/共享语义 MVP；⑥ 3 个记忆文件（`MEMORY.md` / 每日记忆 / `DREAMS.md`——第三个要开启 dreaming 后才出现，`MEMORY.md` 含 10K 注入预算）+ dreaming 三处区分 + 按文件名形态归因记忆的三条自动来路（flush 抢救 / hook 快照 / dreaming 沉淀）；⑦ hybrid RAG 的 0.7/0.3 加权检索来源 + `memory_search` 定位 / `memory_get` 精读的双步范式。这 7 件产物，对照开头的产物清单逐一确认，缺哪块就回去翻对应章节。